In [ ]:
# Latest part's summary figure (Part 12). Every number on it is measured again further down.
import glob, os, warnings
import pandas as pd
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from matplotlib.patches import FancyBboxPatch

SURFACE, PAGE = "#fcfcfb", "#f9f9f7"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, RING = "#e1e0d9", "#c3c2b7", "#e6e5df"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
SEQ = ["#f0f6fe", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]
GOOD = "#006300"


def _style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 10, "text.color": INK,
        "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.facecolor": SURFACE,
        "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
        "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
        "axes.grid": False, "figure.facecolor": PAGE, "savefig.facecolor": PAGE,
    })


def _title(ax, title, sub):
    ax.text(0, 1.115, title, transform=ax.transAxes, fontsize=12.5, fontweight="bold", color=INK, va="bottom")
    ax.text(0, 1.035, sub, transform=ax.transAxes, fontsize=9.5, color=INK2, va="bottom")


def _tile(fig, x, y, w, h, label, value, note=None):
    ax = fig.add_axes([x, y, w, h])
    ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=0.06",
                                transform=ax.transAxes, facecolor=SURFACE, edgecolor=RING, lw=1))
    ax.text(0.06, 0.70, label, transform=ax.transAxes, fontsize=10, color=INK2, va="center")
    ax.text(0.06, 0.40, value, transform=ax.transAxes, fontsize=22, fontweight="bold", color=INK, va="center")
    if note:
        ax.text(0.06, 0.13, note, transform=ax.transAxes, fontsize=8.8,
                color=GOOD if note.startswith("+") else MUTED, va="center")


def _heat(fig, rect, heat):
    ax = fig.add_axes(rect)
    rate = np.asarray(heat["rate"], dtype=float)
    cmap = LinearSegmentedColormap.from_list("seq", SEQ)
    im = ax.imshow(rate, origin="lower", cmap=cmap, norm=PowerNorm(0.45, vmin=0, vmax=rate.max()), aspect="auto")
    n = rate.shape[0]
    ax.set_xticks([0, n - 1], ["low", "high"])
    ax.set_yticks([0, n - 1], ["low", "high"])
    ax.set_xlabel(f"{heat['xname']}  (deciles)  →")
    ax.set_ylabel(f"{heat['yname']}  (deciles)  →")
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(length=0)
    for (i, j) in [np.unravel_index(rate.argmax(), rate.shape), np.unravel_index(rate.argmin(), rate.shape)]:
        v = rate[i, j]
        ax.text(j, i, f"{v:.1%}" if v >= 0.001 else "<0.1%", ha="center", va="center", fontsize=8.5,
                fontweight="bold", color="#ffffff" if v > rate.max() * 0.3 else INK)
    cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=8, colors=MUTED, length=0)
    cb.set_ticks([t for t in (0, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30) if t <= rate.max()])
    cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    _title(ax, "Where the fraud is", f"fraud rate per cell · base rate {heat['base']:.2%}")


def _dumbbell(fig, rect, te_gain):
    """te_gain: list of (column, raw single-feature AUC, target-encoded AUC), top first."""
    ax = fig.add_axes(rect)
    cols = [c for c, _, _ in te_gain][::-1]
    raw = np.array([r for _, r, _ in te_gain][::-1])
    enc = np.array([e for _, _, e in te_gain][::-1])
    yy = np.arange(len(cols))
    ax.hlines(yy, raw, enc, color=AXIS, lw=2, zorder=1)
    ax.scatter(raw, yy, s=52, color=MUTED, edgecolor=SURFACE, linewidth=2, zorder=2, label="raw value")
    ax.scatter(enc, yy, s=52, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3, label="target-encoded value")
    for yv, r, e in zip(yy, raw, enc):
        ax.text(max(r, e) + 0.004, yv, f"+{(e - r) * 100:.1f}", va="center", fontsize=8.5, color=INK2)
    ax.set_yticks(yy, cols)
    ax.tick_params(axis="y", length=0, labelsize=9, labelcolor=INK)
    ax.spines["left"].set_visible(False)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_xlim(min(raw.min(), enc.min()) - 0.01, max(raw.max(), enc.max()) + 0.03)
    ax.set_xlabel("single-feature AUC (fold 0)")
    ax.legend(loc="upper right", frameon=False, fontsize=8.5, handletextpad=0.2)
    _title(ax, "Every value has its own risk", "per-value target encoding vs the raw number (AUC × 100 gain)")


def _depth(fig, rect, depth):
    """depth: dict(leaves=[...], series={label: [auc per leaves or None]})"""
    ax = fig.add_axes(rect)
    lv = np.asarray(depth["leaves"])
    colors = [MUTED, BLUE]
    for (lab, au), col in zip(depth["series"].items(), colors):
        au = np.array([np.nan if v is None else v for v in au], dtype=float)
        ok = ~np.isnan(au)
        ax.plot(lv[ok], au[ok], color=col, lw=2, solid_capstyle="round", zorder=2)
        ax.scatter(lv[ok], au[ok], s=46, color=col, edgecolor=SURFACE, linewidth=2, zorder=3)
        last = np.where(ok)[0][-1]
        ax.text(lv[last] * 1.12, au[last], lab, va="center", fontsize=9, color=INK2)
        k = int(np.nanargmax(au))
        ax.annotate(f"{au[k]:.4f}", (lv[k], au[k]), xytext=(0, 9), textcoords="offset points",
                    ha="center", fontsize=9, fontweight="bold", color=INK)
    ax.set_xscale("log", base=2)
    ax.set_xticks(lv, [str(v) for v in lv])
    ax.minorticks_off()
    ax.set_xlim(lv.min() / 1.3, lv.max() * 2.6)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_xlabel("LightGBM num_leaves (log scale)")
    ax.set_ylabel("fold-0 AUC")
    lo, hi = np.nanmin([v for a in depth["series"].values() for v in a if v is not None]), \
        np.nanmax([v for a in depth["series"].values() for v in a if v is not None])
    ax.set_ylim(lo - (hi - lo) * 0.08, hi + (hi - lo) * 0.16)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    _title(ax, "Shallow trees win", "the signal is close to additive, like a logistic model")


def _ladder(fig, rect, ladder, title="What moved the score",
            sub="5-fold out-of-fold AUC, the same folds for every model",
            xlabel="5-fold CV AUC gain over the baseline (× 1e-4)"):
    ax = fig.add_axes(rect)
    steps = [s for s, _ in ladder]
    cv = np.array([v for _, v in ladder])
    yy = np.arange(len(steps))[::-1]
    gain = (cv - cv[0]) * 1e4
    colors = [AXIS] + [AQUA] * (len(cv) - 2) + [ORANGE]
    ax.barh(yy, np.maximum(gain, 0.15), height=0.5, color=colors, zorder=2)
    for yv, g, v in zip(yy, gain, cv):
        ax.text(max(g, 0) + gain.max() * 0.02, yv, f"{v:.5f}" + (f"   +{g:.1f}" if g > 0 else "   baseline"),
                va="center", fontsize=9.5, color=INK)
    ax.set_yticks(yy, steps)
    ax.tick_params(axis="y", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(0, gain.max() * 1.45)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(xlabel)
    _title(ax, title, sub)


def _blend(fig, rect, blend):
    ax = fig.add_axes(rect)
    ax.set_axis_off()
    cols = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA]
    bottom = 1.0
    for (nm, w), c in zip(blend, cols):
        ax.add_patch(FancyBboxPatch((0.0, bottom - w + 0.006), 0.30, w - 0.012,
                                    boxstyle="round,pad=0,rounding_size=0.012", transform=ax.transAxes,
                                    facecolor=c, edgecolor="none"))
        if w >= 0.06:
            ax.text(0.03, bottom - w / 2, f"{w:.0%}", transform=ax.transAxes, va="center", fontsize=10,
                    fontweight="bold", color=INK if c == YELLOW else "#ffffff")
        ax.text(0.36, bottom - w / 2, nm, transform=ax.transAxes, va="center", fontsize=9.5, color=INK)
        bottom -= w
    _title(ax, "The stack", "share of the stacker's logit weights")


def draw(stats, heat, te_gain, depth, ladder, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _heat(fig, [0.035, 0.425, 0.24, 0.29], heat)
    _dumbbell(fig, [0.43, 0.425, 0.22, 0.29], te_gain)
    _depth(fig, [0.72, 0.425, 0.20, 0.29], depth)
    _ladder(fig, [0.20, 0.05, 0.47, 0.26], ladder)
    _blend(fig, [0.74, 0.05, 0.22, 0.26], blend)
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


# ---------------------------------------------------------------------------------------------
# Part 2 panels: two views of the data (target-encoded vs raw), what each candidate adds,
# smoothing as an extra view, the Part 1 -> Part 2 slope, the hill-climb weights.

VIEW_COLORS = {"target-encoded view": BLUE, "raw view": ORANGE, "neural net": AQUA, "joint GLM": YELLOW}


def _corr(fig, rect, names, corr):
    ax = fig.add_axes(rect)
    c = np.asarray(corr, dtype=float)
    cmap = LinearSegmentedColormap.from_list("seq", SEQ)
    lo = np.floor(c[c < 1].min() * 100) / 100
    im = ax.imshow(c, cmap=cmap, vmin=lo, vmax=1.0)
    ax.set_xticks(range(len(names)), names, rotation=35, ha="right")
    ax.set_yticks(range(len(names)), names)
    ax.tick_params(length=0, labelsize=8.8, labelcolor=INK)
    for s in ax.spines.values():
        s.set_visible(False)
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j:
                ax.text(j, i, f"{c[i, j]:.3f}", ha="center", va="center", fontsize=8,
                        color="#ffffff" if c[i, j] > lo + (1 - lo) * 0.55 else INK)
    _title(ax, "Two views, two blocks", "Spearman correlation of the out-of-fold predictions")


def _gains(fig, rect, gains):
    """gains: list of (candidate, view, gain x 1e-4), any order."""
    ax = fig.add_axes(rect)
    g = sorted(gains, key=lambda t: t[2])
    yy = np.arange(len(g))
    ax.barh(yy, [v for _, _, v in g], height=0.55, color=[VIEW_COLORS[w] for _, w, _ in g], zorder=2)
    for yv, (_, _, v) in zip(yy, g):
        ax.text(v + (0.04 if v >= 0 else -0.04), yv, f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right",
                fontsize=8.5, color=INK2)
    ax.set_yticks(yy, [n for n, _, _ in g])
    ax.tick_params(axis="y", length=0, labelsize=8.8, labelcolor=INK)
    ax.axvline(0, color=AXIS, lw=1)
    ax.spines["left"].set_visible(False)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_xlim(min(0, min(v for *_, v in g)) - 0.35, max(v for *_, v in g) * 1.75)
    ax.set_xlabel("stack CV gain when added alone to Part 1 (× 1e-4)")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=c, label=k) for k, c in VIEW_COLORS.items()], loc="lower right",
              frameon=False, fontsize=8.5)
    _title(ax, "What each candidate adds", "one at a time on top of Part 1's five-model stack")


def _smoothing(fig, rect, sm):
    """sm: dict(weights=[...], cv=[...], kept=[weights in the blend])"""
    ax = fig.add_axes(rect)
    w, cv = np.asarray(sm["weights"]), np.asarray(sm["cv"])
    ax.plot(w, cv, color=BLUE, lw=2, zorder=2)
    for x, v in zip(w, cv):
        kept = x in sm["kept"]
        ax.scatter([x], [v], s=60 if kept else 46, color=BLUE if kept else SURFACE, edgecolor=BLUE if not kept else SURFACE,
                   linewidth=2, zorder=3)
        ax.annotate(f"{v:.6f}", (x, v), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8.5,
                    color=INK, fontweight="bold" if kept else "normal")
    ax.set_xscale("log")
    ax.set_xticks(w, [str(int(x)) for x in w])
    ax.minorticks_off()
    ax.set_xlim(w.min() / 1.6, w.max() * 1.6)
    span = cv.max() - cv.min()
    ax.set_ylim(cv.min() - span * 0.4, cv.max() + span * 0.6)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.5f}"))
    ax.set_xlabel("target-encoding smoothing weight (log scale)")
    ax.set_ylabel("5-fold CV AUC, 4-leaf LightGBM")
    ax.text(0.02, 0.04, "filled = kept in the blend", transform=ax.transAxes, fontsize=8.5, color=INK2)
    _title(ax, "Smoothing is a free extra view", "same model, different encodings: 100 and 300 both make the blend")


def _slope(fig, rect, parts, title="Part 1 → Part 2", sub="the public score moved with the CV"):
    """parts: list of (label, cv, lb)."""
    ax = fig.add_axes(rect)
    x = np.arange(len(parts))
    for k, (key, col, lab) in enumerate([(1, MUTED, "5-fold CV"), (2, BLUE, "public LB")]):
        v = [p[key] for p in parts]
        ax.plot(x, v, color=col, lw=2, zorder=2)
        ax.scatter(x, v, s=52, color=col, edgecolor=SURFACE, linewidth=2, zorder=3)
        for xi, vi in zip(x, v):
            last = xi == x[-1]
            ax.annotate(f"{vi:.5f}" + (f"  {lab}" if last else ""), (xi, vi), xytext=(10 if last else -10, 0),
                        textcoords="offset points", ha="left" if last else "right", va="center", fontsize=9, color=INK)
    ax.set_xticks(x, [p[0] for p in parts])
    ax.tick_params(axis="x", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(-0.6, len(parts) - 1 + 1.5)
    allv = [p[1] for p in parts] + [p[2] for p in parts]
    span = max(allv) - min(allv)
    ax.set_ylim(min(allv) - span * 0.3, max(allv) + span * 0.3)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    _title(ax, title, sub)


def _weights(fig, rect, blend, sub="rank hill climbing, steps of 1/8"):
    """blend: list of (member, view, weight)."""
    ax = fig.add_axes(rect)
    ax.set_axis_off()
    bottom = 1.0
    for nm, view, w in blend:
        c = VIEW_COLORS[view]
        ax.add_patch(FancyBboxPatch((0.0, bottom - w + 0.006), 0.26, w - 0.012,
                                    boxstyle="round,pad=0,rounding_size=0.012", transform=ax.transAxes,
                                    facecolor=c, edgecolor="none"))
        ax.text(0.03, bottom - w / 2, f"{w:.1%}".replace(".0%", "%"), transform=ax.transAxes, va="center", fontsize=9.5,
                fontweight="bold", color=INK if c == YELLOW else "#ffffff")
        ax.text(0.31, bottom - w / 2, nm, transform=ax.transAxes, va="center", fontsize=9.2, color=INK)
        bottom -= w
    _title(ax, "The blend", sub)


def draw_part2(stats, names, corr, gains, smoothing, parts, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _corr(fig, [0.075, 0.43, 0.25, 0.28], names, corr)
    _gains(fig, [0.48, 0.43, 0.17, 0.28], gains)
    _smoothing(fig, [0.73, 0.43, 0.22, 0.28], smoothing)
    _slope(fig, [0.075, 0.05, 0.36, 0.25], parts)
    _weights(fig, [0.60, 0.05, 0.30, 0.25], blend)
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


# ---------------------------------------------------------------------------------------------- Part 3
DIV = ["#eb6834", "#f3a27d", "#f6c9b4", "#e9e8e3", "#b9d3f3", "#6da7ec", "#2a78d6"]   # less fraud ... more fraud


def _digits(fig, rect, dig):
    """dig: dict(cols=[...], rate=[[10 rates] per col], base=float)."""
    ax = fig.add_axes(rect)
    rate = np.asarray(dig["rate"], dtype=float)
    lift = rate / dig["base"] - 1
    lim = max(0.25, np.abs(lift).max())
    cmap = LinearSegmentedColormap.from_list("div", DIV)
    im = ax.imshow(lift, cmap=cmap, vmin=-lim, vmax=lim, aspect="auto")
    for i in range(lift.shape[0] + 1):
        ax.axhline(i - 0.5, color=PAGE, lw=3)
    for j in range(lift.shape[1] + 1):
        ax.axvline(j - 0.5, color=PAGE, lw=3)
    for i in range(lift.shape[0]):
        for j in range(lift.shape[1]):
            if abs(lift[i, j]) >= 0.10:
                ax.text(j, i, f"{rate[i, j]:.2%}", ha="center", va="center", fontsize=8.6,
                        color="#ffffff" if abs(lift[i, j]) > lim * 0.55 else INK)
    ax.set_xticks(range(10), [f".{d}" for d in range(10)])
    ax.set_yticks(range(len(dig["cols"])), dig["cols"])
    ax.tick_params(length=0, labelsize=10)
    ax.tick_params(axis="y", labelcolor=INK)
    ax.set_xlabel("first digit after the decimal point")
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=8, colors=MUTED, length=0)
    cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:+.0%}"))
    cb.set_label("fraud rate vs base", color=MUTED, fontsize=8.5)
    _title(ax, "The digits leak the label",
           f"fraud rate by the first decimal digit · base {dig['base']:.2%} · the same digits stand out in every column")


def _steps(fig, rect, steps, title, sub, xlabel):
    """steps: list of (label, auc); bars show the gain over the first step (× 1e-4)."""
    ax = fig.add_axes(rect)
    lab = [s for s, _ in steps]
    v = np.array([a for _, a in steps])
    yy = np.arange(len(lab))[::-1]
    gain = (v - v[0]) * 1e4
    ax.barh(yy, np.maximum(gain, 0.3), height=0.5, color=[AXIS] + [BLUE] * (len(v) - 1), zorder=2)
    for yv, g, a in zip(yy, gain, v):
        ax.text(max(g, 0) + gain.max() * 0.02, yv, f"{a:.5f}" + (f"   +{g:.1f}" if g > 0 else "   start"),
                va="center", fontsize=9.5, color=INK)
    ax.set_yticks(yy, lab)
    ax.tick_params(axis="y", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(0, gain.max() * 1.5)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(xlabel)
    _title(ax, title, sub)


def _families(fig, rect, fam, left="Part 2 features", right="+ digit keys", title="Every model family gains",
              sub="5-fold CV AUC, same folds · (gain × 1e-4)", dec=0):
    """fam: list of (model family, CV before, CV after), best first."""
    ax = fig.add_axes(rect)
    names = [f for f, _, _ in fam][::-1]
    a = np.array([x for _, x, _ in fam][::-1])
    b = np.array([x for _, _, x in fam][::-1])
    yy = np.arange(len(names))
    ax.hlines(yy, a, b, color=AXIS, lw=2, zorder=1)
    ax.scatter(a, yy, s=60, color=MUTED, edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.scatter(b, yy, s=70, color=BLUE, edgecolor=SURFACE, linewidth=2, zorder=3)
    for yv, x0, x1 in zip(yy, a, b):
        ax.text(x1 + 0.00012, yv, f"{x1:.5f}  (+{(x1 - x0) * 1e4:.{dec}f})", va="center", fontsize=9, color=INK)
    ax.set_yticks(yy, names)
    ax.tick_params(axis="y", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(min(a.min(), b.min()) - 0.0004, max(b) + 0.0019)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.{3 if dec == 0 else 4}f}"))
    if left:
        ax.text(a[-1], yy[-1] + 0.45, left, ha="center", fontsize=8.5, color=MUTED)
        ax.text(b[-1], yy[-1] + 0.45, right, ha="center", fontsize=8.5, color=INK2)
    ax.set_ylim(-0.6, len(names) - 0.2)
    _title(ax, title, sub)


def draw_part3(stats, digits, steps, families, parts, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _digits(fig, [0.155, 0.45, 0.40, 0.24], digits)
    _steps(fig, [0.72, 0.45, 0.17, 0.24], steps, "Four keys per number",
           "one 4-leaf LightGBM, fold 0", "AUC gain over per-value TE alone (× 1e-4)")
    _families(fig, [0.155, 0.06, 0.30, 0.25], families)
    _slope(fig, [0.53, 0.06, 0.25, 0.25], parts, "Part 1 → Part 3", "public LB follows the CV")
    _weights(fig, [0.84, 0.06, 0.13, 0.25], blend,
             f"rank hill climbing, steps of 1/{round(1 / min(w for _, _, w in blend))}")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


# ---------------------------------------------------------------------------------------------- Part 4
def _csweep(fig, rect, sw):
    """sw: dict(C=[...], series={label: [fold-0 AUC or None per C]}, ref=(label, auc))."""
    ax = fig.add_axes(rect)
    C = np.asarray(sw["C"], dtype=float)
    for (lab, au), col in zip(sw["series"].items(), [MUTED, BLUE]):
        au = np.array([np.nan if v is None else v for v in au], dtype=float)
        ok = ~np.isnan(au)
        ax.plot(C[ok], au[ok], color=col, lw=2, zorder=2)
        ax.scatter(C[ok], au[ok], s=46, color=col, edgecolor=SURFACE, linewidth=2, zorder=3)
        k = int(np.nanargmax(au))
        ax.annotate(f"{au[k]:.5f}", (C[k], au[k]), xytext=(0, 9 if col == BLUE else -15), textcoords="offset points",
                    ha="center", fontsize=9, fontweight="bold", color=INK)
        last = np.where(ok)[0][-1]
        ax.text(C[last] * 1.25, au[last], lab, va="center", fontsize=9, color=INK2)
    rl, rv = sw["ref"]
    ax.axhline(rv, color=AXIS, lw=1.2, ls=(0, (4, 3)), zorder=1)
    ax.text(C.max() * 1.25, rv, rl, va="center", fontsize=8.5, color=MUTED)
    ax.set_xscale("log")
    tk = [c for c in C if c != 0.005]
    ax.set_xticks(tk, [f"{c:g}" for c in tk])
    ax.minorticks_off()
    ax.set_xlim(C.min() / 1.6, C.max() * 9)
    vals = [v for a in sw["series"].values() for v in a if v is not None] + [rv]
    span = max(vals) - min(vals)
    ax.set_ylim(min(vals) - span * 0.08, max(vals) + span * 0.18)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    ax.set_xlabel("inverse L2 strength C (log scale)")
    ax.set_ylabel("fold-0 AUC")
    _title(ax, "One coefficient per value, fitted jointly",
           "logistic regression with every value and digit key one-hot, plus the TE logits")


def draw_part4(stats, steps, csweep, families, parts, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _steps(fig, [0.20, 0.45, 0.25, 0.24], steps, "Same models, more data",
           "public LB · step 2 changes only the test predictions", "public LB gain over Part 3 (× 1e-4)")
    _csweep(fig, [0.56, 0.45, 0.28, 0.24], csweep)
    _families(fig, [0.20, 0.06, 0.25, 0.25], families, left=None, right=None, dec=1,
              title="Steadier members", sub="grey = Part 3 member, blue = Part 4 · 5-fold CV (gain × 1e-4)")
    _slope(fig, [0.53, 0.06, 0.25, 0.25], parts, "Part 1 → Part 4", "public LB and 5-fold CV")
    _weights(fig, [0.84, 0.06, 0.13, 0.25], blend,
             f"rank hill climbing, steps of 1/{round(1 / min(w for _, _, w in blend))}")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


def _leaves(fig, rect, lv):
    """lv: dict(seeds={leaves: [5-fold CV per seed]}, bag={leaves: CV of the seed mean})."""
    ax = fig.add_axes(rect)
    ks = sorted(lv["seeds"], key=int)
    for i, k in enumerate(ks):
        s = lv["seeds"][k]
        ax.scatter(np.full(len(s), i) + np.linspace(-0.08, 0.08, len(s)), s, s=40, color=MUTED, edgecolor=SURFACE,
                   linewidth=1.5, zorder=3)
        if k in lv["bag"]:
            b = lv["bag"][k]
            ax.hlines(b, i - 0.22, i + 0.22, color=BLUE, lw=2.5, zorder=4)
            ax.text(i + 0.28, b, f"{b:.5f}", va="center", fontsize=9.5, fontweight="bold", color=INK)
    allv = [v for s in lv["seeds"].values() for v in s]
    ax.text(0.14, lv["seeds"][ks[0]][0], f"{lv['seeds'][ks[0]][0]:.5f}", va="center", fontsize=9, color=INK2)
    ax.set_xticks(range(len(ks)), [f"{k} leaves" for k in ks])
    ax.tick_params(axis="x", length=0, labelsize=10, labelcolor=INK)
    ax.set_xlim(-0.5, len(ks) - 0.1)
    span = max(allv) - min(allv)
    ax.set_ylim(min(allv) - span * 0.2, max(allv) + span * 0.35)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    _title(ax, "Three leaves, not four", "5-fold CV · grey = one seed, blue = mean of 3 seeds")


def _deltas(fig, rect, rows, title, sub):
    """rows: list of (label, auc); bars show the change from the first row (× 1e-4)."""
    ax = fig.add_axes(rect)
    ref = rows[0][1]
    lab = [r for r, _ in rows[1:]]
    d = np.array([(a - ref) * 1e4 for _, a in rows[1:]])
    yy = np.arange(len(lab))[::-1]
    ax.barh(yy, d, height=0.55, color=[BLUE if v > 0 else ORANGE for v in d], zorder=2)
    for yv, v in zip(yy, d):
        ax.text(v + (0.4 if v >= 0 else -0.4), yv, f"{v:+.1f}", va="center", ha="left" if v >= 0 else "right",
                fontsize=9.5, color=INK)
    ax.axvline(0, color=INK2, lw=1, zorder=3)
    ax.set_yticks(yy, lab)
    ax.tick_params(axis="y", length=0, labelsize=9.5, labelcolor=INK)
    ax.set_xlim(d.min() * 1.35, max(d.max(), 0) + 6)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(f"fold-0 AUC change vs the plain GLM ({ref:.5f}), × 1e-4")
    _title(ax, title, sub)


def _cvlb(fig, rect, pts):
    """pts: list of (label, cv, lb, group); group 'glm' = contains the plain GLM."""
    ax = fig.add_axes(rect)
    for lab, cv, lb, g in pts:
        col = ORANGE if g == "glm" else BLUE
        big = "Part 5" in lab
        ax.scatter(cv, lb, s=150 if big else 70, color=col, edgecolor=INK if big else SURFACE, linewidth=1.5 if big else 2,
                   zorder=4 if big else 3)
    off = {"Part 3 refit": (8, -4), "Part 4": (-8, 6), "5 seeds": (7, 7), "+ EBM i25": (8, -3),
           "3 leaves": (4, -10), "plain GLM swap": (8, 0), "CV best, refit": (-8, 8),
           "CV best, refit + folds": (-8, -12), "half plain GLM": (-8, -7), "CV best without it (Part 5)": (10, 0)}
    for lab, cv, lb, g in pts:
        dx, dy = off.get(lab, (8, 0))
        ax.annotate(lab, (cv, lb), xytext=(dx, dy), textcoords="offset points", fontsize=9, color=INK,
                    ha="left" if dx > 0 else "right", va="center")
    cvs, lbs = [p[1] for p in pts], [p[2] for p in pts]
    ax.set_xlim(min(cvs) - 0.00006, max(cvs) + 0.00005)
    ax.set_ylim(min(lbs) - 0.00007, max(lbs) + 0.00007)
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.5f}"))
    ax.set_xlabel("5-fold CV AUC (700k rows)")
    ax.set_ylabel("public LB (60k rows)")
    from matplotlib.lines import Line2D
    ax.legend(handles=[Line2D([], [], ls="", marker="o", ms=8, color=c, label=t) for c, t in
                       [(BLUE, "without the plain GLM"), (ORANGE, "with the plain GLM")]],
              loc="lower left", frameon=False, fontsize=9, handletextpad=0.2)
    _title(ax, "The CV and the public board disagree", "every blend we submitted after Part 3")


def _refit(fig, rect, rows):
    """rows: list of (held-out fold, fold average, refit, rank mean); bars = change vs the fold average."""
    ax = fig.add_axes(rect)
    x = np.arange(len(rows))
    b = np.array([(r[2] - r[1]) * 1e4 for r in rows])
    c = np.array([(r[3] - r[1]) * 1e4 for r in rows])
    ax.bar(x - 0.19, b, width=0.34, color=MUTED, zorder=2)
    ax.bar(x + 0.19, c, width=0.34, color=BLUE, zorder=2)
    for xi, v in zip(np.r_[x - 0.19, x + 0.19], np.r_[b, c]):
        ax.text(xi, v + (0.15 if v >= 0 else -0.15), f"{v:+.1f}", ha="center", va="bottom" if v >= 0 else "top",
                fontsize=8.8, color=INK)
    ax.axhline(0, color=INK2, lw=1, zorder=3)
    ax.set_xticks(x, [f"held out\n{r[0]}" for r in rows])
    ax.tick_params(axis="x", length=0, labelsize=9.5, labelcolor=INK)
    ax.set_ylim(min(b.min(), c.min()) - 0.8, max(b.max(), c.max()) + 0.8)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_ylabel("AUC change vs fold average (× 1e-4)")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=MUTED, label="refit on all rows"), Patch(color=BLUE, label="rank mean of both")],
              loc="lower left", frameon=False, fontsize=9)
    _title(ax, "Refit or fold average?", "3-leaf trees, 3 seeds, scored on a held-out fold")


def draw_part5(stats, leaves, glm_design, cvlb, refit, blend, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _leaves(fig, [0.06, 0.45, 0.28, 0.24], leaves)
    _deltas(fig, [0.60, 0.45, 0.30, 0.24], glm_design, "A plain GLM is the strongest GLM",
            "logistic regression on the TE logits · what each extra block does")
    _cvlb(fig, [0.06, 0.06, 0.40, 0.27], cvlb)
    _refit(fig, [0.53, 0.06, 0.22, 0.25], refit)
    _weights(fig, [0.83, 0.06, 0.14, 0.25], blend, "best CV without the plain GLM, 1/9 steps")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


def _cvlb6(fig, rect, pts):
    """pts: list of (label, cv, lb, group); group 'glm' = a GLM-centred submission, 'mix' = several families."""
    from matplotlib.lines import Line2D
    ax = fig.add_axes(rect)
    for lab, cv, lb, g in pts:
        big = lab.startswith("Part 6")
        ax.scatter(cv, lb, s=170 if big else 70, color=ORANGE if g == "glm" else BLUE,
                   edgecolor=INK if big else SURFACE, linewidth=1.6 if big else 2, zorder=4 if big else 3)
    off = {"Part 4": (-8, 0), "Part 5": (-8, 6), "CV-best with plain GLM": (8, -2), "plain GLM swap": (-8, 0),
           "A: GLM boost alone": (-8, 6), "D: same, rank-mean test": (-8, -6), "B: 0.3 Part 5 + 0.7 GLM boost": (-8, 0),
           "C: stack, stopped early": (-8, 4), "E: stack, 72 members": (-6, -10), "Part 6: stack, converged": (-10, 10)}
    for lab, cv, lb, g in pts:
        dx, dy = off.get(lab, (8, 0))
        ax.annotate(lab, (cv, lb), xytext=(dx, dy), textcoords="offset points", fontsize=9,
                    color=INK, fontweight="bold" if lab.startswith("Part 6") else "normal",
                    ha="left" if dx > 0 else "right", va="center")
    cvs, lbs = [p[1] for p in pts], [p[2] for p in pts]
    ax.set_xlim(min(cvs) - 0.00028, max(cvs) + 0.00008)
    ax.set_ylim(min(lbs) - 0.00006, max(lbs) + 0.00008)
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.5f}"))
    ax.set_xlabel("5-fold CV AUC (nested for the stacks)")
    ax.set_ylabel("public LB")
    ax.legend(handles=[Line2D([], [], ls="", marker="o", ms=8, color=c, label=t) for c, t in
                       [(BLUE, "several model families"), (ORANGE, "a GLM-centred model on its own")]],
              loc="upper left", frameon=False, fontsize=9, handletextpad=0.2)
    _title(ax, "The public board rewards variety", "every submission since Part 4")


def _coefs(fig, rect, coef, ours):
    """coef: list of (member, stack coefficient); ours: names trained by us (blue), the rest public (aqua)."""
    ax = fig.add_axes(rect)
    lab = [n.replace("_", " ").replace(":", ": ") for n, _ in coef]
    v = np.array([c for _, c in coef])
    yy = np.arange(len(v))[::-1]
    ax.barh(yy, v, height=0.6, color=[BLUE if n in ours else AQUA for n, _ in coef], zorder=2)
    for yv, c in zip(yy, v):
        ax.text(c + (0.008 if c >= 0 else -0.008), yv, f"{c:+.2f}", va="center", ha="left" if c >= 0 else "right",
                fontsize=9, color=INK)
    ax.axvline(0, color=INK2, lw=1, zorder=3)
    ax.set_yticks(yy, [s if len(s) <= 30 else s[:29] + "…" for s in lab])
    ax.tick_params(axis="y", length=0, labelsize=9, labelcolor=INK)
    ax.set_xlim(v.min() - 0.09, v.max() + 0.09)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel("coefficient on the member's rank-logit")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color=BLUE, label="ours"), Patch(color=AQUA, label="public notebooks")],
              loc="lower right", frameon=False, fontsize=9)
    _title(ax, "Who the stack listens to", "10 largest of 62 coefficients · negative = corrects the others")


def draw_part6(stats, ladder, cvlb, nested, coef, ours, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _ladder(fig, [0.20, 0.45, 0.26, 0.24], ladder, "Start the trees from the GLM",
            "one model each, 5-fold out-of-fold AUC", "AUC gain over the GLM alone (× 1e-4)")
    _cvlb6(fig, [0.56, 0.45, 0.41, 0.24], cvlb)
    _ladder(fig, [0.26, 0.06, 0.20, 0.25], nested, "Pooling public members, nested",
            "the stacker never sees the fold it is scored on", "nested CV gain over our 10 members (× 1e-4)")
    _coefs(fig, [0.70, 0.06, 0.27, 0.25], coef, ours)
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


def _p7gains(fig, rect, rows):
    """rows: (candidate, nested CV gain × 1e-4 when added to the stack); the last row is highlighted."""
    ax = fig.add_axes(rect)
    lab = [r for r, _ in rows]
    v = np.array([g for _, g in rows])
    yy = np.arange(len(v))[::-1]
    ax.barh(yy, np.maximum(v, 0.03), height=0.55, color=[AXIS] * (len(v) - 2) + [AQUA, ORANGE], zorder=2)
    for yv, g in zip(yy, v):
        ax.text(g + v.max() * 0.02, yv, f"+{g:.2f}", va="center", fontsize=9.5, color=INK)
    ax.set_yticks(yy, lab)
    ax.tick_params(axis="y", length=0, labelsize=9.8, labelcolor=INK)
    ax.set_xlim(0, v.max() * 1.18)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel("nested CV gain when added to the Part 6 stack (× 1e-4)")
    _title(ax, "One new member does the work", "each candidate added alone to the same 62 members")


def _memberMap(fig, rect, pts):
    """pts: (name, OOF AUC, rank correlation with the GLM boost bag)."""
    ax = fig.add_axes(rect)
    others = [p for p in pts if p[0] != "gpt2" and not p[0].startswith("gbn")]
    ax.scatter([p[2] for p in others], [p[1] for p in others], s=26, color=MUTED, edgecolor=SURFACE, linewidth=0.8, zorder=2)
    for p in pts:
        if p[0].startswith("gbn"):
            ax.scatter(p[2], p[1], s=60, color=AQUA, edgecolor=SURFACE, linewidth=1.2, zorder=3)
    g = next(p for p in pts if p[0] == "gpt2")
    ax.scatter(g[2], g[1], s=170, color=ORANGE, edgecolor=INK, linewidth=1.5, zorder=4)
    ax.annotate("GPT-2 target-first", (g[2], g[1]), xytext=(12, 2), textcoords="offset points", fontsize=9.5,
                fontweight="bold", color=INK, va="center")
    m = max((p for p in pts if p[0].startswith("gbn")), key=lambda p: p[2])
    ax.annotate("residual MLPs", (m[2], m[1]), xytext=(-10, 10), textcoords="offset points", fontsize=9, color=INK, ha="right")
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_ylim(0.9472, 0.9518)
    ax.yaxis.set_major_locator(plt.MaxNLocator(5))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.4f}"))
    ax.set_xlabel("rank correlation with the GLM boost bag  (← more different)")
    ax.set_ylabel("OOF AUC")
    _title(ax, "Different and strong at once", "every stack member · the others line up: more different means weaker")


def _foldgain(fig, rect, f0, f1):
    ax = fig.add_axes(rect)
    d = (np.array(f1) - np.array(f0)) * 1e4
    x = np.arange(len(d))
    ax.bar(x, d, width=0.55, color=[BLUE if v > 0 else ORANGE for v in d], zorder=2)
    for xi, v in zip(x, d):
        ax.text(xi, v + (0.2 if v >= 0 else -0.2), f"{v:+.1f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=9.5, color=INK)
    ax.axhline(0, color=INK2, lw=1, zorder=3)
    ax.set_xticks(x, [f"fold {i}" for i in x])
    ax.tick_params(axis="x", length=0, labelsize=9.5, labelcolor=INK)
    ax.set_ylim(min(d.min(), 0) - 1, d.max() + 1.2)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_ylabel("nested AUC change (× 1e-4)")
    _title(ax, "Fold by fold", "140k rows each, and still from 0 to +8")


def _noise(fig, rect, d, obs):
    ax = fig.add_axes(rect)
    d = np.asarray(d)
    ax.hist(d, bins=34, color=SEQ[3], edgecolor=SURFACE, linewidth=0.6, zorder=2)
    top = ax.get_ylim()[1]
    ax.axvline(0, color=INK2, lw=1, zorder=3)
    ax.axvline(d.mean(), color=BLUE, lw=2, zorder=4)
    ax.axvline(obs, color=ORANGE, lw=2.5, zorder=5)
    ax.text(min(obs, 0) - 0.3, top * 0.97, f"the public board\n{obs:+.1f}", ha="right", va="top", fontsize=9.3, color=INK, fontweight="bold")
    ax.text(d.mean() + 0.15, top * 0.97, f"the CV says\n{d.mean():+.1f} ± {d.std():.1f}", ha="left", va="top", fontsize=9.3, color=INK)
    ax.yaxis.set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel("AUC gain on a random 60,000 rows (× 1e-4)")
    _title(ax, "Why the public board cannot see it", "500 draws of out-of-fold rows, the size of the public board")


def draw_part7(stats, gains, points, folds0, folds1, noise, observed, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _p7gains(fig, [0.22, 0.45, 0.28, 0.24], gains)
    _memberMap(fig, [0.60, 0.45, 0.37, 0.24], points)
    _foldgain(fig, [0.06, 0.06, 0.38, 0.25], folds0, folds1)
    _noise(fig, [0.53, 0.06, 0.44, 0.25], noise, observed)
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


def _p8curve(fig, rect, curve):
    """curve: (weight of the Part 7 stack, nested CV, public LB or None); both shown as gain over weight 0."""
    ax = fig.add_axes(rect)
    w = [c[0] for c in curve]
    cv = [(c[1] - curve[0][1]) * 1e4 for c in curve]
    ax.plot(w, cv, color=BLUE, lw=2, marker="o", ms=7, zorder=3)
    pts = [(c[0], (c[2] - curve[0][2]) * 1e4) for c in curve if c[2] is not None]
    ax.plot([p[0] for p in pts], [p[1] for p in pts], color=ORANGE, lw=2, marker="o", ms=7, zorder=3)
    ax.text(0.27, cv[1] + 0.3, "nested CV", color=INK, fontsize=9.5, ha="right")
    ax.text(0.72, pts[2][1] - 0.7, "public LB", color=INK, fontsize=9.5, ha="center")
    ax.axvline(0.5, color=INK2, lw=1, ls=(0, (3, 3)), zorder=1)
    ax.text(0.51, 0.25, "Part 8", fontsize=9.5, fontweight="bold", color=INK)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1], ["0\n(Part 6)", "0.25", "0.5", "0.75", "1\n(Part 7)"])
    ax.tick_params(axis="x", length=0, labelsize=9.5, labelcolor=INK)
    ax.yaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.set_ylim(-0.4, 5)
    ax.set_xlabel("weight of the Part 7 stack in the rank mean")
    ax.set_ylabel("AUC gain over Part 6 (× 1e-4)")
    _title(ax, "Blending two of our own stacks", "the public board peaks in the middle, the CV at the right")


def _p8bars(fig, rect, rows, title, sub, xlabel):
    ax = fig.add_axes(rect)
    lab = [r for r, _ in rows]
    v = np.array([g for _, g in rows])
    yy = np.arange(len(v))[::-1]
    ax.barh(yy, v, height=0.55, color=[BLUE if g > 0 else ORANGE for g in v], zorder=2)
    span = max(abs(v).max(), 0.1)
    for yv, g in zip(yy, v):
        ax.text(g + (0.03 if g >= 0 else -0.03) * span * 2, yv, f"{g:+.2f}", va="center", ha="left" if g >= 0 else "right", fontsize=9.3, color=INK)
    ax.axvline(0, color=INK2, lw=1, zorder=3)
    ax.set_yticks(yy, lab)
    ax.tick_params(axis="y", length=0, labelsize=9.5, labelcolor=INK)
    ax.set_xlim(min(v.min(), 0) - span * 0.32, max(v.max(), 0) + span * 0.32)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(xlabel)
    _title(ax, title, sub)


def _p8dist(fig, rect, names, dist):
    ax = fig.add_axes(rect)
    d = np.asarray(dist, dtype=float)
    ax.imshow(d, cmap=LinearSegmentedColormap.from_list("seq", SEQ[:6]), aspect="auto", vmin=0, vmax=d.max())
    for i in range(len(names)):
        for j in range(len(names)):
            if i != j:
                ax.text(j, i, f"{d[i, j]:.1f}", ha="center", va="center", fontsize=9.5, color="#ffffff" if d[i, j] > d.max() * 0.62 else INK)
    ax.set_xticks(range(len(names)), [n.replace(" (Nov 4)", "") for n in names], rotation=18, ha="right")
    ax.set_yticks(range(len(names)), [n.replace(" (Nov 4)", "") for n in names])
    ax.tick_params(length=0, labelsize=9.3, labelcolor=INK)
    for s in ax.spines.values():
        s.set_visible(False)
    _title(ax, "The top of the board is one family", "1 − rank correlation between files (× 1e-4)")


def draw_part8(stats, curve, pool, settings, names, dist, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _p8curve(fig, [0.06, 0.45, 0.40, 0.24], curve)
    _p8dist(fig, [0.60, 0.46, 0.34, 0.23], names, dist)
    _p8bars(fig, [0.20, 0.06, 0.26, 0.25], pool, "More members: small, then nothing", "added to the 65 members, alone", "nested CV gain (× 1e-4)")
    _p8bars(fig, [0.70, 0.06, 0.26, 0.25], settings, "Settings the stack does not want", "changed from logit, C = 0.01, plain stack", "nested CV gain (× 1e-4)")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


def draw_part9(stats, cv_rows, lb_rows, tenf_rows, pseudo_rows, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _p8bars(fig, [0.22, 0.45, 0.26, 0.24], cv_rows, "What the CV promised", "nested CV change over Part 8", "× 1e-4")
    _p8bars(fig, [0.70, 0.45, 0.26, 0.24], lb_rows, "What the public board paid", "public LB change over Part 8 (0.95235)", "× 1e-4")
    _p8bars(fig, [0.22, 0.06, 0.26, 0.25], tenf_rows, "10 folds lift the out-of-fold score only", "single-model OOF gain over 5 folds; the test side was a full refit already", "× 1e-4")
    _p8bars(fig, [0.70, 0.06, 0.26, 0.25], pseudo_rows, "Soft labels in the encodings", "OOF change, validation labels unused, fold 0 and fold 1", "× 1e-4")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


def draw_part10(stats, ladder, names, dist, title, subtitle, path=None):
    _style()
    fig = plt.figure(figsize=(16, 9.2), dpi=110)
    fig.text(0.035, 0.945, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.898, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.74, tw, 0.125, lab, val, note)
    _ladder(fig, [0.20, 0.09, 0.28, 0.48], ladder, "The series on the public board", "each row is a scored file", "public LB gain over Part 6 (× 1e-4)")
    _p8dist(fig, [0.62, 0.11, 0.32, 0.46], names, dist)
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


def draw_part11(stats, ladder, nested, sources, title, subtitle, path=None, src_title="200 members, 12 sources"):
    _style()
    fig = plt.figure(figsize=(16, 12.4), dpi=110)
    fig.text(0.035, 0.962, title, fontsize=23, fontweight="bold", color=INK, va="center")
    fig.text(0.035, 0.928, subtitle, fontsize=12.5, color=INK2, va="center")
    tw, gap = 0.2175, 0.0125
    for i, (lab, val, note) in enumerate(stats):
        _tile(fig, 0.035 + i * (tw + gap), 0.80, tw, 0.095, lab, val, note)
    _ladder(fig, [0.20, 0.45, 0.28, 0.25], ladder, "The series on the public board", "each row is a scored file", "public LB gain over Part 6 (× 1e-4)")
    _ladder(fig, [0.70, 0.45, 0.20, 0.25], nested, "Nested CV of the stacks", "the stacker never sees the fold it is scored on", "gain over Part 6 (× 1e-4)")
    _p8bars(fig, [0.20, 0.06, 0.28, 0.27], sources, src_title, "same five folds, near-copies removed", "members")
    if path:
        fig.savefig(path, dpi=110, bbox_inches="tight", pad_inches=0.3)
    return fig


draw_part11(
    stats=[('Public LB', '0.95285', '+0.00046 over Part 10'), ('Nested CV', '0.95245', '+0.00039 over Part 7'), ('Members', '200', 'from 12 sources, same folds'), ('Public files used', '0', 'all out-of-fold stacks')],
    ladder=[('Part 6', 0.95215), ('Part 7', 0.95222), ('Part 8', 0.95235), ('Part 10', 0.95239), ('wide stack', 0.95267), ('Part 11', 0.95285)],
    nested=[('Part 6', 0.951632), ('Part 9', 0.951987), ('Part 7', 0.952057), ('Part 11', 0.952452), ('wide stack', 0.952504)],
    sources=[('goodpjw2008', 70.0), ('alexchilton', 36.0), ('sameershaikh27', 21.0), ('malyshevdanil', 21.0), ('lazyyeou', 17.0), ('sadamtorres', 13.0), ('megayak', 7.0), ('wguesdon', 7.0), ('masayakawamata', 3.0), ('queyrusi', 2.0), ('nilanjansaha123', 2.0), ('yekenot', 1.0)],
    title="S6E11 Credit Card Fraud  ·  Part 11",
    subtitle="Every published member on one split, then 30% of a stack that sees them differently")
plt.show()

# S6E11 Part 12 | Wide Stack + 2nd LM | LB 0.95298

> **This notebook grows.** Every part is a new version of this same notebook: one new idea on top of the
> previous ones, measured on the same folds, with the whole history kept below — what was tried, what each
> idea was worth, and what did not work. If you upvoted an earlier part, thank you; the newest part is
> always at the top.

| part | idea | 5-fold CV | public LB |
|---|---|---|---|
| **Part 1** | every value of every column has its own fraud rate (out-of-fold TE); 4-leaf LightGBM, GLMs, RealMLP, logistic stack | 0.94892 | 0.94928 |
| **Part 2** | raw-view trees next to target-encoded ones, TE smoothing as an extra view, rank hill climbing | 0.94923 | 0.94945 |
| **Part 3** | the generator writes numbers digit by digit: target-encoded cents, int(x), int(x/10), int(10x) and leading digits; every family retrained; EBMs | 0.95073 | 0.95113 |
| **Part 4** | refit every member on all training rows, three-seed bags, a GLM fitting one-hot values and digit keys jointly with the encodings | 0.95079 | 0.95154 |
| **Part 5** | 3-leaf trees; a plain GLM on the encodings (CV 0.95051) that the public board rejects; the best-CV blend without it; refit vs fold average on held-out folds | 0.95092 | 0.95160 |
| **Part 6** | trees that start from the GLM's logit (5-seed bag, CV 0.95151); 55 public same-fold members pooled with ours in a converged logistic stack, scored by nested CV | 0.95163 | 0.95215 |
| **Part 7** | a target-first GPT-2 classifier and residual MLPs join the stack; why one public score cannot confirm a gain of this size | 0.95206 | 0.95222 |
| **Part 8** | 28 more public members and TabPFN add nothing; a rank mean of the stacks with and without GPT-2 peaks on the public board; a second GPT-2 seed measured | 0.95199 | 0.95235 |
| **Part 9** | average every refit on the test side only; 10-fold members, soft-label encodings, a digit CNN and public-file fusions all fail to raise the public score | 0.95199 | 0.95236 |
| **Part 10** | a rank mean of Part 9 and amanatar's public Sovereign Fusion file; the first public file in this series, with no CV behind it | 0.95199 | 0.95239 |
| **Part 11** | 200 same-fold members from 12 sources in one converged stack, blended 70/30 with Part 9; a correction of Part 9's verdict on 10-fold members | 0.95245 | 0.95285 |
| **Part 12** (this version) | megayak's target-first language model joins the wide stack as its 201st member, out of fold instead of as a public file | 0.95252 | 0.95298 |

## What is new in Part 12

**Public LB 0.95285 (our best, +0.00046 over Part 10) · nested CV 0.95245 (+0.00039 over Part 7).** No public submission
file: every part of this file has out-of-fold predictions and a nested score.

1. **A much wider pool** (11.1). [Sameer Shaikh's Wide Pooled Stack](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-wide-pooled-stack-every-bank-found)
   showed the pool was far from saturated. We rebuilt it with every bank we could find plus all our members: 200 members from
   12 sources, nested CV 0.95250, public 0.95267.
2. **Blend it with Part 9** (11.2). 70% wide stack + 30% Part 9 keeps most of the nested CV and scores **0.95285**: this notebook's
   `submission.csv`. Part 9 is 95.0e-4 away from the wide stack, so the two are strong and different.
3. **A correction** (11.3). Part 9 called 10-fold members a measurement illusion. That was too strong, and
   the wide stack shows why.

| | what it is | nested CV | public LB |
|---|---|---|---|
| Part 10 | Part 9 + a public file | none | 0.95239 |
| wide stack | 200 members, converged logistic stack | 0.95250 | 0.95267 |
| **Part 11 blend** | 0.7 × wide stack + 0.3 × Part 9 | **0.95245** | **0.95285** |

**Part 10 in one paragraph:** a rank mean of Part 9 and amanatar's public Sovereign Fusion file, the first public file in this
series, with no CV behind it. LB 0.95239.

**Part 9 in one paragraph:** averaging every refit on the test side gave +0.00001; 10-fold members raised only the
out-of-fold score, soft-label encodings lost 0.0006-0.0011, and a digit CNN and more members lowered the public score.
Nested CV 0.95199, LB 0.95236.

**Part 8 in one paragraph:** the top of the board is public files mixing each other; 28 more members and TabPFN add
nothing; a rank mean of the stack with GPT-2 and the stack without it peaks on the public board. Nested CV
0.95199, LB 0.95235.

**Part 7 in one paragraph:** tree variants stopped adding to the stack; residual MLPs added a little, and a
target-first GPT-2 classifier added +0.00041 in nested CV. Nested CV 0.95206, LB 0.95222.

**Part 6 in one paragraph:** trees that start from a GLM's logit (5-seed bag, CV 0.95151), 55 public same-fold
members pooled with ours, and a logistic stack solved to convergence so that any machine reproduces it. Nested
CV 0.95163, LB 0.95215.

**Part 5 in one paragraph:** 3-leaf trees beat 4 on five folds; a plain GLM on the encodings scored the best
CV of any GLM but lowered every public score it was part of; refits vs fold averages checked on held-out
folds. CV 0.95092, LB 0.95160.

**Part 4 in one paragraph:** the same signal with less noise: test predictions from one more model per member
trained on every training row (the Part 3 blend went from LB 0.95113 to 0.95132), three-seed bags, and a GLM
that fits one-hot values and digit keys jointly with the encodings. CV 0.95079, LB 0.95154.

**Part 3 in one paragraph:** the synthetic generator writes numbers digit by digit and the digits carry the
label; four target-encoded keys per float column (cents, `int(x)`, `int(x/10)`, `int(10x)`) lift every model
family by 0.0008-0.0017. CV 0.95073, LB 0.95113.

**Part 2 in one paragraph:** shallow trees on the *raw* columns next to the target-encoded ones; the TE
smoothing weight as a free extra view; rank hill climbing. Six members: CV 0.94923, LB 0.94945.

**Part 1 in one paragraph:** the original 20k rows are almost a logistic model; out-of-fold target
encoding of *every* column (each distinct number is a category) lifts every model family; with those
encodings the best trees have 4 leaves. Five models stacked: CV 0.94892, LB 0.94928.

## Credits and thanks

| Whose work | What we took from it |
|---|---|
| **Udit Jain** — [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) | The original dataset this competition was generated from (Part 1, section 2; value counts in Part 2). |
| **aphoristicartist** — [S6E11 Fraud: EDA + 3 GBDTs + Blend](https://www.kaggle.com/code/aphoristicartist/s6e11-fraud-eda-3-gbdts-blend-lb-0-9468) | The day-one baseline we measured against (3 GBDTs, OOF ≈ 0.9463, LB 0.9468), the same fold scheme, and the reminder that the public board scores only ~20% of the test rows. |
| **Will** — [S6E11 Refined LightGBM](https://www.kaggle.com/code/wguesdon/s6e11-refined-lightgbm) | A 15-leaf, heavily regularised LightGBM — the first hint that small trees win here. We pushed it down to 4 leaves. |
| **Vladimir Demidov** — [PS S6E10 RealMLP · PyTabKit](https://www.kaggle.com/code/yekenot/ps-s6-e10-realmlp-pytabkit) | The RealMLP hyper-parameters (from last month's competition), used here almost unchanged. |
| **David Holzmüller et al.** — [pytabkit](https://github.com/dholzmueller/pytabkit) | RealMLP itself. |
| **Ravi Ramakrishnan** — [PlaygroundS6E11 Public EDA V1](https://www.kaggle.com/code/ravi20076/playgrounds6e11-public-eda-v1) | Target encodings of rounded values: the idea behind Part 3's `int(x)` and `int(x/10)` keys. |
| **alex chilton** — [Credit Card Fraud: Metric, EDA, 4-Model Ensemble](https://www.kaggle.com/code/alexchilton/credit-card-fraud-metric-eda-4-model-ensemble) | The first decimal digit and rounded values as features (Part 3). We measured the two-decimal `cents` key next to them. |
| **InterpretML** — [Explainable Boosting Machine](https://github.com/interpretml/interpret) | The EBM in Part 3. |
| **Bulşah Keçici** — [S6E11 \| Value TE + Digit & Token Keys \| LR Stack](https://www.kaggle.com/code/bulsahkecici/s6e11-value-te-digit-token-keys-lr-stack) | A logistic regression on the encoding logits without extra blocks (their CV 0.95035): the plain GLM of Part 5. The GPT-2 token keys we tested in 5.3. |
| **Akmal Xodarev** — [S6E11 \| Token Keys, EBM, Full Refit \| CV 0.95079](https://www.kaggle.com/code/busyaprime/s6e11-token-keys-ebm-full-refit-cv-0-95079) | 3 leaves beating 4 on five folds (Part 5). |
| **Masaya Kawamata** — [S6E11 \| Five XGBoost folds beat one full fit](https://www.kaggle.com/code/masayakawamata/s6e11-five-xgboost-folds-beat-one-full-fit); **Georgy Mamarin** — [S6E11 Fraud \| LB 0.95095 \| Shallow Digit Models](https://www.kaggle.com/code/georgymamarin/s6e11-fraud-lb-0-95095-shallow-digit-models) | Held-out comparisons of fold averages and full refits, which made us re-check Part 4's refit (5.4). Georgy also asked about column sampling in the comments. |

| **megayak (Artificial Idiocy Lab)** — [S6E11 \| Per-Value + Digit TE \| Shallow-Tree Stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack) | **Boosting from the GLM** (Part 6), the joint keys, the token-pooling GLM and the value counts of every key. `gboost.py` in 6.2 is their notebook's code (Apache 2.0) with our seed and option driver around it; their published predictions are 7 of the stack's members. |
| **lazyyeou** — [S6E11 \| Digit-Trie TE \| Joint GLM & Nested Stack](https://www.kaggle.com/code/lazyyeou/s6e11-digit-trie-te-joint-glm-nested-stack); **Nilanjan_Saha123** — [96-Key Digit-Trie + GPT-2 \| Full-Refit+Hybrid](https://www.kaggle.com/code/nilanjansaha123/96-key-digit-trie-gpt-2-full-refit-hybrid) | Pooling public same-fold predictions into a nested stack; 5 members each (XGBoost depth 2 and 4-leaf bags, 96-key GLMs). |
| **alex chilton** — [Credit Card Fraud \| LB 0.9521 \| Shallow-Tree Stack](https://www.kaggle.com/code/alexchilton/credit-card-fraud-lb-0-9521-shallow-tree-stack) | A published bank of 37 models with a manifest; 26 of them are stack members. |
| **Malyshev Danil** — [S6E11 pooled stack v9](https://www.kaggle.com/datasets/malyshevdanil/s6e11-pooled-stack-v9) | Their own 10 models of a 32-model bank (EBMs, token GLM, refits) as stack members. |
| **Vladimir Demidov** — [RealMLP · PyTorch](https://www.kaggle.com/code/yekenot/ps-s6-e11-realmlp-pytorch), [RealMLP · PyTabKit](https://www.kaggle.com/code/yekenot/ps-s6-e11-realmlp-pytabkit) | Two RealMLPs as stack members. |
| **masayakawamata** — [S6E11 \| GPT-2 Target-First Classifier \| CV 0.9500](https://www.kaggle.com/code/masayakawamata/s6e11-gpt-2-target-first-classifier-cv-0-9500) | The GPT-2 member of Part 7: about 120 GPU hours of training and scoring, published as predictions (CC BY 4.0). We use `oof.npy` and `test.npy` as they are. |
| **Chris Deotte** — [1st Place - TFMs and LLMs with tabular data!](https://www.kaggle.com/competitions/playground-series-s6e10/discussion/754683) | The target-first generative classifier recipe behind that member. |
| **Mohammed Majeed** — [How Far Public Same-Split OOFs Go](https://www.kaggle.com/code/mojjeed/s6e11-public-oof-stack-write-up-cv-0-95210); **Sadam Torres** — [TabPFN-3.5 + GBDT OOF/test pairs](https://www.kaggle.com/datasets/sadamtorres/ps-s6e11-tabpfn-gbdt-oof-test-pairs); **Will** — [Tree Ensemble Predictions](https://www.kaggle.com/datasets/wguesdon/s6e11-tree-ensemble-predictions) | The source-by-source ladder and the published members we tested in 8.2. |
| **Areeb Ali** — [GLM-Boost + Residual MLP Pool](https://www.kaggle.com/code/areebbb/s6e11-glm-boost-residual-mlp-pool-lb-0-95231) | The hill-and-stack mix we measured in 8.2, and the file whose make-up led to 8.3. |
| **Sameer Shaikh** — [Wide Pooled Stack, Every Bank Found](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-wide-pooled-stack-every-bank-found) | The wide pool of Part 11, its member loader, and 21 of its members. **robotoil** — [Pool Stacking v4](https://www.kaggle.com/code/robotoil/s6e11-fraud-pool-stacking-v4) measured blends of pools with and without GPT-2. |
| **Aman Atar** — [S6E11 Sovereign Fusion Solution](https://www.kaggle.com/code/amanatar/s6e11-sovereign-fusion-solution); **Malyshev Danil** — [Getting Started, version 13](https://www.kaggle.com/code/malyshevdanil/predicting-credit-card-fraud-getting-started) | The public file in Part 10 (50% of its submission) and the idea of blending it with this series. |
| **queyrusi** — [Fraud: a shared digit CNN](https://www.kaggle.com/code/queyrusi/fraud-a-shared-digit-cnn) | The digit CNN and tree predictions tested in 9.4. Sadam Torres's notebook gave the 10-fold idea of 9.2. |
| **Hemingweb** — [Deep EDA and list of all findings](https://www.kaggle.com/competitions/playground-series-s6e11/discussion/754727) | The label-first, temperature-0.7 picture of the generator that megayak built on. |

Our own code is everything except `gboost.py`'s feature and encoding part (megayak, above); ideas from the
notebooks above are marked where they are used. Thank you!

## Setup and data

Inputs: the competition data, the original dataset `uditjain13/credit-card-fraud-detection-2026`, and our
dataset `goodpjw2008/s6e11-fraud-series-artifacts` (RealMLP predictions trained on a GPU, EBM predictions
trained on 16 CPU cores, and the bagged and refit members of Parts 4 to 6). Part 6 also reads the published
predictions of seven public notebooks and one public dataset (listed in 6.4 and 7.3; add them as inputs when you copy
this notebook). **CPU only**, about 5 hours for every part together. Set `RETRAIN_NN = True` (and turn the GPU
on) to retrain the RealMLPs, `RETRAIN_EBM = True` (Part 3) to retrain the EBMs, `RETRAIN_P4 = True` (Part 4),
`RETRAIN_P5 = True` (Part 5) and `RETRAIN_P6 = True` (Part 6) to retrain the seed bags, the GLMs and every
refit here (several more hours).

24 features: 13 numeric, 6 yes/no flags, 5 categoricals; no missing values. About 1 row in 60 is fraud.
Every model in every part uses the same `StratifiedKFold(5, shuffle=True, random_state=42)` split.

In [ ]:
import subprocess, sys
from importlib.metadata import version

# The LB 0.94945 submission was trained with XGBoost 3.2.0. The newer release on Kaggle grows slightly
# different trees from the same data (xgb_te_d2 CV 0.94872 instead of 0.94881), so pin it to reproduce.
if version("xgboost") != "3.2.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "xgboost==3.2.0"], check=True)

import time
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from scipy import sparse
from scipy.stats import rankdata, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler

INPUT = os.environ.get("INPUT_DIR", "/kaggle/input")


def find(name, hint=""):
    hits = sorted(glob.glob(os.path.join(INPUT, "**", name), recursive=True))
    return ([h for h in hits if hint in h] or hits)[0]


SMOKE = os.environ.get("SMOKE") == "1"     # tiny dry run of the whole notebook
RETRAIN_NN = False                         # True: retrain RealMLP (needs pytabkit + a GPU)
SEED, N_FOLDS, TARGET = 42, 5, "is_fraud"

CATS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
FLAGS = ["is_foreign_transaction", "is_new_merchant", "used_vpn",
         "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
NUMS = ["amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km",
        "card_age_months", "customer_age", "account_balance_usd", "cvv_retry_count",
        "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score", "prior_disputes"]
FEATS = NUMS + FLAGS + CATS

tr = pd.read_csv(find("train.csv", "playground-series-s6e11"))
te = pd.read_csv(find("test.csv", "playground-series-s6e11"))
og = pd.read_csv(find("credit_card_fraud_2026.csv")).drop(columns="transaction_id")
if SMOKE:
    tr, te = tr.iloc[:40000].copy(), te.iloc[:10000].copy()
for df in (tr, te, og):
    df[FLAGS] = df[FLAGS].astype("int8")
y = tr[TARGET].astype(int).values
og[TARGET] = og[TARGET].astype(int)
FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))

full = pd.concat([tr[FEATS], te[FEATS]], ignore_index=True)
allv = pd.concat([tr[FEATS], te[FEATS], og[FEATS]], ignore_index=True)
for c in CATS:
    full[c] = pd.Categorical(full[c], categories=sorted(allv[c].unique()))
Xtr, Xte = full.iloc[:len(tr)].reset_index(drop=True), full.iloc[len(tr):].reset_index(drop=True)
OOF, PRED, FULL, ITS = {}, {}, {}, {}      # out-of-fold, 5-fold-average test, refit test (Part 4), best iterations


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


print(f"train {tr.shape}, test {te.shape}, original {og.shape}")
print(f"fraud rate: train {y.mean():.4%}, original {og[TARGET].mean():.4%}")

---
# Part 1 — Per-value target encoding and 4-leaf trees

*Public LB 0.94928 · 5-fold CV 0.94892 · first version of this notebook*

In [ ]:
_y = tr[TARGET]
_rx = pd.qcut(tr["merchant_risk_score"].rank(method="first"), 10, labels=False)
_vy = pd.qcut(tr["velocity_score"].rank(method="first"), 10, labels=False)
draw(
    stats=[("Public LB", "0.94928", "#1 on the board when submitted (Nov 1)"),
           ("5-fold CV AUC", "0.94892", "+0.0019 over a 31-leaf LightGBM"),
           ("Fraud rate", f"{_y.mean():.2%}", f"{int(_y.sum()):,} frauds in {len(_y):,} rows"),
           ("Stack", "5 models", "LightGBM · 2 GLMs · 2 RealMLPs")],
    heat=dict(rate=_y.groupby([_vy, _rx]).mean().unstack().values,
              xname="merchant_risk_score", yname="velocity_score", base=_y.mean()),
    te_gain=[['account_balance_usd', 0.5073, 0.5375], ['hours_since_last_txn', 0.5582, 0.5808], ['customer_age', 0.6278, 0.6487], ['amount_usd', 0.5412, 0.5553], ['distance_from_home_km', 0.5023, 0.5113]],
    depth=dict(leaves=[2, 4, 7, 15, 31, 63], series={'raw features': [0.945423, 0.945879, 0.945708, 0.944967, 0.944632, 0.943917], '+ target encoding': [0.947167, 0.947402, 0.946927, 0.946777, 0.946502, 0.945609]}),
    ladder=[('LightGBM, 31 leaves', 0.947061), ('4 leaves', 0.947772), ('+ per-value target encoding', 0.948845), ('stack with 2 GLMs + 2 RealMLPs', 0.94892)],
    blend=[('LightGBM, 4 leaves + TE', 0.6), ('RealMLP ×2', 0.31), ('GLM ×2', 0.09)],
    title="S6E11 Credit Card Fraud  ·  Part 1",
    subtitle="Per-value target encoding + 4-leaf LightGBM + GLM + RealMLP, stacked")
plt.show()

## 1.1 The original data is a logistic model

Fit two models on the 20k original rows only, scored by their own 5-fold CV. If the original labels were
drawn from a logistic formula, the linear model should win, and it does by a wide margin
(**0.9435** vs 0.9302). The coefficients read like a rulebook: velocity, CVV
retries, merchant risk and the mismatch flags push fraud up; biometric or PIN authentication pushes it down.

In [ ]:
def onehot_frame(df):
    X = pd.get_dummies(df[FEATS], columns=CATS, dtype=float)
    return (X - X.mean()) / X.std()


Xo = onehot_frame(og)
p_lr, p_gb = np.zeros(len(og)), np.zeros(len(og))
for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(Xo, og[TARGET]):
    p_lr[b] = LogisticRegression(C=1.0, max_iter=3000).fit(Xo.iloc[a], og[TARGET].iloc[a]).predict_proba(Xo.iloc[b])[:, 1]
    gb = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=40,
                            subsample=0.85, subsample_freq=1, colsample_bytree=0.85, reg_lambda=5, verbosity=-1)
    p_gb[b] = gb.fit(Xo.iloc[a], og[TARGET].iloc[a]).predict_proba(Xo.iloc[b])[:, 1]
print(f"original data, 5-fold AUC: logistic regression {roc_auc_score(og[TARGET], p_lr):.4f} | "
      f"LightGBM {roc_auc_score(og[TARGET], p_gb):.4f}")

coef = pd.Series(LogisticRegression(C=1.0, max_iter=3000).fit(Xo, og[TARGET]).coef_[0], index=Xo.columns)
top = coef.reindex(coef.abs().sort_values().index[-14:])
_style()
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.barh(range(len(top)), top.values, height=0.55, color=[BLUE if v > 0 else ORANGE for v in top.values], zorder=2)
ax.set_yticks(range(len(top)), top.index)
ax.tick_params(axis="y", length=0, labelcolor=INK)
ax.axvline(0, color=AXIS, lw=1)
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True); ax.spines["left"].set_visible(False)
ax.set_xlabel("logistic-regression coefficient (standardized inputs)")
_title(ax, "The original data reads like a scorecard", "blue raises the fraud odds, orange lowers them · top 14 of 54 terms")
plt.show()

## 1.2 Every value has its own risk

Treat every column as a category — each distinct value of `account_balance_usd` (43k values),
`hours_since_last_txn` (3.6k), `customer_age` (64), ... — and encode it by its fraud rate, **out of fold**:
inside each training fold, a 5-fold inner split gives the training rows their encodings, and the validation
fold and test get encodings fitted on the whole training fold. Smoothing toward the prior with weight 20
keeps rare values near the base rate.

The numbers that look useless on their own (`account_balance_usd` 0.507) carry signal value by value: the
synthetic generator gave individual values their own fraud rate, which no smooth function of the number
can recover. Value counts carry a weaker form of the same signal (+0.0008 for a 31-leaf LightGBM on fold 0).

In [ ]:
def add_te(Xa, ya, others, cols, smooth=20.0, inner=5, seed=0):
    """Out-of-fold target encoding of `cols` (each value = a category), appended as te_<col>."""
    prior = ya.mean()
    Xa, others = Xa.copy(), [o.copy() for o in others]
    key = lambda X, c: X[c].astype(str).values if str(X[c].dtype) == "category" else X[c].values
    rate = lambda k, t: (lambda s: (s["sum"] + prior * smooth) / (s["count"] + smooth))(
        pd.DataFrame({"k": k, "y": t}).groupby("k")["y"].agg(["sum", "count"]))
    for c in cols:
        ka, enc = key(Xa, c), np.zeros(len(Xa), dtype=np.float32)
        for i, j in StratifiedKFold(inner, shuffle=True, random_state=seed).split(ka, ya):
            enc[j] = pd.Series(ka[j]).map(rate(ka[i], ya[i])).fillna(prior).values
        Xa[f"te_{c}"] = enc
        r = rate(ka, ya)
        for o in others:
            o[f"te_{c}"] = pd.Series(key(o, c)).map(r).fillna(prior).values.astype(np.float32)
    return Xa, others


a, b = FOLDS[0]
_, (enc_b,) = add_te(Xtr.iloc[a], y[a], [Xtr.iloc[b]], FEATS)
rows = []
for c in FEATS:
    raw = roc_auc_score(y[b], enc_b[f"te_{c}"] if c in CATS else enc_b[c])
    rows.append((c, max(raw, 1 - raw), roc_auc_score(y[b], enc_b[f"te_{c}"])))
gain = pd.DataFrame(rows, columns=["column", "raw AUC", "encoded AUC"]).assign(gain=lambda d: d["encoded AUC"] - d["raw AUC"])
gain = gain.sort_values("gain", ascending=False)
display(gain.head(8).round(4).reset_index(drop=True))
_style()
fig = plt.figure(figsize=(9, 5))
_dumbbell(fig, [0.25, 0.1, 0.68, 0.72], [tuple(r) for r in gain.head(8)[["column", "raw AUC", "encoded AUC"]].values])
plt.show()

## 1.3 LightGBM with 4 leaves

Once the encodings are in, depth hurts — and even without them. Fold-0 AUC by tree size, every other
setting equal (learning rate 0.05, `min_child_samples=100`):

| num_leaves | raw features | + target encoding |
|---|---|---|
| 2 | 0.94542 | 0.94717 |
| 4 | 0.94588 | 0.94740 |
| 7 | 0.94571 | 0.94693 |
| 15 | 0.94497 | 0.94678 |
| 31 | 0.94463 | 0.94650 |
| 63 | 0.94392 | 0.94561 |

The final model uses `num_leaves=4`, learning rate 0.03 and early stopping on the validation fold.

In [ ]:
LGB = dict(n_estimators=20000, learning_rate=0.03, num_leaves=4, min_child_samples=100, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, max_bin=255, n_jobs=-1, verbose=-1)


def run_lgb(name, XA, XT, use_te, smooth=20.0):
    oof, pred, t0 = np.zeros(len(tr)), np.zeros(len(te)), time.time()
    for f, (a, b) in enumerate(FOLDS):
        Xa, Xb, Xt = XA.iloc[a], XA.iloc[b], XT
        if use_te:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt], FEATS, smooth)
        m = lgb.LGBMClassifier(**LGB, random_state=SEED)
        m.fit(Xa, y[a], eval_set=[(Xb, y[b])], eval_metric="auc", callbacks=[lgb.early_stopping(300, verbose=False)])
        oof[b] = m.predict_proba(Xb)[:, 1]
        pred += m.predict_proba(Xt)[:, 1] / N_FOLDS
        print(f"{name} fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f}, {m.best_iteration_} trees ({time.time() - t0:.0f}s)")
    OOF[name], PRED[name] = oof, pred
    print(f"==> {name}: CV AUC {roc_auc_score(y, oof):.5f}\n")


run_lgb("lgb_te_l4", Xtr, Xte, use_te=True)

## 1.4 Two GLMs

A logistic regression on a sparse design: standardized numbers (log for money and hours), cubic B-splines
with quantile knots, the flags, one-hot categoricals, and every numeric column with at most 1000 distinct
values one-hot encoded (values seen fewer than 50 times are pooled). `glm_te` adds the standardized logits
of the target encodings and is regularized harder (`C=0.3`); `glm_sb` has no encodings.

In [ ]:
def design(max_card, min_freq=50, knots=8):
    num = full[NUMS].astype(float).copy()
    for c in ("amount_usd", "account_balance_usd", "hours_since_last_txn"):
        num[c] = np.log1p(num[c])
    z = StandardScaler().fit_transform(num)
    small = [c for c in NUMS if full[c].nunique() <= max_card]
    return sparse.hstack([
        sparse.csr_matrix(z), sparse.csr_matrix(full[FLAGS].values.astype(float)),
        OneHotEncoder(handle_unknown="ignore").fit_transform(full[CATS].astype(str)),
        sparse.csr_matrix(SplineTransformer(n_knots=knots, degree=3, knots="quantile").fit_transform(z)),
        OneHotEncoder(handle_unknown="ignore", min_frequency=min_freq).fit_transform(full[small]),
    ]).tocsr()


def run_glm(name, max_card, C, with_te, src=None, te_cols=FEATS, extra=None, refit=False, base=None, smooth=20.0):
    X = design(max_card) if base is None else base                       # Part 5: a plain design without splines
    X = X if extra is None else sparse.hstack([X, extra]).tocsr()        # Part 4: + one-hot keys
    XA, XT = X[:len(tr)], X[len(tr):]
    SA, ST = src if src is not None else (Xtr, Xte)      # frames whose te_cols get encoded (Part 3 adds digit keys)

    def fit(a, others):
        Ma, Ms = XA[a], [M for M, _ in others]
        if with_te:
            ta, enc = add_te(SA.iloc[a], y[a], [S for _, S in others], te_cols, smooth)
            cols = [f"te_{c}" for c in te_cols]
            sc = StandardScaler().fit(logit(ta[cols].values))
            Ma, *Ms = (sparse.hstack([M, sparse.csr_matrix(sc.transform(logit(T[cols].values)))]).tocsr()
                       for M, T in zip([Ma] + Ms, [ta] + enc))
        m = LogisticRegression(C=C, max_iter=3000, tol=1e-6).fit(Ma, y[a])
        return [m.predict_proba(M)[:, 1] for M in Ms]

    oof, pred, t0 = np.zeros(len(tr)), np.zeros(len(te)), time.time()
    for f, (a, b) in enumerate(FOLDS):
        oof[b], p = fit(a, [(XA[b], SA.iloc[b]), (XT, ST)])
        pred += p / N_FOLDS
        print(f"{name} fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f} ({time.time() - t0:.0f}s)")
    OOF[name], PRED[name] = oof, pred
    print(f"==> {name}: CV AUC {roc_auc_score(y, oof):.5f}\n")
    if refit:                                            # Part 4: one more fit on every training row
        FULL[name], = fit(np.arange(len(tr)), [(XT, ST)])


run_glm("glm_te", max_card=1000, C=0.3, with_te=True)
run_glm("glm_sb", max_card=100, C=1.0, with_te=False)

## 1.5 RealMLP

RealMLP from pytabkit with the hyper-parameters of Vladimir Demidov's S6E10 notebook (4 ensemble members,
4 epochs, no early stopping). `realmlp_te` gets the target encodings as extra numeric inputs (+0.0019 CV over
plain `realmlp_v1`). Both were trained on our GPU with the same folds; their out-of-fold and test
predictions are loaded from our dataset. The training code is below (`RETRAIN_NN = True`).

In [ ]:
REALMLP = dict(
    n_ens=4, n_epochs=4, batch_size=512, use_early_stopping=False,
    lr=0.04, wd=0.0236, sq_mom=0.988, lr_sched="lin_cos_log_15", wd_sched="cos_log_15",
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act="silu", p_drop=0.05, p_drop_sched="expm4t",
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name="gelu", plr_lr_factor=0.1151, plr_sigma=2.33,
    ls_eps=0.01, ls_eps_sched="sqrt_cos", add_front_scale=False, bias_init_mode="neg-uniform-dynamic-2",
    tfms=["one_hot", "median_center", "robust_scale", "smooth_clip", "embedding", "l2_normalize"])


def train_realmlp(with_te):
    from pytabkit import RealMLP_TD_Classifier
    Xc = full.copy()
    for c in CATS:
        Xc[c] = Xc[c].astype(str).astype("category")
    XA, XT = Xc.iloc[:len(tr)].reset_index(drop=True), Xc.iloc[len(tr):].reset_index(drop=True)
    oof, pred = np.zeros(len(tr)), np.zeros(len(te))
    for f, (a, b) in enumerate(FOLDS):
        Xa, Xb, Xt = XA.iloc[a], XA.iloc[b], XT
        if with_te:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt], FEATS)
        m = RealMLP_TD_Classifier(device="cuda", random_state=SEED, val_metric_name="1-auc_ovr", verbosity=0, **REALMLP)
        m.fit(Xa, y[a], X_val=Xb, y_val=y[b], cat_col_names=CATS)
        oof[b], pred = m.predict_proba(Xb)[:, 1], pred + m.predict_proba(Xt)[:, 1] / N_FOLDS
    return oof, pred


for name, with_te in (("realmlp_te", True), ("realmlp_v1", False)):
    if RETRAIN_NN:
        OOF[name], PRED[name] = train_realmlp(with_te)
    else:
        OOF[name] = np.load(find(f"{name}_oof.npy", "s6e11-fraud-series-artifacts"))[:len(tr)].astype(float)
        PRED[name] = np.load(find(f"{name}_test.npy", "s6e11-fraud-series-artifacts"))[:len(te)].astype(float)
    print(f"{name}: CV AUC {roc_auc_score(y, OOF[name]):.5f}")

## 1.6 Part 1 stack

A logistic regression on the five logits, cross-validated on a second 5-fold split so the stacker never
scores rows it was fitted on. Part 1's submission is written to `submission_part1.csv`.

In [ ]:
P1_NAMES = ["lgb_te_l4", "glm_te", "realmlp_te", "glm_sb", "realmlp_v1"]
Lo = np.column_stack([logit(OOF[n]) for n in P1_NAMES])
Lp = np.column_stack([logit(PRED[n]) for n in P1_NAMES])
st = np.zeros(len(y))
for a, b in StratifiedKFold(5, shuffle=True, random_state=7).split(Lo, y):
    st[b] = LogisticRegression(C=1.0, max_iter=1000).fit(Lo[a], y[a]).decision_function(Lo[b])
stack = LogisticRegression(C=1.0, max_iter=1000).fit(Lo, y)
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P1_NAMES] + [roc_auc_score(y, st)],
                      "stack coefficient": list(stack.coef_[0]) + [np.nan]},
                     index=P1_NAMES + ["stack (cross-validated)"]).round({"CV AUC": 5, "stack coefficient": 3}))
pd.DataFrame({"id": te["id"], TARGET: 1 / (1 + np.exp(-stack.decision_function(Lp)))}).to_csv("submission_part1.csv", index=False)

## 1.7 What did not help in Part 1 (fold 0 unless noted)

| idea | result | note |
|---|---|---|
| original 20k rows appended to training | -0.0003 | the original rows are distributed differently |
| ratios (amount/balance, ...) | -0.0004 |  |
| number of flags set | -0.0001 | trees find it anyway |
| numeric columns as LightGBM categoricals | -0.0054 | per-value splits overfit; smoothed out-of-fold encodings are the way |
| pair target encodings (91 pairs) | -0.0006 | 4-leaf trees already model the pairs |
| value counts on top of the encodings | +0.00004 | noise |
| TabM | 0.9425 | vs RealMLP 0.9443 |
| GLM with pairwise products and category pairs | -0.0016 | overfits |
| CatBoost with every number as a categorical (ordered target statistics) | 0.9448 | vs 0.9474 for 4-leaf LightGBM + TE, and 30x slower |

---
# Part 2 — Two views of the data

*Public LB 0.94945 · 5-fold CV 0.94923 · second version of this notebook*

In [ ]:
draw_part2(
    stats=[('Public LB', '0.94945', '#1 · +0.00017 over Part 1'), ('5-fold CV AUC', '0.94923', '+0.00031 over Part 1'), ('Candidates', '12 new', '6 members in the final blend'), ('Best single model', '0.94885', '4-leaf LightGBM + TE, smoothing 100')],
    names=['lgb_te_s100', 'lgb_te_s300', 'xgb_te_d2', 'realmlp_te', 'lgb_raw', 'lgb_raw_cnt'],
    corr=[[1, 0.9959, 0.993, 0.9747, 0.9767, 0.9763], [0.9959, 1, 0.99, 0.9723, 0.9756, 0.9753], [0.993, 0.99, 1, 0.9772, 0.976, 0.9746], [0.9747, 0.9723, 0.9772, 1, 0.9659, 0.9639], [0.9767, 0.9756, 0.976, 0.9659, 1, 0.9912], [0.9763, 0.9753, 0.9746, 0.9639, 0.9912, 1]],
    gains=[('lgb_raw_l4', 'raw view', 1.43), ('lgb_raw_l4_cnt', 'raw view', 1.47), ('xgb_raw_d3', 'raw view', 1.12), ('lgb_raw_l31', 'raw view', 0.72), ('lgb_te_l4_s100', 'target-encoded view', 0.91), ('lgb_te_l4_s300', 'target-encoded view', 1.0), ('lgb_te_l4_s5', 'target-encoded view', 0.35), ('xgb_te_d2', 'target-encoded view', 0.64), ('realmlp_te_s2', 'neural net', 0.21), ('realmlp_te_cnt', 'neural net', 0.23), ('realmlp_cnt', 'neural net', 0.54), ('realmlp_allcat', 'neural net', -0.2)],
    smoothing={'weights': [5, 20, 100, 300], 'cv': [0.948777, 0.948845, 0.948852, 0.94873], 'kept': [100, 300]},
    parts=[('Part 1', 0.94892, 0.94928), ('Part 2', 0.949229, 0.94945)],
    blend=[('LightGBM 4 leaves + TE (smoothing 100)', 'target-encoded view', 0.25), ('XGBoost depth 2 + TE', 'target-encoded view', 0.25), ('LightGBM 4 leaves + TE (smoothing 300)', 'target-encoded view', 0.125), ('LightGBM 4 leaves, raw columns', 'raw view', 0.125), ('LightGBM 4 leaves, raw + value counts', 'raw view', 0.125), ('RealMLP + TE', 'neural net', 0.125)],
    title="S6E11 Credit Card Fraud  ·  Part 2",
    subtitle="Two views of the same data: target-encoded and raw shallow trees, blended by rank hill climbing")
plt.show()

Part 1's members all look at the data through the target encodings. Part 2 asks what to put *next to* them.
The **raw view** is the 24 columns as they are, optionally with value counts (how often each value occurs
in train + test + original). Both views share the folds.

## 2.1 Five more tree members

All shallow: LightGBM with 4 leaves (target-encoded with smoothing 100 and 300, raw, raw + counts) and
XGBoost with depth 2 on the target-encoded view.

In [ ]:
CNT = pd.DataFrame({f"cnt_{c}": full[c].map(allv[c].value_counts()).astype("float32") for c in NUMS})
Ctr, Cte = CNT.iloc[:len(tr)].reset_index(drop=True), CNT.iloc[len(tr):].reset_index(drop=True)
XGB = dict(n_estimators=20000, learning_rate=0.03, max_depth=2, subsample=0.8, colsample_bytree=0.5,
           min_child_weight=5, reg_lambda=1.0, max_bin=256, tree_method="hist", n_jobs=-1, eval_metric="auc",
           early_stopping_rounds=300, enable_categorical=True, max_cat_to_onehot=1)

run_lgb("lgb_te_l4_s100", Xtr, Xte, use_te=True, smooth=100.0)
run_lgb("lgb_te_l4_s300", Xtr, Xte, use_te=True, smooth=300.0)
run_lgb("lgb_raw_l4", Xtr, Xte, use_te=False)
run_lgb("lgb_raw_l4_cnt", pd.concat([Xtr, Ctr], axis=1), pd.concat([Xte, Cte], axis=1), use_te=False)

oof, pred, t0 = np.zeros(len(tr)), np.zeros(len(te)), time.time()
for f, (a, b) in enumerate(FOLDS):
    Xa, (Xb, Xt) = add_te(Xtr.iloc[a], y[a], [Xtr.iloc[b], Xte], FEATS)
    m = xgb.XGBClassifier(**XGB, random_state=SEED)
    m.fit(Xa, y[a], eval_set=[(Xb, y[b])], verbose=False)
    oof[b] = m.predict_proba(Xb)[:, 1]
    pred += m.predict_proba(Xt)[:, 1] / N_FOLDS
    print(f"xgb_te_d2 fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f}, {m.best_iteration} trees ({time.time() - t0:.0f}s)")
OOF["xgb_te_d2"], PRED["xgb_te_d2"] = oof, pred
print(f"==> xgb_te_d2: CV AUC {roc_auc_score(y, oof):.5f}")

## 2.2 Two views, two blocks

The target-encoded trees agree with each other at 0.99+, so a second or third one adds little. The raw-view
trees sit at about 0.976 from them, which is far enough to matter, and RealMLP is further still.

In [ ]:
P2_NAMES = ["lgb_te_l4_s100", "lgb_te_l4_s300", "xgb_te_d2", "realmlp_te", "lgb_raw_l4", "lgb_raw_l4_cnt"]
display(pd.DataFrame({n: OOF[n] for n in P2_NAMES}).corr(method="spearman").round(4))

What each candidate adds, one at a time on top of Part 1's five-model stack (CV 0.94892; cross-validated
stack AUC, × 1e-4), all on the same folds:

| candidate | view | own CV | gain over Part 1 |
|---|---|---|---|
| `lgb_raw_l4_cnt` | raw | 0.94778 | +1.47 |
| `lgb_raw_l4` | raw | 0.94777 | +1.43 |
| `xgb_raw_d3` | raw | 0.94749 | +1.12 |
| `lgb_te_l4_s300` | target-encoded | 0.94873 | +1.00 |
| `lgb_te_l4_s100` | target-encoded | 0.94885 | +0.91 |
| `lgb_raw_l31` | raw | 0.94706 | +0.72 |
| `xgb_te_d2` | target-encoded | 0.94881 | +0.64 |
| `realmlp_cnt` | neural net, raw + counts | 0.94664 | +0.54 |
| `lgb_te_l4_s5` | target-encoded | 0.94878 | +0.35 |
| `realmlp_te_cnt` | neural net | 0.94760 | +0.23 |
| `realmlp_te_s2` | neural net | 0.94762 | +0.21 |
| `realmlp_allcat` | neural net, per-value embeddings | 0.94640 | -0.20 |

The raw-view trees lead, although each of them is weaker than any target-encoded tree on its own.
More neural variants (other seeds, counts, per-value embeddings) add almost nothing.

## 2.3 Rank hill climbing

AUC only depends on the order of the predictions, so the members become ranks (scaled to [0, 1]) and are
added greedily: at each step, add the member (repeats allowed) that raises the out-of-fold AUC most; stop
when nothing helps. The weights come out in steps of 1/8 with six members, so there is little room to
overfit. On all 13 candidates the same procedure picks exactly these six. Part 2's submission is written
to `submission_part2.csv`.

In [ ]:
def hill_climb(R, names, max_iter=60):
    counts, cur, best = np.zeros(R.shape[1]), np.zeros(len(y)), 0.0
    for _ in range(max_iter):
        scores = [roc_auc_score(y, cur + R[:, j]) for j in range(R.shape[1])]
        j = int(np.argmax(scores))
        if scores[j] <= best + 1e-7:
            break
        best, cur = scores[j], cur + R[:, j]
        counts[j] += 1
        print(f"  add {names[j]:16s} -> OOF AUC {best:.6f}")
    return counts / counts.sum(), best


Ro = np.column_stack([rankdata(OOF[n]) / len(y) for n in P2_NAMES])
Rp = np.column_stack([rankdata(PRED[n]) / len(te) for n in P2_NAMES])
w, auc = hill_climb(Ro, P2_NAMES)
print(f"\nPart 2 blend OOF AUC {auc:.6f}")
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P2_NAMES], "weight": w}, index=P2_NAMES).round(5))

sub = pd.DataFrame({"id": te["id"], TARGET: Rp @ w})
sub.to_csv("submission_part2.csv", index=False)
sub.head()

## 2.4 What did not help in Part 2

| idea | result | note |
|---|---|---|
| TE smoothing 5 as an extra member | +0.35 alone, -0.08 in the final set | too close to smoothing 20 |
| XGBoost + TE, depth 3 / 4 | fold 0: 0.94693 / 0.94686 | depth 2: 0.94726 |
| CatBoost with every number as a categorical (GPU) | fold 0: 0.9448 | 32 min per fold |
| CatBoost on the raw columns, depth 6 (GPU) | fold 0: 0.9426 | 17 min per fold, stopped |
| RealMLP with per-value embeddings | CV 0.94640, -0.20e-4 in the stack |  |
| pair target encodings (91 pairs) | fold 0: -0.0006 | 4-leaf trees already model pairs |
| logistic stack over all 13 candidates | CV 0.94920 | hill climbing over the same 13: 0.94923 |

---
# Part 3 — The digits carry the label

*Public LB 0.95113 · 5-fold CV 0.95073*

Parts 1 and 2 treated every number as a value: its size, or the fraud rate of rows with exactly that value.
Part 3 looks at how the numbers are *written*. The synthetic rows were made by a generator trained on the
original data, and if it writes a number digit by digit, the digits themselves can carry the label.

In [ ]:
draw_part3(
    stats=[('Public LB', '0.95113', '+0.00168 over Part 2'), ('5-fold CV AUC', '0.95073', '+0.00150 over Part 2'), ('Best single model', '0.95040', '4-leaf LightGBM + digit keys'), ('New keys', '4 per number', 'cents · int(x) · int(x/10) · int(10x)')],
    digits={'cols': ['amount_usd', 'hours_since_last_txn', 'distance_from_home_km', 'account_balance_usd'], 'rate': [[0.01904, 0.01499, 0.01563, 0.01542, 0.01925, 0.01772, 0.01478, 0.01729, 0.01712, 0.01633], [0.02204, 0.01584, 0.01603, 0.0139, 0.01919, 0.0185, 0.01265, 0.01491, 0.01626, 0.01679], [0.01942, 0.01621, 0.01493, 0.01505, 0.01813, 0.01727, 0.01482, 0.0173, 0.01675, 0.01677], [0.01752, 0.01647, 0.01557, 0.01632, 0.01678, 0.01784, 0.01584, 0.01639, 0.01743, 0.01686]], 'base': 0.0167},
    steps=[('per-value TE', 0.947461), ('+ cents', 0.947794), ('+ int(x), int(x/10)', 0.949663), ('+ int(10x)', 0.949783)],
    families=[('LightGBM, 4 leaves', 0.9488452589277478, 0.9503587999739997), ('XGBoost, depth 2', 0.9488134428126807, 0.950254313009181), ('EBM', 0.948771, 0.9498084549664615), ('GLM', 0.9470568398079565, 0.9487861923393974), ('RealMLP', 0.9477104886524447, 0.9485288055034796)],
    parts=[('Part 1', 0.94892, 0.94928), ('Part 2', 0.94923, 0.94945), ('Part 3', 0.95073, 0.95113)],
    blend=[('lgb_diglead_l4', 'target-encoded view', 0.3333), ('lgb_dig_s100', 'target-encoded view', 0.3333), ('ebm_dig', 'target-encoded view', 0.1667), ('ebm_b4', 'raw view', 0.1667)],
    title="S6E11 Credit Card Fraud  ·  Part 3",
    subtitle="The synthetic generator writes numbers digit by digit, and the digits carry the label")
plt.show()

## 3.1 A fingerprint of the generator

Fraud rate by the first digit after the decimal point, for the four float columns with two decimals.
A physical quantity should not care about its first decimal, yet the same digits stand out in every column:
`.0` and `.4` above the base rate, `.2`, `.3` and `.6` below it.

In [ ]:
D4 = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd"]


def by_digit(df, col, digit):
    k = np.round(df[col].values * 100).astype(np.int64)        # the number in cents
    return pd.Series(df[TARGET].values).groupby(digit(k)).mean().reindex(range(10))


first_decimal = pd.DataFrame({c: by_digit(tr, c, lambda k: (k // 10) % 10) for c in D4}).T
display((first_decimal * 100).round(2).rename(columns=lambda d: f".{d}"))
_style()
fig = plt.figure(figsize=(11, 3.6))
_digits(fig, [0.2, 0.14, 0.62, 0.62], dict(cols=D4, rate=first_decimal.values.tolist(), base=y.mean()))
plt.show()

The integer part shows the same kind of pattern: a tens digit of 4 is above the base rate in `amount_usd`,
`hours_since_last_txn` and `account_balance_usd`, a 3 below it. The digit frequencies tell the same story
from the other side: train and test agree to within 0.2 percentage points, the original 20k rows differ by
up to 2.4. The generator left its handwriting on both train and test, so whatever the digits say about the
label in train should hold on the test rows.

In [ ]:
tens = pd.DataFrame({c: by_digit(tr, c, lambda k: (k // 1000) % 10) for c in D4}).T
display((tens * 100).round(2).rename(columns=lambda d: f"{d}_"))

FLOATS = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd",
          "velocity_score", "merchant_risk_score"]


def digit_share(df, col):
    k = np.round(df[col].values * 100).astype(np.int64)
    return pd.Series((k // 10) % 10).value_counts(normalize=True).sort_index()


gap = pd.DataFrame({c: [(digit_share(tr, c) - digit_share(te, c)).abs().max(),
                        (digit_share(tr, c) - digit_share(og, c)).abs().max()] for c in FLOATS},
                   index=["train vs test", "train vs original"]).T * 100
display(gap.round(2).rename(columns=lambda s: s + " (max gap, pp)"))

## 3.2 Four keys per number

For each of the six float columns we add four keys and target-encode each one exactly like the values in
Part 1 (out of fold, smoothing 20):

| key | example for 1371.37 | what it groups |
|---|---|---|
| `cents` | 37 | the two decimals (also kept as a plain number) |
| `int(x)` | 1371 | the integer part |
| `int(x/10)` | 137 | the integer part without its last digit |
| `int(10x)` | 13713 | the value cut to one decimal |

The keys themselves are dropped after encoding; only their encodings (and `cents`) reach the model. Each
step on one 4-leaf LightGBM, fold 0 (the Part 2 features score 0.94746 there):

| features | fold-0 AUC | gain |
|---|---|---|
| Part 2 features (per-value TE of every column) | 0.94746 | — |
| + `cents` | 0.94779 | +3.3e-4 |
| + `int(x)`, `int(x/10)` (no `cents`) | 0.94893 | +14.6e-4 |
| + `cents`, `int(x)`, `int(x/10)` | 0.94966 | +22.0e-4 |
| + `cents`, `int(x)`, `int(x/10)`, `int(10x)` | 0.94978 | **+23.2e-4** |
| … + leading one and two digits of `int(x)` | 0.94988 | +24.2e-4 |

Finer histogram bins (`max_bin=4095`) on the old features give nothing (0.94751): the signal is not a
resolution problem but the digits themselves.

In [ ]:
def digit_frame(df):
    out = {}
    for c in FLOATS:
        out[f"cents_{c}"] = (np.round(df[c] * 100) % 100).astype("int16").values
    for c in FLOATS:
        out[f"key_int_{c}"] = np.floor(df[c]).astype("int64").values
        out[f"key_ten_{c}"] = np.floor(df[c] / 10).astype("int64").values
    for c in FLOATS:
        out[f"key_dec_{c}"] = np.floor(df[c] * 10 + 1e-6).astype("int64").values
    return pd.DataFrame(out)


DG = digit_frame(full)
CENTS = [c for c in DG if c.startswith("cents_")]
KEYS = [c for c in DG if c.startswith("key_")]
XDtr = pd.concat([Xtr, DG.iloc[:len(tr)].reset_index(drop=True)], axis=1)
XDte = pd.concat([Xte, DG.iloc[len(tr):].reset_index(drop=True)], axis=1)
DIG_TE = FEATS + CENTS + KEYS          # every column, the cents and the three rounded keys


def drop_keys(D):
    return D[[c for c in D.columns if not c.startswith("key_")]]


def run_tree(name, kind, XA, XT, te_cols, smooth=20.0, seed=SEED, teseed=0, **over):
    oof, pred, its, t0 = np.zeros(len(tr)), np.zeros(len(te)), [], time.time()
    for f, (a, b) in enumerate(FOLDS):
        Xa, Xb, Xt = XA.iloc[a], XA.iloc[b], XT
        if te_cols:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt], te_cols, smooth, seed=teseed)
        Xa, Xb, Xt = drop_keys(Xa), drop_keys(Xb), drop_keys(Xt)
        if kind == "lgb":
            m = lgb.LGBMClassifier(**{**LGB, **over}, random_state=seed)
            m.fit(Xa, y[a], eval_set=[(Xb, y[b])], eval_metric="auc", callbacks=[lgb.early_stopping(300, verbose=False)])
            it = m.best_iteration_
        else:
            m = xgb.XGBClassifier(**{**XGB, **over}, random_state=seed)
            m.fit(Xa, y[a], eval_set=[(Xb, y[b])], verbose=False)
            it = m.best_iteration
        oof[b] = m.predict_proba(Xb)[:, 1]
        pred += m.predict_proba(Xt)[:, 1] / N_FOLDS
        its.append(it)
        print(f"{name} fold {f}: AUC {roc_auc_score(y[b], oof[b]):.5f}, {it} trees ({time.time() - t0:.0f}s)")
    OOF[name], PRED[name], ITS[name] = oof, pred, its
    print(f"==> {name}: CV AUC {roc_auc_score(y, oof):.5f}\n")


run_tree("lgb_dig_l4", "lgb", XDtr, XDte, DIG_TE)

## 3.3 Every family, retrained with the digits

The same keys for the other members: the 4-leaf LightGBM with the leading one and two digits of the
integer part as two more keys (the best single model, CV 0.95040), the same model with TE smoothing 100
as a second view, and a depth-2 XGBoost. Two more views were measured and left out because the blend did
not pick them (section 3.5).

In [ ]:
# the leading one and two digits of the integer part, with its length (1371.37 -> 41 and 413)
def lead_frame(df):
    out = {}
    for c in FLOATS:
        i = np.floor(df[c]).astype("int64").astype(str)
        out[f"key_l1_{c}"] = (i.str.len() * 10 + i.str[0].astype(int)).values
        out[f"key_l2_{c}"] = (i.str.len() * 100 + i.str[:2].astype(int)).values
    return pd.DataFrame(out)


LD = lead_frame(full)
XLtr = pd.concat([XDtr, LD.iloc[:len(tr)].reset_index(drop=True)], axis=1)
XLte = pd.concat([XDte, LD.iloc[len(tr):].reset_index(drop=True)], axis=1)
DEC = [c for c in KEYS if c.startswith("key_dec_")]
LEAD_TE = FEATS + CENTS + [c for c in KEYS if c not in DEC] + list(LD.columns) + DEC

run_tree("lgb_diglead_l4", "lgb", XLtr, XLte, LEAD_TE)              # + the leading digits
run_tree("lgb_dig_s100", "lgb", XDtr, XDte, DIG_TE, smooth=100.0)   # smoothing as a second view
run_tree("xgb_dig_d2", "xgb", XDtr, XDte, DIG_TE)                     # depth-2 XGBoost

A GLM with the encodings of all the keys, and the EBM / RealMLP members. The RealMLP members were trained
on our GPU and the EBMs on 16 CPU cores (about 2 hours each), so their out-of-fold and test predictions are
loaded from our dataset; the training code is here (`RETRAIN_NN`, `RETRAIN_EBM`).

In [ ]:
run_glm("glm_dig", max_card=1000, C=0.3, with_te=True, src=(XDtr, XDte), te_cols=DIG_TE)

RETRAIN_EBM = False


def train_ebm(dig, outer_bags=4):
    from interpret.glassbox import ExplainableBoostingClassifier
    XA, XT = (XDtr, XDte) if dig else (Xtr, Xte)
    oof, pred = np.zeros(len(tr)), np.zeros(len(te))
    for f, (a, b) in enumerate(FOLDS):
        Xa, Xb, Xt = XA.iloc[a], XA.iloc[b], XT
        if dig:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt], DIG_TE)
            Xa, Xb, Xt = drop_keys(Xa), drop_keys(Xb), drop_keys(Xt)
        m = ExplainableBoostingClassifier(interactions=10, max_bins=256, outer_bags=outer_bags, n_jobs=-1,
                                          random_state=42).fit(Xa, y[a])
        oof[b], pred = m.predict_proba(Xb)[:, 1], pred + m.predict_proba(Xt)[:, 1] / N_FOLDS
    return oof, pred


def train_realmlp_dig(seed=SEED, smooth=20.0):
    from pytabkit import RealMLP_TD_Classifier
    Xc = XDtr.copy(), XDte.copy()
    for X in Xc:
        for c in CATS:
            X[c] = X[c].astype(str).astype("category")
    oof, pred = np.zeros(len(tr)), np.zeros(len(te))
    for f, (a, b) in enumerate(FOLDS):
        Xa, (Xb, Xt) = add_te(Xc[0].iloc[a], y[a], [Xc[0].iloc[b], Xc[1]], DIG_TE, smooth)
        Xa, Xb, Xt = drop_keys(Xa), drop_keys(Xb), drop_keys(Xt)
        m = RealMLP_TD_Classifier(device="cuda", random_state=seed, val_metric_name="1-auc_ovr", verbosity=0, **REALMLP)
        m.fit(Xa, y[a], X_val=Xb, y_val=y[b], cat_col_names=CATS)
        oof[b], pred = m.predict_proba(Xb)[:, 1], pred + m.predict_proba(Xt)[:, 1] / N_FOLDS
    return oof, pred


for name in ['realmlp_dig', 'ebm_dig', 'ebm_b4']:
    if RETRAIN_EBM and name.startswith("ebm"):
        OOF[name], PRED[name] = train_ebm(dig="dig" in name)
    elif RETRAIN_NN and name.startswith("realmlp"):
        OOF[name], PRED[name] = train_realmlp_dig()
    else:
        OOF[name] = np.load(find(f"{name}_oof.npy", "s6e11-fraud-series-artifacts"))[:len(tr)].astype(float)
        PRED[name] = np.load(find(f"{name}_test.npy", "s6e11-fraud-series-artifacts"))[:len(te)].astype(float)
    print(f"{name}: CV AUC {roc_auc_score(y, OOF[name]):.5f}")

| family | Part 2 features | + digit keys | gain |
|---|---|---|---|
| LightGBM, 4 leaves | 0.94885 | 0.95036 | +15e-4 |
| XGBoost, depth 2 | 0.94881 | 0.95025 | +14e-4 |
| EBM | 0.94877 | 0.94981 | +10e-4 |
| GLM | 0.94706 | 0.94879 | +17e-4 |
| RealMLP | 0.94771 | 0.94853 | +8e-4 |

## 3.4 The Part 3 blend

The same rank hill climbing as in Part 2, over the digit-aware members and everything from Parts 1 and 2.
Out of 18 candidates it keeps 4, in steps of 1/6: `lgb_diglead_l4` 2/6, `lgb_dig_s100` 2/6, `ebm_dig` 1/6, `ebm_b4` 1/6 (OOF AUC 0.95073). None of the Part 1 and Part 2 members is picked except the raw-view EBM, which looks at the data most differently. Part 3's submission is written to `submission_part3.csv`.

In [ ]:
P3_NAMES = ['lgb_dig_l4', 'lgb_diglead_l4', 'lgb_dig_s100', 'xgb_dig_d2', 'glm_dig', 'realmlp_dig', 'ebm_dig', 'ebm_b4', 'lgb_te_l4', 'glm_te', 'realmlp_te', 'glm_sb', 'realmlp_v1', 'lgb_te_l4_s100', 'lgb_te_l4_s300', 'xgb_te_d2', 'lgb_raw_l4', 'lgb_raw_l4_cnt']
Ro = np.column_stack([rankdata(OOF[n]) / len(y) for n in P3_NAMES])
Rp = np.column_stack([rankdata(PRED[n]) / len(te) for n in P3_NAMES])
w, auc = hill_climb(Ro, P3_NAMES)
print(f"\nPart 3 blend OOF AUC {auc:.6f}")
keep = w > 0
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P3_NAMES], "weight": w},
                     index=P3_NAMES)[keep].round(5))
display(pd.DataFrame({n: OOF[n] for n in np.array(P3_NAMES)[keep]}).corr(method="spearman").round(4))

sub = pd.DataFrame({"id": te["id"], TARGET: Rp @ w})
sub.to_csv("submission_part3.csv", index=False)
sub.head()

## 3.5 What did not help in Part 3

| idea | result | note |
|---|---|---|
| Per-value encoding of the blend's residuals | no gain | only negative weights help (a leak inside the OOF): the values are used up |
| First and second decimal as two separate keys | −0.00003 (fold 0) | `cents` already holds both |
| Ones and tens digit of the integer part | −0.00015 | `int(x)` and `int(x/10)` cover them |
| `int(x/100)` | ±0.00000 |  |
| Pairs of first decimals across columns (6 keys) | −0.00005 | the digit effects add up column by column |
| Finer LightGBM histograms (`max_bin=4095`), no keys | +0.00005 | not a resolution problem |
| TE smoothing 10 / 100 for the digit model | −0.00036 / −0.00038 | 20 stays; 100 still joins the blend as a second view |
| 7 leaves instead of 4 | CV 0.95016 | shallow still wins |
| Only the rounded keys encoded, no per-value TE (`lgb_dk_l4`) | CV 0.94901 | a different view, but not picked by the blend |
| Raw columns + decimals as numbers, no encodings (`lgb_rawdig_l4`) | CV 0.94838 | +0.0006 over the raw view of Part 2, not picked: the raw EBM is the better raw view |
| RealMLP on raw columns + decimals (no encodings) | CV 0.94615 | the network needs the encodings |
| Label correlation along `id` | |r| < 0.002 | no batch structure in the generated rows |

---
# Part 4 — Same models, more data

*Public LB 0.95154 · 5-fold CV 0.95079*

Part 4 finds no new signal. It estimates Part 3's signal with less noise, in three ways: one more model per
member trained on every training row, the mean of several seeds, and a GLM that fits one coefficient per
value for all values at once.

In [ ]:
draw_part4(
    stats=[('Public LB', '0.95154', '+0.00041 over Part 3'), ('5-fold CV AUC', '0.95079', '+0.00006 over Part 3'), ('Refit on all rows', '+0.00019', 'public LB, same models and weights'), ('Joint GLM', '0.94991', "+0.00112 over Part 3's GLM")],
    steps=[('Part 3 blend, 5-fold average', 0.95113), ('same blend, refit on all rows', 0.95132), ('+ seed bags, joint GLM', 0.95154)],
    csweep={'C': [0.003, 0.005, 0.01, 0.03, 0.1, 0.3], 'series': {'one-hot only': [0.9432, None, 0.946641, 0.947582, 0.946357, 0.942588], 'one-hot + TE logits': [0.949056, 0.949178, 0.94915, 0.948535, 0.946593, None]}, 'ref': ['TE logits only\n(Part 3 GLM)', 0.947748]},
    families=[('LightGBM + leading digits', 0.9503999740916812, 0.9505103584240805), ('LightGBM, smoothing 100', 0.9502855033473656, 0.9504258336734523), ('GLM', 0.9487861923393974, 0.9499073661219074)],
    parts=[('Part 1', 0.94892, 0.94928), ('Part 2', 0.94923, 0.94945), ('Part 3', 0.95073, 0.95113), ('Part 4', 0.95079, 0.95154)],
    blend=[('lgb_diglead_bag', 'target-encoded view', 0.3333), ('glm_ohk_c01', 'joint GLM', 0.2222), ('lgb_dig_s100_bag', 'target-encoded view', 0.1111), ('ebm_dig', 'target-encoded view', 0.1111), ('lgb_rawdig_l4', 'raw view', 0.1111), ('lgb_dig_l7', 'target-encoded view', 0.1111)],
    title="S6E11 Credit Card Fraud  ·  Part 4",
    subtitle="Same models, more data: refit on every row, bag the seeds, and fit every value jointly in a GLM")
plt.show()

## 4.1 Refit on every training row

Every test prediction so far was the average of the five fold models, each trained on 80% of the training
rows. With per-value target encodings, 80% of the rows also means noisier encodings (in
`account_balance_usd` the median value has 3 rows). So each member gets one more model, trained on all
training rows:

- the encodings are fitted on all training rows (inner folds for the training rows themselves, as before),
- the number of trees is 1.25 x the mean best iteration of the five folds (25% more rows), no early stopping.

The out-of-fold predictions, and with them the CV and the blend weights, do not change; only the test
predictions do. That alone moved the Part 3 blend from LB 0.95113 to **0.95132**. Here is the refit of
Part 3's `lgb_diglead_l4` next to its 5-fold average (rank correlation 0.9955), and checked
against the refit stored in our dataset.

In [ ]:
ART = "s6e11-fraud-series-artifacts"


def refit_full(kind, XA, XT, te_cols, n_trees, smooth=20.0, seed=SEED, teseed=0, **over):
    """One model on every training row: encodings fitted on all of train, n_trees trees, no early stopping."""
    Xa, Xt = XA, XT
    if te_cols:
        Xa, (Xt,) = add_te(XA, y, [XT], te_cols, smooth, seed=teseed)
    Xa, Xt = drop_keys(Xa), drop_keys(Xt)
    if kind == "lgb":
        m = lgb.LGBMClassifier(**{**LGB, **over, "n_estimators": n_trees}, random_state=seed)
    else:
        params = {k: v for k, v in {**XGB, **over}.items() if k != "early_stopping_rounds"}
        m = xgb.XGBClassifier(**{**params, "n_estimators": n_trees}, random_state=seed)
    return m.fit(Xa, y).predict_proba(Xt)[:, 1]


n_trees = int(round(np.mean(ITS["lgb_diglead_l4"]) * 1.25))
FULL["lgb_diglead_l4"] = refit_full("lgb", XLtr, XLte, LEAD_TE, n_trees)
saved = np.load(find("lgb_diglead_l4_full.npy", ART))[:len(te)]
print(f"lgb_diglead_l4: the folds stopped at {ITS['lgb_diglead_l4']} trees -> refit with {n_trees}")
print(f"rank correlation, refit vs 5-fold average: {spearmanr(FULL['lgb_diglead_l4'], PRED['lgb_diglead_l4']).correlation:.4f}")
print(f"largest difference to the refit in our dataset: {np.abs(FULL['lgb_diglead_l4'] - saved).max():.1e}")

## 4.2 Bag the seeds

Two things are random in a tree member: the model seed (row and column subsampling) and the inner folds of
the target encoding. Changing both moves a 4-leaf LightGBM by up to 0.0002 CV, as much as many feature
ideas. The mean of three seeds is steadier than any one of them (5-fold CV AUC, probabilities averaged):

| member | seed 42, TE folds 0 | seed 1, TE folds 1 | seed 2, TE folds 2 | mean of the three |
|---|---|---|---|---|
| `lgb_diglead_l4` (Part 3) | 0.95040 | 0.95020 | 0.95023 | **0.95051** |
| `lgb_dig_s100` (Part 3) | 0.95029 | 0.95021 | 0.95014 | **0.95043** |

Five seeds gave 0.95054 for the first one: most of the gain is in the first three. Seed 42 happened to be
the lucky one (0.95040, the other four 0.95020-0.95035), a reminder of how much a single run can vary.
The bags are trained with the code in 4.4 (`RETRAIN_P4`).

## 4.3 A GLM that fits every value jointly

Part 3's GLM sees each value through its target encoding: the fraud rate of rows with that value, column by
column. Each encoding is marginal (it knows nothing about the other columns), and a rare value's encoding is
shrunk towards the global fraud rate.

Part 4's GLM adds a one-hot column for every exact value of every numeric column and for every digit key of
the float columns (values seen fewer than 5 times share a column), next to the encoding logits, and fits all
coefficients together under one L2 penalty. The coefficients then correct the encodings for each other, and
because a value's effect is the sum of its exact-value, `int(10x)`, `int(x)` and `int(x/10)` coefficients, a
rare value falls back to its coarser keys instead of to the global rate. The same idea as a tree feature
(encodings shrunk towards the parent key) did not help (4.5): it works here because the fit is joint.

| GLM, fold 0 | AUC |
|---|---|
| Part 3 GLM: splines + bins + TE logits of every key, C 0.3 | 0.94775 |
| one-hot of every value and key + splines, no TE, best C (0.03) | 0.94758 |
| both: TE logits + one-hot, C 0.03 | 0.94854 |
| both, C 0.01 (`glm_ohk_c01`, the Part 4 member) | 0.94915 |
| both, C 0.005 | 0.94918 |
| … + leading digits, more key levels, TE smoothing 50 (measured later) | 0.94941 |

With tens of thousands of one-hot columns it needs a much stronger penalty than Part 3's GLM (`C=0.01`
instead of 0.3). It trains in minutes, so it runs here; the blend below uses the predictions stored in our
dataset so that this notebook reproduces the submitted file exactly.

In [ ]:
def key_onehot(min_freq=5):
    """One-hot of every exact numeric value and of the digit keys of the float columns."""
    keys = {f"v_{c}": full[c] for c in NUMS}
    for c in FLOATS:
        i = np.floor(full[c]).astype("int64")
        keys.update({f"cents_{c}": np.round(full[c] * 100) % 100, f"int_{c}": i, f"ten_{c}": i // 10,
                     f"dec_{c}": np.floor(full[c] * 10 + 1e-6),
                     f"l2_{c}": i.astype(str).str.len() * 100 + i.astype(str).str[:2].astype(int)})
    K = pd.DataFrame({k: np.asarray(v, dtype=float) for k, v in keys.items()})
    return OneHotEncoder(handle_unknown="ignore", min_frequency=min_freq).fit_transform(K)


KO = key_onehot()
print(f"{KO.shape[1]:,} one-hot columns for values and digit keys\n")
run_glm("glm_ohk_c01", max_card=1000, C=0.01, with_te=True, src=(XDtr, XDte), te_cols=DIG_TE, extra=KO, refit=True)
for kind, arr, n in (("OOF", OOF, len(tr)), ("refit test", FULL, len(te))):
    saved = np.load(find(f"glm_ohk_c01_{'oof' if kind == 'OOF' else 'full'}.npy", ART))[:n]
    print(f"glm_ohk_c01 {kind}: largest difference to our dataset {np.abs(arr['glm_ohk_c01'] - saved).max():.1e}")

## 4.4 The Part 4 blend

The other members were trained outside this notebook (the three-seed bags with their refits take several
hours on 4 CPU cores); their out-of-fold and refit test predictions are loaded from our dataset, and the code
that made them is here (`RETRAIN_P4 = True` retrains them). Then the same rank hill climbing as before:
weights from the out-of-fold predictions, test side from the refits. Out of eleven candidates (also the
`C=0.03` GLM, the raw EBM, the depth-2 XGBoost, RealMLP and the keys-only LightGBM) it keeps these six. This
is Part 4's submission, written to `submission_part4.csv`.

In [ ]:
RETRAIN_P4 = False                      # True: retrain the seed bags and every refit here (several hours)
SEEDS = [(SEED, 0), (1, 1), (2, 2)]     # (model seed, seed of the encoding's inner folds)


def bag_trees(name, kind, XA, XT, te_cols, smooth=20.0, **over):
    """Three seeds of one tree member: mean out-of-fold prediction and mean refit test prediction."""
    oofs, fulls = [], []
    for s, ts in SEEDS:
        run_tree(f"{name}_seed{s}", kind, XA, XT, te_cols, smooth, seed=s, teseed=ts, **over)
        n = int(round(np.mean(ITS[f"{name}_seed{s}"]) * 1.25))
        oofs.append(OOF[f"{name}_seed{s}"])
        fulls.append(refit_full(kind, XA, XT, te_cols, n, smooth, seed=s, teseed=ts, **over))
    return np.mean(oofs, axis=0), np.mean(fulls, axis=0)


def tree_with_refit(name, kind, XA, XT, te_cols, **over):
    run_tree(name, kind, XA, XT, te_cols, **over)
    return OOF[name], refit_full(kind, XA, XT, te_cols, int(round(np.mean(ITS[name]) * 1.25)), **over)


def raw_digit_frames():
    """Raw columns + cents and the first / second decimal as plain numbers (no encodings)."""
    k = {c: np.round(full[c] * 100).astype("int64").values for c in FLOATS}
    RD = pd.concat([DG[CENTS], pd.DataFrame({n: v for c in FLOATS for n, v in
                    ((f"d1_{c}", ((k[c] // 10) % 10).astype("int8")), (f"d2_{c}", (k[c] % 10).astype("int8")))})], axis=1)
    return (pd.concat([Xtr, RD.iloc[:len(tr)].reset_index(drop=True)], axis=1),
            pd.concat([Xte, RD.iloc[len(tr):].reset_index(drop=True)], axis=1))


def ebm_refit():
    from interpret.glassbox import ExplainableBoostingClassifier
    Xa, (Xt,) = add_te(XDtr, y, [XDte], DIG_TE)
    m = ExplainableBoostingClassifier(interactions=10, max_bins=256, outer_bags=4, n_jobs=-1, random_state=42)
    return m.fit(drop_keys(Xa), y).predict_proba(drop_keys(Xt))[:, 1]


TRAIN_P4 = {
    "lgb_diglead_bag": lambda: bag_trees("lgb_diglead", "lgb", XLtr, XLte, LEAD_TE),
    "lgb_dig_s100_bag": lambda: bag_trees("lgb_dig_s100", "lgb", XDtr, XDte, DIG_TE, smooth=100.0),
    "glm_ohk_c01": lambda: (OOF["glm_ohk_c01"], FULL["glm_ohk_c01"]),          # trained live in 4.3
    "ebm_dig": lambda: (OOF["ebm_dig"], ebm_refit()),                          # OOF from Part 3
    "lgb_rawdig_l4": lambda: tree_with_refit("lgb_rawdig_l4", "lgb", *raw_digit_frames(), []),
    "lgb_dig_l7": lambda: tree_with_refit("lgb_dig_l7", "lgb", XDtr, XDte, DIG_TE, num_leaves=7),
}
P4_NAMES = ['lgb_diglead_bag', 'lgb_dig_s100_bag', 'glm_ohk_c01', 'ebm_dig', 'lgb_rawdig_l4', 'lgb_dig_l7']
for name in P4_NAMES:
    if RETRAIN_P4:
        OOF[name], FULL[name] = TRAIN_P4[name]()
    else:
        OOF[name] = np.load(find(f"{name}_oof.npy", ART))[:len(tr)].astype(float)
        FULL[name] = np.load(find(f"{name}_full.npy", ART))[:len(te)].astype(float)
    print(f"{name}: CV AUC {roc_auc_score(y, OOF[name]):.5f}")

Ro = np.column_stack([rankdata(OOF[n]) / len(y) for n in P4_NAMES])
Rp = np.column_stack([rankdata(FULL[n]) / len(te) for n in P4_NAMES])
w, auc = hill_climb(Ro, P4_NAMES)
print(f"\nPart 4 blend OOF AUC {auc:.6f}")
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P4_NAMES], "weight": w}, index=P4_NAMES).round(5))

sub = pd.DataFrame({"id": te["id"], TARGET: Rp @ w})
sub.to_csv("submission_part4.csv", index=False)
sub.head()

## 4.5 What the public board can and cannot tell

After this submission we kept going: more seeds, a GLM with more key levels (CV 0.95012), and EBMs with the
leading digits and 25 interactions (CV 0.95026, the best EBM so far). The CV kept rising, the public score
did not:

| submission | what changed | 5-fold CV | public LB |
|---|---|---|---|
| Part 4 | | 0.95079 | 0.95154 |
| `ens_v5_full` | 5 seeds per tree bag, improved GLM (CV 0.95012), 2 EBM seeds | 0.95085 | 0.95151 |
| `ens_v7_full` | fixed weights, EBM with leading digits and 25 interactions (CV 0.95026) | 0.95088 | 0.95145 |

The differences are 0.0001, inside the noise of a public board that scores 20% of the test rows (we
estimate the noise of the difference between two close submissions at about ±0.00014). The CV itself is optimistic for tuned
blends; scored honestly (weights chosen on four folds, scored on the fifth), the hill-climbing gain shrinks:

| blend weights | mean fold AUC |
|---|---|
| hill climbing, fitted on all OOF rows (in-sample) | 0.950843 |
| hill climbing, fitted on 4 folds, scored on the 5th | 0.950785 |
| weight grid, chosen on 4 folds, scored on the 5th | 0.950804 |
| equal weights on the 4 strongest bags | 0.950803 |

Simple equal weights on the four strongest bags score as well as the tuned weights. From here, the final
choice should follow honest CV rather than the public board.

## 4.6 What did not help in Part 4

| idea | result | note |
|---|---|---|
| GPT-2 tokens of the integer part as keys (`1269` → `12`·`69`) | −0.00016 (fold 0) | the digit keys already cover it |
| Hierarchical TE for the trees (rare values shrunk to their int(10x), int(x) parent) | −0.0003 (fold 0) | works in the GLM (jointly fitted), not as a tree feature |
| Inner folds of the TE: 10 / 20 instead of 5 | −0.0006 (fold 0) | the noise of 5 inner folds regularizes |
| LightGBM: lr 0.015, 3 leaves, colsample 0.3 / 0.8, path_smooth, min_child 50 / 200 / 300 | −0.00003 to −0.0003 | Part 3 settings are a local optimum |
| CatBoost on GPU with the digit keys as categoricals | 0.94603 (fold 0) |  |
| Factorization machine over per-value embeddings (pairwise terms) | 0.937–0.946 (fold 0) | pairs of values are noise: the signal is additive per column |
| DeepFM / embedding MLP | 0.9488 (fold 0) | RealMLP level |
| LightGBM boosted from the GLM's logit, or with it as an input | 0.94950 / 0.94921 (fold 0) | below the plain LightGBM; combine by rank blending only |
| Is the value in the original data? Value counts, velocity residual | no residual signal | screened on top of the best member |
| Digit key × category pairs (96) | max +0.00006 | noise |

---
# Part 5 — Three leaves, a plain GLM, and a public board that disagrees

*Public LB 0.95160 · 5-fold CV 0.95092*

Part 5 started from the public notebooks posted after Part 4 (credits at the top). Two of their ideas hold up
on our folds: trees with three leaves, and a GLM without the extra blocks ours carried. One does not: GPT-2
token keys. Then the public board weighs in, and Part 4's refit is checked on held-out folds.

In [ ]:
draw_part5(
    stats=[('Public LB', '0.95160', '+0.00006 over Part 4'), ('5-fold CV AUC', '0.95092', '+0.00013 over Part 4'), ('Plain GLM, CV', '0.95051', "+0.0006 over Part 4's GLM"), ('Its public verdict', '−0.00018', 'swapped into Part 4: CV +0.00013')],
    leaves={'seeds': {'2': [0.950204575897117], '3': [0.9506444964240918, 0.9503154204418328, 0.9503331027872026], '4': [0.9503999740916812, 0.9501976208180694, 0.9502346223233641]}, 'bag': {'3': 0.950626761687232, '4': 0.9505103584240805}},
    glm_design=[('TE logits + raw columns (plain)', 0.949617), ('+ TE smoothing 50 instead of 20', 0.949759), ('+ splines', 0.949316), ('+ one-hot of values (≤ 1000 distinct)', 0.947958), ('+ both (the Part 1-3 GLM design)', 0.947762), ('+ GPT-2 token keys', 0.949509), ('+ one-hot values and keys, C 0.01 (Part 4)', 0.949368)],
    cvlb=[('Part 3 refit', 0.950728, 0.95132, 'base'), ('Part 4', 0.95079, 0.95154, 'base'), ('5 seeds', 0.950846, 0.95151, 'base'), ('+ EBM i25', 0.950883, 0.95145, 'base'), ('3 leaves', 0.950818, 0.95154, 'base'), ('plain GLM swap', 0.950914, 0.95136, 'glm'), ('CV best, refit', 0.951032, 0.95132, 'glm'), ('CV best, refit + folds', 0.951032, 0.95119, 'glm'), ('half plain GLM', 0.950865, 0.95147, 'glm'), ('CV best without it (Part 5)', 0.950924, 0.9516, 'base')],
    refit=[('fold 0', 0.949717, 0.949734, 0.949807), ('fold 1', 0.951252, 0.951252, 0.951336), ('fold 2', 0.955742, 0.955561, 0.95572)],
    blend=[('lgb_diglead_l3_bag', 'target-encoded view', 0.2222), ('glm_ohk2_bag', 'joint GLM', 0.2222), ('ebm_i25_bag4', 'target-encoded view', 0.2222), ('lgb_dig_s100_bag5', 'target-encoded view', 0.1111), ('lgb_rawdig_l4', 'raw view', 0.1111), ('lgb_dig_l7', 'target-encoded view', 0.1111)],
    title="S6E11 Credit Card Fraud  ·  Part 5",
    subtitle="Three leaves, a plain GLM, and a public board that disagrees with the CV")
plt.show()

## 5.1 Three leaves, not four

Part 1 settled on 4-leaf trees from fold 0, where 3 leaves scored 0.00003 lower. On all five folds, with
three seeds each (model seed and the encoding's inner folds, as in Part 4), 3 leaves win:

| LightGBM, leading-digit features | seed 42 / 1 / 2 | mean of the seeds | three-seed bag |
|---|---|---|---|
| 2 leaves | 0.95020 | 0.95020 |  |
| 3 leaves | 0.95064 / 0.95032 / 0.95033 | 0.95043 | **0.95063** |
| 4 leaves | 0.95040 / 0.95020 / 0.95023 | 0.95028 | **0.95051** |

Seed 42 happened to be the lucky seed again (0.95064); the honest gain is the one on the seed mean,
+0.00015. One fold is not enough to choose between settings this close: the same happened with column
sampling, where fold 0 alone said −0.00026 and five folds say ±0.00005. One seed of the 3-leaf model:

In [ ]:
run_tree("lgb_diglead_l3", "lgb", XLtr, XLte, LEAD_TE, num_leaves=3)
saved = np.load(find("lgb_diglead_l3_oof.npy", ART))[:len(tr)]
print(f"lgb_diglead_l3 OOF: largest difference to our dataset {np.abs(OOF['lgb_diglead_l3'] - saved).max():.1e}")

## 5.2 A plain GLM is the strongest GLM

Since Part 1 our GLMs have had cubic splines of every numeric column and one-hot columns for the values of
the numeric columns with up to 1000 distinct values, next to the target-encoding logits. A public notebook
fitted the logits with nothing else and scored higher. Adding our blocks back one at a time (fold 0,
`C=0.3`, every value and digit key encoded):

| GLM, fold 0 | AUC | change |
|---|---|---|
| TE logits + raw columns (plain) | 0.94962 | |
| + TE smoothing 50 instead of 20 | 0.94976 | +1.4 × 1e-4 |
| + splines | 0.94932 | -3.0 × 1e-4 |
| + one-hot of values (≤ 1000 distinct) | 0.94796 | -16.6 × 1e-4 |
| + both (the Part 1-3 GLM design) | 0.94776 | -18.6 × 1e-4 |
| + GPT-2 token keys | 0.94951 | -1.1 × 1e-4 |
| + one-hot values and keys, C 0.01 (Part 4) | 0.94937 | -2.5 × 1e-4 |

The encodings already give each column its shape, value by value. The value one-hots add about 1,500 free
coefficients that re-learn the same shape from fewer rows each and cost 0.0017; the splines cost a little
too. Part 4's joint one-hot GLM was probably in part repairing that damage with its strong penalty: on top of the plain
design the same one-hots cost 0.0002 to 0.0026. The penalty itself hardly matters for the plain GLM
(`C` 0.1 / 0.3 / 1.0: 0.94962 / 0.94962 / 0.94962); smoothing the encodings more (50 instead of 20) helps a
little. Five folds:

| GLM | 5-fold CV AUC |
|---|---|
| Part 3: splines + one-hot + TE logits (`glm_dig`) | 0.94879 |
| Part 4: + joint one-hot of values and keys (`glm_ohk_c01`) | 0.94991 |
| Part 5: plain, TE smoothing 20 (`glm_dl`) | 0.95041 |
| Part 5: plain, TE smoothing 50, 3 seeds (`glm_dl_s50_bag`) | 0.95051 |

It trains in a few minutes, so it runs here:

In [ ]:
def plain_design():
    """Raw numeric columns (log for the skewed ones), standardized; the flags; one-hot categoricals. Nothing else."""
    num = full[NUMS].astype(float).copy()
    for c in ("amount_usd", "account_balance_usd", "hours_since_last_txn"):
        num[c] = np.log1p(num[c])
    return sparse.hstack([sparse.csr_matrix(StandardScaler().fit_transform(num)),
                          sparse.csr_matrix(full[FLAGS].values.astype(float)),
                          OneHotEncoder(handle_unknown="ignore").fit_transform(full[CATS].astype(str))]).tocsr()


run_glm("glm_dl_s50", max_card=0, C=0.3, with_te=True, src=(XLtr, XLte), te_cols=LEAD_TE, base=plain_design(), smooth=50.0)
saved = np.load(find("glm_dl_s50_oof.npy", ART))[:len(tr)]
print(f"glm_dl_s50 OOF: largest difference to our dataset {np.abs(OOF['glm_dl_s50'] - saved).max():.1e}")
print(f"rank correlation with Part 4's GLM: {spearmanr(OOF['glm_dl_s50'], OOF['glm_ohk_c01']).correlation:.4f}, "
      f"with the 3-leaf LightGBM: {spearmanr(OOF['glm_dl_s50'], OOF['lgb_diglead_l3']).correlation:.4f}")

## 5.3 GPT-2 token keys: no gain

If the generator is a language model, it writes `1269.35` as the GPT-2 tokens `' 12' '69' '.' '35'`, not
digit by digit. Keys built from those tokens (first, second and last token, the number of tokens, first
token with the count, the integer-part tokens) for all six float columns, target-encoded like the digit keys,
read from the CSV text:

```python
from tokenizers import Tokenizer
gpt2 = Tokenizer.from_pretrained("gpt2")
tokens = {v: gpt2.encode(" " + v).tokens for v in csv_text[col].unique()}   # ' 1269.35' -> ['Ġ12', '69', '.', '35']
```

| GLM, fold 0 | without | with the token keys |
|---|---|---|
| plain GLM | 0.94962 | 0.94951 |
| GLM with splines + one-hot | 0.94776 | 0.94761 |

The token keys cut the numbers at other places than the digit keys, but they add nothing the digit keys do
not already carry (Part 4 found the same for the trees). The plain GLM's lead over ours came from the design,
not from the tokens.

## 5.4 Refit or fold average, on held-out folds

Part 4 replaced every test prediction by one refit on all training rows because one public-board comparison
said +0.00019. Public notebooks measured the opposite for deeper XGBoost. So here is the comparison on our
own data, one level down: each of three folds is held out as a "test set", and the other four folds play the
training set. On them, the 3-leaf recipe is trained exactly as for the real test set (four fold models,
early stopping, three seeds) and scored on the held-out fold:

| held out | fold average | refit on all four folds | rank mean of both |
|---|---|---|---|
| fold 0 | 0.94972 | 0.94973 (+0.2) | 0.94981 (+0.9) |
| fold 1 | 0.95125 | 0.95125 (+0.0) | 0.95134 (+0.8) |
| fold 2 | 0.95574 | 0.95556 (-1.8) | 0.95572 (-0.2) |

(AUC, changes vs the fold average × 1e-4.) Single models: the refit loses to the four-model average in 7 of
9 seeds. As three-seed bags it is no better on average (−0.00005), while the rank mean of the two is best
on average (+0.00005). The public board disagrees once more: the CV-best blend scored 0.95132 with refits and
0.95119 with the rank mean. The differences are small either way; the code is below (`RUN_REFIT_CHECK`).

In [ ]:
RUN_REFIT_CHECK = False      # True: 3 held-out folds x 3 seeds x 5 LightGBM fits (an hour or more on 4 cores)


def refit_check(k, seeds=((SEED, 0), (1, 1), (2, 2)), **over):
    """Fold k as the test set, the other four folds as the training set; returns AUC of (a), (b), (c)."""
    hold, inner = FOLDS[k][1], [FOLDS[j][1] for j in range(N_FOLDS) if j != k]
    pool, res = np.concatenate(inner), []
    for s, ts in seeds:
        avg, its = np.zeros(len(hold)), []
        for j, b in enumerate(inner):
            a = np.concatenate([inner[i] for i in range(len(inner)) if i != j])
            Xa, (Xb, Xh) = add_te(XLtr.iloc[a], y[a], [XLtr.iloc[b], XLtr.iloc[hold]], LEAD_TE, seed=ts)
            m = lgb.LGBMClassifier(**{**LGB, **over}, random_state=s)
            m.fit(drop_keys(Xa), y[a], eval_set=[(drop_keys(Xb), y[b])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(300, verbose=False)])
            avg += m.predict_proba(drop_keys(Xh))[:, 1] / len(inner)
            its.append(m.best_iteration_)
        Xa, (Xh,) = add_te(XLtr.iloc[pool], y[pool], [XLtr.iloc[hold]], LEAD_TE, seed=ts)
        m = lgb.LGBMClassifier(**{**LGB, **over, "n_estimators": int(round(np.mean(its) * 1.25))}, random_state=s)
        res.append((avg, m.fit(drop_keys(Xa), y[pool]).predict_proba(drop_keys(Xh))[:, 1]))
    a, b = np.mean([r[0] for r in res], axis=0), np.mean([r[1] for r in res], axis=0)
    return [roc_auc_score(y[hold], v) for v in (a, b, (rankdata(a) + rankdata(b)) / 2)]


if RUN_REFIT_CHECK:
    for k in range(3):
        print(f"held-out fold {k}: fold average, refit, rank mean =", np.round(refit_check(k, num_leaves=3), 6))

## 5.5 What the public board said

Six submissions after Part 4. "Expected" is the CV difference to Part 4 and the standard deviation of that
difference over random 60,000-row subsets of our out-of-fold rows (the size of the public board):

| submission | 5-fold CV | expected vs Part 4 | public LB (Part 4: 0.95154) |
|---|---|---|---|
| CV best: 3-leaf bag, plain GLM 3/9, EBM i25, ... (refit) | 0.95103 | +0.00024 ± 0.00016 | 0.95132 |
| same, test = rank mean of fold average and refit | 0.95103 | +0.00024 ± 0.00016 | 0.95119 |
| Part 4 with the 3-leaf bag | 0.95082 | +0.00003 ± 0.00006 | 0.95154 |
| Part 4 with the plain GLM | 0.95091 | +0.00013 ± 0.00010 | 0.95136 |
| Part 4 with half of its GLM weight on the plain GLM | 0.95086 | +0.00008 ± 0.00005 | 0.95147 |
| **CV best without the plain GLM (this notebook)** | 0.95092 | +0.00014 ± 0.00010 | **0.95160** |

The 3-leaf swap landed where expected. The GLM swap landed about three standard deviations below its
expectation, and so did the CV-best blend (which gives the plain GLM 3/9). Half the GLM weight cost about
half as much; that is consistent, but not independent evidence (the same public rows score the same GLM
predictions). Without the plain GLM, the best-CV blend landed within its expected range, at 0.95160. On 60,000 public rows that is unlikely
to be chance alone, but Part 4's 0.95154 was also the best of several submissions, so it is probably a
little lucky itself. We looked for something in the test rows that the plain GLM could be reacting to and
found nothing: its test predictions correlate with the other members exactly as its out-of-fold
predictions do (0.989 vs 0.986 with the 4-leaf bag), its logits are spread the same on training, validation
and test rows, and its largest coefficient is 0.7.

So two readings remain: the public 20% is noise and the CV (700,000 rows) is right, or the plain GLM is
worse on the test rows for a reason we have not found. The private board (the other 80%) decides. Kaggle
lets us pick two final submissions; ours will be one of each: the CV-best blend and the blend both
sides agree on (this notebook's `submission.csv`).

## 5.6 The Part 5 blend

The best-CV blend without the plain GLM: the 3-leaf bag; the joint GLM of Part 4 with more key levels
(one-hots also of `int(x/100)`, the first digit, the ones digit and both decimals; the leading digits
encoded; TE smoothing 50, `C=0.005`; two encoding seeds); the EBM with the leading digits and 25
interactions (four seeds); the smoothing-100 LightGBM with five seeds; Part 3's raw-view and 7-leaf trees.
Weights by rank hill climbing on the out-of-fold predictions, test predictions from the refits as in
Part 4: the notebook's `submission.csv`. The CV-best blend (with the plain GLM) goes to
`submission_cv_best.csv`. The bags are loaded from our dataset; `RETRAIN_P5 = True` retrains the 3-leaf bag
with Part 4's `bag_trees` (the other three are variants of the Part 3 and Part 4 code).

In [ ]:
RETRAIN_P5 = False
P5_W = {'lgb_diglead_l3_bag': 2, 'glm_ohk2_bag': 2, 'ebm_i25_bag4': 2, 'lgb_dig_s100_bag5': 1, 'lgb_rawdig_l4': 1, 'lgb_dig_l7': 1}
CV_BEST_W = {'lgb_diglead_l3_bag': 1, 'glm_dl_s50_bag': 3, 'lgb_dig_s100_bag5': 1, 'ebm_i25_bag4': 2, 'lgb_rawdig_l4': 1, 'lgb_dig_l7': 1}


def rank_blend(weights, side):
    w = np.array(list(weights.values()), dtype=float)
    return np.column_stack([rankdata(side[n]) / len(side[n]) for n in weights]) @ (w / w.sum())


for name in ("lgb_diglead_l3_bag", "glm_dl_s50_bag", "glm_ohk2_bag", "lgb_dig_s100_bag5", "ebm_i25_bag4"):
    if RETRAIN_P5 and name == "lgb_diglead_l3_bag":
        OOF[name], FULL[name] = bag_trees("lgb_diglead_l3", "lgb", XLtr, XLte, LEAD_TE, num_leaves=3)
    else:
        OOF[name] = np.load(find(f"{name}_oof.npy", ART))[:len(tr)].astype(float)
        FULL[name] = np.load(find(f"{name}_full.npy", ART))[:len(te)].astype(float)

for label, W in (("Part 5 blend", P5_W), ("CV-best blend", CV_BEST_W)):
    print(f"{label}: OOF AUC {roc_auc_score(y, rank_blend(W, OOF)):.6f}")
display(pd.DataFrame({"CV AUC": [roc_auc_score(y, OOF[n]) for n in P5_W],
                      "weight": [f"{v}/9" for v in P5_W.values()]}, index=list(P5_W)).round(5))

pd.DataFrame({"id": te["id"], TARGET: rank_blend(CV_BEST_W, FULL)}).to_csv("submission_cv_best.csv", index=False)
sub = pd.DataFrame({"id": te["id"], TARGET: rank_blend(P5_W, FULL)})
sub.to_csv("submission_part5.csv", index=False)     # Part 6 writes submission.csv
sub.head()

## 5.7 What did not help in Part 5

| idea | result | note |
|---|---|---|
| GPT-2 token keys (first, second, last token, count, …) of all six float columns | −0.00011 (GLM, fold 0) | idea from bulsahkecici; also no gain in our trees (Part 4) |
| Splines and one-hot values in the GLM next to the TE logits | −0.0003 / −0.0017 (fold 0) | the encodings already carry the shape |
| Joint one-hot keys (Part 4) on top of the plain GLM | −0.0002 to −0.0026 (fold 0) | C 0.01 / 0.03 / 0.1 |
| 2 leaves | 0.95021 (5-fold) | 3 leaves 0.95064, 4 leaves 0.95040 (seed 42) |
| GLM: TE smoothing 5 / 100 / 200 | −0.0003 / +0.00015 / +0.00004 (fold 0) | 50 kept (+0.00014) |
| GLM: more digit keys (ones and tens digit, int(x/100)) | −0.00005 / 0.0000 (fold 0) |  |
| LightGBM column sampling 1.0 instead of 0.5 | ±0.00005 (5-fold) | digit-key gain the same with and without (asked in the comments) |

**Planned after Part 5:** - Find out why the plain GLM wins on 700k out-of-fold rows and loses on the public 20%.
- Pick the two final submissions so that one follows the CV and one the public board.

---
# Part 6 — Start the trees from the GLM, then stack what everyone published

*Public LB 0.95215 · nested CV 0.95163*

Between Part 5 and Part 6 the public board moved by 0.0005. This part looks at where that came from, keeps
what holds up on our folds, and checks every step on the public board once.

In [ ]:
draw_part6(
    stats=[('Public LB', '0.95215', '+0.00055 over Part 5'), ('Nested CV of the stack', '0.95163', 'stacker fitted on 4 folds, scored on the 5th'), ('GLM → trees, 5 seeds', '0.95151', '+0.00058 over the Part 5 blend'), ('Members', '62', '7 ours + 55 public, same folds')],
    ladder=[('GLM on the encodings', 0.9505105987910156), ('4-leaf LightGBM, same inputs', 0.9506947470100463), ('Part 5 blend (6 members)', 0.9509244015947096), ('GLM → 4-leaf trees, one seed', 0.9513972450955384), ('… mean of 5 seeds (bag)', 0.9515058163094932)],
    cvlb=[('Part 4', 0.95079, 0.95154, 'mix'), ('Part 5', 0.950924, 0.9516, 'mix'), ('CV-best with plain GLM', 0.951032, 0.95132, 'glm'), ('plain GLM swap', 0.950914, 0.95136, 'glm'), ('A: GLM boost alone', 0.951506, 0.95154, 'glm'), ('D: same, rank-mean test', 0.951506, 0.95146, 'glm'), ('B: 0.3 Part 5 + 0.7 GLM boost', 0.951558, 0.95177, 'mix'), ('C: stack, stopped early', 0.951549, 0.9521, 'mix'), ('E: stack, 72 members', 0.951573, 0.95207, 'mix'), ('Part 6: stack, converged', 0.951629, 0.95215, 'mix')],
    nested=[('our 10 members', 0.950965), ('+ 55 public members', 0.951511), ('+ our GLM boost bag', 0.951549), ('… stack solved to convergence', 0.951629)],
    coef=[('gb_bag5', 0.3801215363378178), ('mega:glm_boost', 0.26709671587590367), ('maly:ebm_private_int5', 0.18992300263849626), ('mega:glm_boost_xgb', 0.1544505575445553), ('alex:RealMLP_digits', -0.15213989364475838), ('alex:XGBoost_tuned', -0.14485589413717884), ('mega:glm_boost_raw', 0.12074380185556811), ('mega:lgb_raw', 0.11295368201725411), ('lgb_rawdig_l4', 0.11159839354931096), ('mega:glm_gen', -0.10053677553481459)],
    ours={'lgb_dig_s100_bag5', 'glm_ohk2_bag', 'lgb_dig_l7', 'ebm_i25_bag4', 'lgb_rawdig_l4', 'lgb_diglead_l3_bag', 'gb_bag5'},
    title="S6E11 Credit Card Fraud  ·  Part 6",
    subtitle="Start the trees from the GLM, then stack 62 members from 7 authors on the same folds")
plt.show()

## 6.1 What is at the top of the public board

We read the highest-scoring public notebooks first. The files at 0.9519-0.9521 are built from each other's
**public submission files**: one takes 80% of another's file and 20% of its own stack; the other extrapolates
the rank change between two public files three times further; a third averages three of them. Their authors
say so openly, and also that there is no out-of-fold score for these recipes: the public board (20% of the
test rows) is the only judge. We do not use them. Two things in the trained notebooks below them do hold up
on our folds, and they are the rest of this part:

* **a better single model**: trees boosted from a GLM (megayak, 6.2);
* **pooling published out-of-fold predictions** on the same folds into one stack (lazyyeou, Nilanjan_Saha123,
  6.4). Many authors publish their members' predictions on the split this series has used since Part 1.

## 6.2 Start the trees from the GLM

The idea comes from [megayak's notebook](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack)
and its picture of the generator: it draws the label first and then writes every field *given the label*,
so most of the signal is additive (each value, each digit pushes the log-odds by its own amount) plus a few
interactions. A logistic regression gets the additive part right but cannot bend; trees can bend but spend
most of their splits re-learning the additive part. So:

1. fit a GLM on the target-encoding logits of every value, digit key and GPT-2 token key, a few joint keys
   (merchant category × risk score, foreign × IP mismatch, …) and token rates pooled across columns;
2. give a 4-leaf LightGBM the GLM's logit as its starting score (`init_score`), so its trees fit only the
   residual. The trees see the raw columns, the encodings and **how many rows stand behind each encoding**
   (the value count of every key).

Inside each fold the GLM is fitted on that fold's training rows only, so the held-out rows are scored
honestly. Fold by fold (seed 0):

| fold | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| GLM | 0.94942 | 0.95066 | 0.95551 | 0.95148 | 0.94545 |
| GLM → 4-leaf trees | **0.95039** | **0.95156** | **0.95612** | **0.95223** | **0.94671** |

The script below is megayak's feature and encoding code (Apache 2.0) with our driver at the end: the
encoding seed (`teseed`, the inner split of the target encoding), the model seed and a few shape options are
arguments, so the member can be bagged. It also refits on all training rows for the test predictions (our
Part 4 refit, which the public board has preferred four times out of four). One seed runs here (about 40
minutes on Kaggle's 4 cores) and is compared with our dataset. Our dataset was trained on an ARM machine, so
LightGBM grows slightly different trees on Kaggle: the OOF AUC agrees to 0.00001, the printed largest difference
in logits does not. Only the check uses this run; the blend reads the dataset:

In [ ]:
%%writefile gboost.py
"""GLM-boost experiments for Part 6 (built on megayak's public notebook v7, Apache 2.0:
https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack).

Keys, out-of-fold TE, joint keys and the token-pooling GLM inputs are megayak's code (copied from
src/megayak_v7.py); the driver at the bottom is ours: inner-TE seed / model seed, leaves, learning rate, GLM
design and the test-prediction kind are options, so a member can be bagged and varied.

Usage: INPUT_DIR=cache/kin python src/gboost.py <name> [teseed=0] [mseed=42] [leaves=4] [lr=0.02] [kind=lgb]
                                                [glm=gen|gen_ohk] [C=0.3] [ohkC=...] [view=joint|raw] [cs=0.5]
Writes cache/gboost/{oof,foldavg,refit,glmoof,glmrefit}_<name>.npy
"""
display = print
import os, glob, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

INPUT = os.environ.get("INPUT_DIR", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
NJOBS = int(os.environ.get("NJOBS", os.cpu_count()))
T0 = time.time()


def log(*a):
    print(f"[{time.time() - T0:6.0f}s]", *a, flush=True)


def find(name):
    hits = sorted(glob.glob(os.path.join(INPUT, "**", name), recursive=True))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


TARGET = "is_fraud"
CATS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
FLAGS = ["is_foreign_transaction", "is_new_merchant", "used_vpn",
         "ip_country_mismatch", "billing_shipping_mismatch", "is_ai_generated_scam_attempt"]
NUMS = ["amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km",
        "card_age_months", "customer_age", "account_balance_usd", "cvv_retry_count",
        "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score", "prior_disputes"]
FEATS = NUMS + FLAGS + CATS
FLOATS = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd",
          "velocity_score", "merchant_risk_score"]

tr = pd.read_csv(find("train.csv"))
te = pd.read_csv(find("test.csv"))
for df in (tr, te):
    df[FLAGS] = df[FLAGS].astype("int8")
y = tr[TARGET].astype(int).values
NTR, NTE = len(tr), len(te)
full = pd.concat([tr[FEATS], te[FEATS]], ignore_index=True)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(NTR), y))
OOF, PRED = {}, {}
def key_table(df):
    K = {}
    for c in FEATS:
        K[f"v_{c}"] = df[c].astype(str).values if c in CATS else df[c].values
    for c in FLOATS:
        x = df[c].values
        K[f"cents_{c}"] = (np.round(x * 100) % 100).astype(np.int64)
        K[f"int_{c}"] = np.floor(x).astype(np.int64)
        K[f"ten_{c}"] = np.floor(x / 10).astype(np.int64)
        K[f"dec_{c}"] = np.floor(x * 10 + 1e-6).astype(np.int64)
    for c in FLOATS:
        i = pd.Series(np.floor(df[c].values).astype(np.int64)).astype(str)
        K[f"l1_{c}"] = (i.str.len() * 10 + i.str[0].astype(int)).values
        K[f"l2_{c}"] = (i.str.len() * 100 + i.str[:2].astype(int)).values
    return K


STR, TOK = None, None     # the floats as written, and their GPT-2 tokens (kept for section 3)


def token_keys():
    global STR, TOK
    try:
        from tokenizers import Tokenizer
        gpt2 = Tokenizer.from_file(find("gpt2-tokenizer.json"))
    except Exception as e:
        log(f"GPT-2 tokenizer unavailable ({type(e).__name__}), token keys skipped")
        return {}
    STR = pd.concat([pd.read_csv(find("train.csv"), usecols=FLOATS, dtype=str),
                     pd.read_csv(find("test.csv"), usecols=FLOATS, dtype=str)], ignore_index=True)
    TOK, K = {}, {}
    for c in FLOATS:
        p = TOK[c] = STR[c].map({v: gpt2.encode(" " + v).tokens for v in STR[c].unique()})
        K[f"t1_{c}"] = p.str[0].values
        K[f"t2_{c}"] = p.map(lambda t: t[1] if len(t) > 1 else "").values
        K[f"tl_{c}"] = p.str[-1].values
        K[f"tn_{c}"] = p.str.len().values
    return K


KEYS = key_table(full)
KEYS.update(token_keys())
KNAMES = list(KEYS)
codes = [pd.factorize(pd.Series(KEYS[n]))[0].astype(np.int32) for n in KNAMES]
KMAT = np.column_stack(codes)
NCAT = np.array([c.max() + 1 for c in codes])
del KEYS, codes

# raw view: the 24 columns as numbers (categoricals as codes) plus the cents of each float
RAW = full[NUMS + FLAGS].copy()
for c in CATS:
    RAW[c] = pd.factorize(full[c], sort=True)[0].astype(np.int16)
for c in FLOATS:
    RAW[f"cents_{c}"] = (np.round(full[c].values * 100) % 100).astype(np.int16)
RAWV = RAW.values.astype(np.float32)
CAT_IDX = [list(RAW.columns).index(c) for c in CATS]
log(f"{len(KNAMES)} target-encoded keys, {RAW.shape[1]} raw columns")
pass

# the 13 pairs from the offline scan; "q:" = deciles of a float (category x risk was scanned with risk deciles and
# is keyed on the exact score, which the G check below and the GLM both prefer)
JOINT = [("j_category_risk", ("merchant_category", "merchant_risk_score")),
         ("j_foreign_ip", ("is_foreign_transaction", "ip_country_mismatch")),
         ("j_category_amount", ("merchant_category", "q:amount_usd")),
         ("j_txn_velocity", ("txn_count_last_24h", "q:velocity_score")),
         ("j_category_age", ("merchant_category", "customer_age")),
         ("j_amount_risk", ("q:amount_usd", "q:merchant_risk_score")),
         ("j_hours_velocity", ("q:hours_since_last_txn", "q:velocity_score")),
         ("j_age_risk", ("customer_age", "q:merchant_risk_score")),
         ("j_newmerchant_weekday", ("is_new_merchant", "day_of_week")),
         ("j_scam_velocity", ("is_ai_generated_scam_attempt", "q:velocity_score")),
         ("j_age_cardage", ("customer_age", "card_age_months")),
         ("j_hour_weekday", ("time_of_day_hour", "day_of_week")),
         ("j_txn_age", ("txn_count_last_24h", "customer_age"))]


def joint_col(df, c):
    return pd.qcut(df[c[2:]], 10, labels=False, duplicates="drop") if c.startswith("q:") else df[c]


def joint_code(df, cols):
    k = np.zeros(len(df), np.int64)
    for c in cols:
        u = pd.factorize(joint_col(df, c))[0]
        k = k * (u.max() + 1) + u
    return pd.factorize(k)[0]


lgt1 = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
rows, JCODES = [], {}
for name, cols in JOINT:
    JCODES[name] = joint_code(full, cols).astype(np.int32)
    k = JCODES[name][:NTR]
    add = np.full(NTR, -(len(cols) - 1) * lgt1(y.mean()))
    dof = 0
    for c in cols:
        u = pd.factorize(joint_col(full, c))[0][:NTR]
        add += lgt1(np.bincount(u, weights=y) / np.bincount(u))[u]
        dof += u.max()
    s, n = np.bincount(k, weights=y), np.bincount(k)
    r = np.bincount(k, weights=1 / (1 + np.exp(-add))) / n
    with np.errstate(divide="ignore", invalid="ignore"):
        g = np.where(s > 0, s * np.log(s / (n * r)), 0) + np.where(n > s, (n - s) * np.log((n - s) / (n * (1 - r))), 0)
    rows.append(dict(key=name, cells=(n > 0).sum(), G=2 * g.sum(), df=(n > 0).sum() - 1 - dof))
GT = pd.DataFrame(rows).set_index("key")
GT["kept"] = GT["G"] > 1.2 * GT["df"]
USED = list(GT.index[GT["kept"]])
NBASE = KMAT.shape[1]          # the trees use the first NBASE keys, the GLM all of them (section 3)
KMAT = np.column_stack([KMAT] + [JCODES[n] for n in USED])
NCAT = np.append(NCAT, [JCODES[n].max() + 1 for n in USED])
KNAMES += USED
log(f"{len(USED)} joint keys kept -> {len(KNAMES)} target-encoded keys")

tab = pd.DataFrame({"foreign": tr["is_foreign_transaction"].values, "ip mismatch": tr["ip_country_mismatch"].values,
                    "y": y})
obs = tab.groupby(["foreign", "ip mismatch"])["y"].agg(["mean", "size"])
ra, rb = tab.groupby("foreign")["y"].mean(), tab.groupby("ip mismatch")["y"].mean()
obs["additive"] = [1 / (1 + np.exp(-(lgt1(ra[i]) + lgt1(rb[j]) - lgt1(y.mean())))) for i, j in obs.index]

def te_rate(k, t, ncat, prior, smooth):
    s = np.bincount(k, weights=t, minlength=ncat)
    n = np.bincount(k, minlength=ncat)
    return (s + prior * smooth) / (n + smooth)


def te_fold(a, b, smooth=20.0, inner=5, seed=0):
    ya = y[a].astype(np.float64)
    Ka, Kb, Kt = KMAT[a], KMAT[b], KMAT[NTR:]
    Ea = np.zeros(Ka.shape, np.float32)
    Eb = np.zeros(Kb.shape, np.float32)
    Et = np.zeros(Kt.shape, np.float32)
    inner_f = list(StratifiedKFold(inner, shuffle=True, random_state=seed).split(np.zeros(len(a)), ya))
    for j in range(KMAT.shape[1]):
        for i, o in inner_f:
            Ea[o, j] = te_rate(Ka[i, j], ya[i], NCAT[j], ya[i].mean(), smooth)[Ka[o, j]]
        r = te_rate(Ka[:, j], ya, NCAT[j], ya.mean(), smooth)
        Eb[:, j], Et[:, j] = r[Kb[:, j]], r[Kt[:, j]]
    return Ea, Eb, Et


t = time.time()
log(f"one outer fold encoded in {time.time() - t:.1f}s")
LGB = dict(n_estimators=20000, learning_rate=0.03, num_leaves=4, min_child_samples=100, subsample=0.8,
           subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0, max_bin=255, verbose=-1)
XGB = dict(n_estimators=20000, learning_rate=0.03, max_depth=2, subsample=0.8, colsample_bytree=0.5,
           min_child_weight=5, reg_lambda=1.0, max_bin=256, tree_method="hist", eval_metric="auc",
           early_stopping_rounds=300)
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


POOL2 = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd"]
POOL1 = ["velocity_score", "merchant_risk_score"]
GEN = TOK is not None
if GEN:
    dec = lambda c: STR[c].str.split(".").str[1].astype(str)
    lit = lambda c: TOK[c].map(lambda t: t[t.index(".") - 1] if "." in t else t[-1])
    GROUPS = []
    for mem, fn in [(POOL2, dec), (POOL2, lambda c: dec(c).str[0]), (POOL1, dec), (POOL1, lit)]:
        v = pd.concat([fn(c) for c in mem], ignore_index=True)
        GROUPS.append(pd.factorize(v)[0].astype(np.int32).reshape(len(mem), -1).T.copy())
    MARKOV = [pd.factorize(lit(c) + "|" + dec(c))[0].astype(np.int32) for c in POOL2]


def gen_encode(i, o, m0=50.0, m1=2000.0, m2=200.0):
    yi = y[i].astype(np.float64)
    prior, cols, field0 = yi.mean(), [], []
    for g, C in enumerate(GROUPS):
        ncat = int(C.max()) + 1
        s = [np.bincount(C[i, j], weights=yi, minlength=ncat) for j in range(C.shape[1])]
        n = [np.bincount(C[i, j], minlength=ncat) for j in range(C.shape[1])]
        pool = (sum(s) + prior * m0) / (sum(n) + m0)
        for j in range(C.shape[1]):
            h = ((s[j] + m1 * pool) / (n[j] + m1))[C[o, j]]
            cols.append(h)
            if g == 0:
                field0.append(h)
    for j, M in enumerate(MARKOV):
        ncat = int(M.max()) + 1
        s, n = np.bincount(M[i], weights=yi, minlength=ncat), np.bincount(M[i], minlength=ncat)
        cols.append((s[M[o]] + m2 * field0[j]) / (n[M[o]] + m2))
    return np.column_stack(cols).astype(np.float32)


def gen_fold(a, b, inner=5, seed=0):
    Ga = None
    for i, o in StratifiedKFold(inner, shuffle=True, random_state=seed).split(np.zeros(len(a)), y[a]):
        E = gen_encode(a[i], a[o])
        Ga = np.zeros((len(a), E.shape[1]), np.float32) if Ga is None else Ga
        Ga[o] = E
    E = gen_encode(a, np.concatenate([b, np.arange(NTR, NTR + NTE)]))
    return Ga, E[:len(b)], E[len(b):]


ONEHOT = pd.get_dummies(full[CATS].astype(str), dtype=np.float32).values
BASE = np.hstack([full[FLAGS].values.astype(np.float32), ONEHOT])


# ---------------------------------------------------------------- driver (ours)
import sys
from scipy import sparse
from sklearn.preprocessing import OneHotEncoder

ARGS = dict(a.split("=", 1) for a in sys.argv[2:])
NAME = sys.argv[1]
TESEED, MSEED = int(ARGS.get("teseed", 0)), int(ARGS.get("mseed", 42))
LEAVES, LR, KIND = int(ARGS.get("leaves", 4)), float(ARGS.get("lr", 0.02)), ARGS.get("kind", "lgb")
GLMV, C, VIEW = ARGS.get("glm", "gen"), float(ARGS.get("C", 0.3)), ARGS.get("view", "joint")
CS, MCS = float(ARGS.get("cs", 0.5)), int(ARGS.get("mcs", 100))
OUTD = os.environ.get("GB_OUT", os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "cache", "gboost"))
os.makedirs(OUTD, exist_ok=True)
ALL = (np.arange(NTR), np.arange(0))
TEST = np.arange(NTR, NTR + NTE)
PSEUDO = float(ARGS.get("pseudo", 0))     # pseudo=a: unlabeled rows join the encodings with soft labels, weight a
GLOBAL_SMOOTH = 20.0


def te_u(a, b, U, pU, smooth=20.0, inner=5, seed=0, alpha=1.0):
    """te_fold with unlabeled rows U (soft labels pU, weight alpha) in every count: value rates from more rows."""
    ya = y[a].astype(np.float64)
    Ka, Kb, Kt, Ku = KMAT[a], KMAT[b], KMAT[NTR:], KMAT[U]
    Ea, Eb, Et = np.zeros(Ka.shape, np.float32), np.zeros(Kb.shape, np.float32), np.zeros(Kt.shape, np.float32)
    inner_f = list(StratifiedKFold(inner, shuffle=True, random_state=seed).split(np.zeros(len(a)), ya))
    for j in range(KMAT.shape[1]):
        su = np.bincount(Ku[:, j], weights=alpha * pU, minlength=NCAT[j])
        nu = np.bincount(Ku[:, j], minlength=NCAT[j]) * alpha
        for i, o in inner_f:
            s = np.bincount(Ka[i, j], weights=ya[i], minlength=NCAT[j]) + su
            n = np.bincount(Ka[i, j], minlength=NCAT[j]) + nu
            prior = (ya[i].sum() + su.sum()) / (len(i) + nu.sum())
            Ea[o, j] = ((s + prior * smooth) / (n + smooth))[Ka[o, j]]
        s = np.bincount(Ka[:, j], weights=ya, minlength=NCAT[j]) + su
        n = np.bincount(Ka[:, j], minlength=NCAT[j]) + nu
        prior = (ya.sum() + su.sum()) / (len(a) + nu.sum())
        # leave-one-out for the unlabeled rows: like a training row, a row never sees its own (soft) label
        own = alpha * pU
        kb, kt = Ku[:len(b), j], Ku[len(b):, j]
        Eb[:, j] = (s[kb] - own[:len(b)] + prior * smooth) / (n[kb] - alpha + smooth)
        Et[:, j] = (s[kt] - own[len(b):] + prior * smooth) / (n[kt] - alpha + smooth)
    return Ea, Eb, Et


NF = int(ARGS.get("nfold", 5))     # nfold=10: out-of-fold predictions from 90% of the rows, closer to the 100% refit
if NF != 5:
    FOLDS = list(StratifiedKFold(NF, shuffle=True, random_state=42).split(np.zeros(NTR), y))
KCNT = np.column_stack([np.bincount(KMAT[:, j])[KMAT[:, j]] for j in range(KMAT.shape[1])]).astype(np.float32)
CNT = np.column_stack([full[c].map(full[c].value_counts()).values for c in NUMS]).astype(np.float32)
OHK = None
if GLMV == "gen_ohk":
    # one-hot of every exact value and every digit/token/joint key (values seen < 5 times pooled), as in our
    # Part 4 joint GLM; scaled by s so one L2 penalty (C) acts on the dense logits and a weaker one on the one-hots
    OHK = OneHotEncoder(handle_unknown="ignore", min_frequency=5, dtype=np.float32).fit_transform(KMAT).tocsr()
    OHK = OHK * float(ARGS.get("ohks", 0.1))
log(f"{NAME}: teseed {TESEED} mseed {MSEED} leaves {LEAVES} lr {LR} kind {KIND} glm {GLMV} C {C} view {VIEW}")


def glm_fit(idx_a, Xa_dense, Xo_list, ya):
    """Fit the GLM on rows idx_a; return its logit on the fit rows and on each (idx, dense) in Xo_list."""
    if OHK is None:
        m = LogisticRegression(C=C, max_iter=200, tol=1e-7, solver="newton-cholesky").fit(Xa_dense, ya)
        return m.decision_function(Xa_dense), [m.decision_function(X) for _, X in Xo_list]
    Xa = sparse.hstack([sparse.csr_matrix(Xa_dense), OHK[idx_a]]).tocsr()
    m = LogisticRegression(C=C, max_iter=3000, tol=1e-6, solver="lbfgs").fit(Xa, ya)
    outs = [m.decision_function(sparse.hstack([sparse.csr_matrix(X), OHK[i]]).tocsr()) for i, X in Xo_list]
    return m.decision_function(Xa), outs


def boost_params():
    if KIND == "lgb":
        return dict(n_estimators=20000, learning_rate=LR, num_leaves=LEAVES, min_child_samples=MCS, subsample=0.8,
                    subsample_freq=1, colsample_bytree=CS, reg_lambda=1.0, max_bin=255, verbose=-1,
                    n_jobs=NJOBS, random_state=MSEED)
    return dict(n_estimators=20000, learning_rate=LR, max_depth=int(ARGS.get("depth", 2)), subsample=0.8,
                colsample_bytree=CS, min_child_weight=5, reg_lambda=1.0, max_bin=256, tree_method="hist",
                eval_metric="auc", n_jobs=NJOBS, random_state=MSEED,
                **({"device": ARGS["device"]} if "device" in ARGS else {}))


class MLPRes:
    """PyTorch MLP on the GPU whose output is added to the GLM's logit (a residual net). Inputs are
    signed-log transformed and standardized with the training rows' statistics; early stopping on epochs."""

    def __init__(self, seed):
        import torch
        self.torch, self.seed = torch, seed
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"

    def _prep(self, X, fit=False):
        X = np.sign(X) * np.log1p(np.abs(X.astype(np.float64)))
        if fit:
            self.mu, self.sd = X.mean(0), X.std(0) + 1e-6
        return self.torch.tensor(((X - self.mu) / self.sd).astype(np.float32), device=self.dev)

    def _net(self, d):
        nn, h = self.torch.nn, int(ARGS.get("hidden", 256))
        drop = float(ARGS.get("drop", 0.2))
        return nn.Sequential(nn.Linear(d, h), nn.SiLU(), nn.Dropout(drop), nn.Linear(h, h), nn.SiLU(),
                             nn.Dropout(drop), nn.Linear(h, 1)).to(self.dev)

    def margin(self, X, z):
        torch = self.torch
        self.net.eval()
        T, out = self._prep(X), []
        with torch.no_grad():
            for i in range(0, len(T), 65536):
                out.append(self.net(T[i:i + 65536]).squeeze(1).float().cpu().numpy())
        return z + np.concatenate(out)

    def fit(self, Xa, ya, za, epochs=None, Xb=None, yb=None, zb=None):
        torch = self.torch
        torch.manual_seed(self.seed)
        A = self._prep(Xa, fit=True)
        Y = torch.tensor(ya.astype(np.float32), device=self.dev)
        Z = torch.tensor(za.astype(np.float32), device=self.dev)
        self.net = self._net(A.shape[1])
        opt = torch.optim.AdamW(self.net.parameters(), lr=float(ARGS.get("nnlr", 1e-3)), weight_decay=1e-4)
        loss_fn, bs = torch.nn.BCEWithLogitsLoss(), int(ARGS.get("bs", 4096))
        g = torch.Generator(device=self.dev).manual_seed(self.seed)
        best, best_ep, best_state, bad = -1, 0, None, 0
        for ep in range(epochs or 40):
            self.net.train()
            perm = torch.randperm(len(A), device=self.dev, generator=g)
            for i in range(0, len(A), bs):
                j = perm[i:i + bs]
                opt.zero_grad()
                loss_fn(self.net(A[j]).squeeze(1) + Z[j], Y[j]).backward()
                opt.step()
            if epochs is None:
                auc = roc_auc_score(yb, self.margin(Xb, zb))
                if auc > best:
                    best, best_ep, bad = auc, ep + 1, 0
                    best_state = {k: v.clone() for k, v in self.net.state_dict().items()}
                else:
                    bad += 1
                    if bad >= 3:
                        break
        if epochs is None:
            self.net.load_state_dict(best_state)
            return best_ep
        return epochs


NOINIT = ARGS.get("init", "glm") == "none"     # init=none: start from 0 instead of the GLM's logit


def bfit(Xa, ya, za, n=None, Xb=None, yb=None, zb=None):
    if NOINIT:
        za = np.zeros_like(za)
        zb = None if zb is None else np.zeros_like(zb)
    if KIND == "mlp":
        m = MLPRes(MSEED)
        return m, m.fit(Xa, ya, za, epochs=n, Xb=Xb, yb=yb, zb=zb)
    if KIND == "cat":      # CatBoost from the GLM's logit (Pool baseline); symmetric trees of depth `depth`
        from catboost import CatBoostClassifier, Pool
        P = dict(iterations=n or 20000, learning_rate=LR, depth=int(ARGS.get("depth", 3)), l2_leaf_reg=3.0,
                 rsm=CS, random_seed=MSEED, thread_count=NJOBS, verbose=0, eval_metric="AUC",
                 border_count=254, bootstrap_type="Bernoulli", subsample=0.8)
        if n:
            return CatBoostClassifier(**P).fit(Pool(Xa, ya, baseline=za)), n
        m = CatBoostClassifier(**P, od_type="Iter", od_wait=300, use_best_model=True)
        m.fit(Pool(Xa, ya, baseline=za), eval_set=Pool(Xb, yb, baseline=zb))
        return m, m.get_best_iteration() + 1
    if KIND == "ebm":      # EBM from the GLM's logit: additive shapes + pairwise interactions on top of the GLM
        from interpret.glassbox import ExplainableBoostingClassifier
        m = ExplainableBoostingClassifier(interactions=int(ARGS.get("inter", 10)), outer_bags=int(ARGS.get("bags", 8)),
                                          learning_rate=LR, n_jobs=NJOBS, random_state=MSEED)
        return m.fit(Xa, ya, init_score=za), 1
    P = boost_params()
    if KIND == "lgb":
        if n:
            return lgb.LGBMClassifier(**{**P, "n_estimators": n}).fit(Xa, ya, init_score=za, categorical_feature=CAT_IDX), n
        m = lgb.LGBMClassifier(**P)
        m.fit(Xa, ya, init_score=za, eval_set=[(Xb, yb)], eval_init_score=[zb], eval_metric="auc",
              categorical_feature=CAT_IDX, callbacks=[lgb.early_stopping(300, verbose=False)])
        return m, m.best_iteration_
    if n:
        return xgb.XGBClassifier(**{**P, "n_estimators": n}).fit(Xa, ya, base_margin=za), n
    m = xgb.XGBClassifier(**P, early_stopping_rounds=300)
    m.fit(Xa, ya, base_margin=za, eval_set=[(Xb, yb)], base_margin_eval_set=[zb], verbose=False)
    return m, m.best_iteration + 1


def bmargin(m, X, z, it):
    if NOINIT:
        z = np.zeros_like(z)
    if KIND == "mlp":
        return m.margin(X, z)
    if KIND == "cat":
        from catboost import Pool
        return m.predict(Pool(X, baseline=z), prediction_type="RawFormulaVal")
    if KIND == "ebm":
        p = np.clip(m.predict_proba(X, init_score=z)[:, 1], 1e-12, 1 - 1e-12)
        return np.log(p / (1 - p))
    if KIND == "lgb":
        return z + m.predict(X, raw_score=True)
    return m.predict(X, base_margin=z, output_margin=True, iteration_range=(0, it))


def view(idx, E):
    return np.hstack([RAWV[idx], CNT[idx]]) if VIEW == "raw" else np.hstack([RAWV[idx], E, KCNT[idx]])


oof, foldavg, glm_oof, its = np.zeros(NTR), np.zeros(NTE), np.zeros(NTR), []
for f, (a, b) in list(enumerate(FOLDS)) + [("all", ALL)]:
    Ea, Eb, Et = te_fold(a, b, smooth=20.0, seed=TESEED)
    Ga_, Gb_, Gt_ = (Ea, Eb, Et)
    if GEN:
        Ga, Gb, Gt = gen_fold(a, b, seed=TESEED)
        Ga_, Gb_, Gt_ = np.hstack([Ea, Ga]), np.hstack([Eb, Gb]), np.hstack([Et, Gt])
    if PSEUDO > 0:
        # first pass: the GLM on the labeled rows only gives the soft labels of the unlabeled ones
        sc = StandardScaler().fit(logit(Ga_))
        D0 = lambda E, i: np.hstack([sc.transform(logit(E)), BASE[i]])
        U = np.concatenate([b, TEST])
        _, (zu,) = glm_fit(a, D0(Ga_, a), [(U, D0(np.vstack([Gb_, Gt_]), U))], y[a])
        Ea, Eb, Et = te_u(a, b, U, 1 / (1 + np.exp(-zu)), smooth=20.0, seed=TESEED, alpha=PSEUDO)
        Ga_, Gb_, Gt_ = (np.hstack([Ea, Ga]), np.hstack([Eb, Gb]), np.hstack([Et, Gt])) if GEN else (Ea, Eb, Et)
    sc = StandardScaler().fit(logit(Ga_))
    D = lambda E, i: np.hstack([sc.transform(logit(E)), BASE[i]])
    if f == "all":
        za, (zt,) = glm_fit(a, D(Ga_, a), [(TEST, D(Gt_, TEST))], y[a])
        n = int(round(np.mean(its) * NF / (NF - 1)))
        m, _ = bfit(view(a, Ea), y, za, n=n)
        refit = bmargin(m, view(TEST, Et), zt, n)
        np.save(os.path.join(OUTD, f"glmrefit_{NAME}.npy"), zt.astype(np.float32))
        log(f"refit on all rows with {n} trees")
        break
    za, (zb, zt) = glm_fit(a, D(Ga_, a), [(b, D(Gb_, b)), (TEST, D(Gt_, TEST))], y[a])
    glm_oof[b] = zb
    m, it = bfit(view(a, Ea), y[a], za, Xb=view(b, Eb), yb=y[b], zb=zb)
    its.append(it)
    oof[b] = bmargin(m, view(b, Eb), zb, it)
    foldavg += bmargin(m, view(TEST, Et), zt, it) / len(FOLDS)
    log(f"{NAME} fold {f}: GLM {roc_auc_score(y[b], zb):.5f} -> boosted {roc_auc_score(y[b], oof[b]):.5f} ({it} trees)")

for k, v in (("oof", oof), ("foldavg", foldavg), ("refit", refit), ("glmoof", glm_oof)):
    np.save(os.path.join(OUTD, f"{k}_{NAME}.npy"), v.astype(np.float32))
np.save(os.path.join(OUTD, f"test_{NAME}.npy"), ((rankdata(foldavg) + rankdata(refit)) / (2 * NTE)).astype(np.float32))
fa = [roc_auc_score(y[b], oof[b]) for _, b in FOLDS]
log(f"==> {NAME}: OOF {roc_auc_score(y, oof):.6f} (GLM {roc_auc_score(y, glm_oof):.6f}) | folds " +
    " ".join(f"{x:.5f}" for x in fa))
with open(os.path.join(OUTD, "results.csv"), "a") as fh:
    fh.write(f"{time.strftime('%m-%d %H:%M')},{NAME},{roc_auc_score(y, oof):.6f},{roc_auc_score(y, glm_oof):.6f},"
             f"{' '.join(sys.argv[2:])}\n")

In [ ]:
import subprocess
GB_OUT = "gboost_out"
os.makedirs(GB_OUT, exist_ok=True)
env = {**os.environ, "INPUT_DIR": INPUT, "GB_OUT": GB_OUT}
r = subprocess.run([sys.executable, "gboost.py", "gb_s0", "teseed=0", "mseed=42"], env=env, capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if "fold" in l or "==>" in l or "refit" in l) or r.stderr[-3000:])
mine = np.load(os.path.join(GB_OUT, "oof_gb_s0.npy"))
saved = np.load(find("gb_s0_oof.npy", ART))
print(f"gb_s0 OOF AUC {roc_auc_score(y, mine):.6f} | largest difference to our dataset {np.abs(mine - saved).max():.1e}")

## 6.3 A bag of five seeds, and what the public board says about it

Five encoding and model seeds (5-fold CV):

| encoding seed / model seed | GLM | GLM → trees | gain (× 1e-4) |
|---|---|---|---|
| 0 / 42 | 0.95047 | 0.95137 | +9.0 |
| 1 / 137 | 0.95053 | 0.95139 | +8.6 |
| 2 / 2026 | 0.95049 | 0.95139 | +9.0 |
| 3 / 7 | 0.95050 | 0.95136 | +8.5 |
| 4 / 99 | 0.95055 | 0.95148 | +9.3 |
| **rank mean of the five** | | **0.95151** | |

The seed spread (0.95136-0.95148) is as large as most of the effects we test, the same lesson as Part 5's
"lucky seed 42". The bag is +0.00011 over the seed mean; three seeds give 0.95148. An XGBoost
(depth 2) version scores 0.95131-0.95135 per seed and adds nothing next to the
LightGBM bag (rank correlation 0.998); the raw-view booster scores 0.95100-0.95108.

Then the public board. Five submissions after Part 5, changing one thing at a time:

| | submission | CV | public LB (Part 5: 0.95160) |
|---|---|---|---|
| A | GLM boost, 5-seed bag, refit test | 0.951506 | 0.95154 |
| B | 0.3 × Part 5 blend + 0.7 × GLM boost | 0.951558 | 0.95177 |
| C | logistic stack: Part 5 members + GLM boost + 55 public members | 0.951549 (nested) | **0.95210** |
| D | A with the rank mean of fold average and refit as the test prediction | 0.951506 | 0.95146 |
| E | C with 15 own members and the XGBoost booster, C = 0.01 | 0.951573 (nested) | 0.95207 |
| Part 6 | C's 62 members, stack C = 0.01 solved to convergence (this notebook) | 0.951629 (nested) | **0.95215** |

* **A**: the best single model we have, alone, scores *below* Part 5 on the public board despite +0.0006 CV.
  That is the third GLM-centred submission to do so (Part 5's plain GLM: −0.00018 and −0.00028 against
  expectation). We still do not know whether the CV or the 20% public sample is right about GLMs; the private
  board will tell. Mixed with other families (B, C) the score follows the CV again.
* **D vs A**: the refit beat the rank mean of fold average and refit once more (+0.00008). That is four out of
  four on the public board, against the held-out check of Part 5 (5.4); every test prediction here is a refit.

The bag is loaded from our dataset; `RETRAIN_P6 = True` trains all five seeds with the script above.

In [ ]:
RETRAIN_P6 = False
GB_SEEDS = [(0, 42), (1, 137), (2, 2026), (3, 7), (4, 99)]
gb_oof, gb_full = [], []
for s, m in GB_SEEDS:
    if RETRAIN_P6 and s > 0:
        subprocess.run([sys.executable, "gboost.py", f"gb_s{s}", f"teseed={s}", f"mseed={m}"], env=env, check=True,
                       capture_output=True)
    if RETRAIN_P6:
        o, f = np.load(os.path.join(GB_OUT, f"oof_gb_s{s}.npy")), np.load(os.path.join(GB_OUT, f"refit_gb_s{s}.npy"))
    else:
        o, f = np.load(find(f"gb_s{s}_oof.npy", ART)), np.load(find(f"gb_s{s}_full.npy", ART))
    gb_oof.append(rankdata(o) / len(o))
    gb_full.append(rankdata(f) / len(f))
    print(f"seed {s}: OOF AUC {roc_auc_score(y, o):.6f}")
OOF["gb_bag5"], FULL["gb_bag5"] = np.mean(gb_oof, 0), np.mean(gb_full, 0)
print(f"gb_bag5: OOF AUC {roc_auc_score(y, OOF['gb_bag5']):.6f}")

## 6.4 Pool the published members, stack, and solve the stacker to convergence

Seven sources publish out-of-fold and test predictions on our split. We take every member whose OOF AUC is
within 0.004 of the best (62 of 73) and drop exact copies (rank correlation ≥ 0.99995, e.g. two notebooks that
published the same model). The 55 public members:

| author | members | best OOF AUC |
|---|---|---|
| megayak | 7 | 0.95137 |
| Nilanjan_Saha123 | 5 | 0.95076 |
| lazyyeou | 5 | 0.95076 |
| alex chilton | 26 | 0.95066 |
| Malyshev Danil | 10 | 0.95051 |
| Vladimir Demidov | 2 | 0.94909 |

plus our seven: the GLM boost bag and the six members of the Part 5 blend. Test predictions are each source's
full-data refit where it publishes one, its fold average otherwise.

**Combiners, scored by nested CV** (the combiner is fitted on four folds' out-of-fold rows and scored on the
fifth, so its own fitting cannot flatter it):

| members and combiner | nested CV AUC |
|---|---|
| our 10 members, hill climbing | 0.950956 |
| our 10 members, logistic stack | 0.950965 |
| + public banks, hill climbing | 0.951401 |
| + public banks, logistic stack | 0.951511 |
| Part 5 members + GLM boost + banks, hill | 0.951433 |
| Part 5 members + GLM boost + banks, stack (C) | 0.951549 |
| 15 own members + XGB boost + banks, stack C 0.01 (E) | 0.951573 |
| … only members within 0.002 of the best | 0.951492 |
| Part 6: C's 62 members, stack C 0.01 solved to convergence | 0.951629 |

Rank hill climbing picks the GLM boost alone (adding any other member does not raise the OOF AUC), so it
cannot use the others. A logistic regression on the members' rank-logits can: it gives the GLM boosters the
largest weights and **negative** weights to a few members (a RealMLP, a tuned XGBoost) whose errors it uses
to correct the rest. Dropping the weaker members costs CV: they carry the variety.

**Solve it to convergence.** Our first stacks used scikit-learn's default solver tolerance, which stops
L-BFGS early; where it stops depends on the machine and the library version, so a notebook re-run could not
reproduce the file we scored. Solved to convergence with `C = 1` the coefficients grow to ±1.2 (the members
are nearly collinear); with `C = 0.01` they stay below 0.4, two different solvers agree to a rank correlation of
0.9999999999, the nested CV is higher (0.951629), and the public board agrees: 0.95215 (early-stopped:
0.95210). That is the stack below, written to `submission_part6.csv`. One RealMLP member was republished
after this part (OOF 0.94909 → 0.94970), so a re-run here gives a slightly different file from the one scored.

The members are listed with the OOF AUC we recorded; if a source publishes a new version, the cell reports
which members changed.

In [ ]:
BANK = {'ours': ['lgb_diglead_l3_bag', 'glm_ohk2_bag', 'ebm_i25_bag4', 'lgb_dig_s100_bag5', 'lgb_rawdig_l4', 'lgb_dig_l7', 'gb_bag5'], 'ext': ['mega:glm_boost', 'mega:glm_boost_xgb', 'mega:glm_boost_raw', 'nil:lgb_unified_96key_bag', 'lazy:xgb_d2_bag', 'nil:xgb_unified_96key_bag', 'lazy:xgb_l4_bag', 'lazy:lgb_s100_bag', 'nil:lgb_unified_s100_bag', 'mega:lgb_te20', 'lazy:lgb_l4_bag', 'alex:LightGBM_3_leaves,_Part_5_recipe,_5_encoder_seeds', 'mega:xgb_te20', 'alex:XGBoost_depth_2,_Part_3_recipe,_5_encoder_seeds,_full_refit', 'alex:LightGBM_3_leaves,_Part_5_recipe', 'maly:full_lgb_te20', 'mega:glm_gen', 'alex:XGBoost_depth_2_boosted_from_the_joint_GLM,_full_refit', 'maly:full_xgb_te20', 'alex:LightGBM_4_leaves,_Part_3_recipe', 'alex:XGBoost_depth_2,_Part_3_recipe', 'nil:lgb_unified_l7', 'maly:joint_token', 'alex:Joint_GLM,_full_refit', 'nil:glm_unified_96key', 'alex:LightGBM_4_leaves_digits_s100', 'alex:LightGBM_4_leaves_digits', 'alex:XGBoost_depth_2_digits', 'alex:LightGBM_4_leaves_digits,_no_pairs', 'alex:XGBoost_depth_2_digits,_no_pairs', 'alex:GLM_on_the_encodings', 'yek:ps-s6-e11-realmlp-pytorc', 'maly:lgb_enriched_s100', 'maly:xgb_enriched_d1_s100', 'yek:ps-s6-e11-realmlp-pytabk', 'maly:lgb_te_s20_cnt', 'maly:xgb_te_s20', 'maly:lgb_te_s100', 'alex:LightGBM_4_leaves_plus', 'mega:lgb_raw', 'alex:XGBoost_depth_2_bag3', 'maly:ebm_private_int5', 'alex:LightGBM_4_leaves_b', 'alex:XGBoost_depth_1_bag3', 'alex:LightGBM_tuned', 'alex:LightGBM_4_leaves', 'alex:XGBoost_depth_2', 'alex:XGBoost_depth_2_plus', 'alex:XGBoost_tuned', 'alex:XGBoost_depth_1', 'maly:ebm_raw', 'lazy:lgb_rawview', 'alex:RealMLP_digits', 'alex:XGBoost_depth_2_raw', 'alex:LightGBM_4_leaves_raw'], 'auc': {'lgb_diglead_l3_bag': 0.950627, 'glm_ohk2_bag': 0.950115, 'ebm_i25_bag4': 0.950262, 'lgb_dig_s100_bag5': 0.950488, 'lgb_rawdig_l4': 0.948375, 'lgb_dig_l7': 0.950155, 'gb_bag5': 0.951506, 'mega:glm_boost': 0.951375, 'mega:glm_boost_xgb': 0.951323, 'mega:glm_boost_raw': 0.951003, 'nil:lgb_unified_96key_bag': 0.950757, 'lazy:xgb_d2_bag': 0.950756, 'nil:xgb_unified_96key_bag': 0.950742, 'lazy:xgb_l4_bag': 0.950724, 'lazy:lgb_s100_bag': 0.95072, 'nil:lgb_unified_s100_bag': 0.950697, 'mega:lgb_te20': 0.950695, 'lazy:lgb_l4_bag': 0.950686, 'alex:LightGBM_3_leaves,_Part_5_recipe,_5_encoder_seeds': 0.950656, 'mega:xgb_te20': 0.950643, 'alex:XGBoost_depth_2,_Part_3_recipe,_5_encoder_seeds,_full_refit': 0.950542, 'alex:LightGBM_3_leaves,_Part_5_recipe': 0.950516, 'maly:full_lgb_te20': 0.950513, 'mega:glm_gen': 0.950472, 'alex:XGBoost_depth_2_boosted_from_the_joint_GLM,_full_refit': 0.95042, 'maly:full_xgb_te20': 0.950404, 'alex:LightGBM_4_leaves,_Part_3_recipe': 0.950394, 'alex:XGBoost_depth_2,_Part_3_recipe': 0.950357, 'nil:lgb_unified_l7': 0.950251, 'maly:joint_token': 0.950145, 'alex:Joint_GLM,_full_refit': 0.950106, 'nil:glm_unified_96key': 0.950096, 'alex:LightGBM_4_leaves_digits_s100': 0.949511, 'alex:LightGBM_4_leaves_digits': 0.94951, 'alex:XGBoost_depth_2_digits': 0.94951, 'alex:LightGBM_4_leaves_digits,_no_pairs': 0.94936, 'alex:XGBoost_depth_2_digits,_no_pairs': 0.949336, 'alex:GLM_on_the_encodings': 0.949159, 'yek:ps-s6-e11-realmlp-pytorc': 0.949703, 'maly:lgb_enriched_s100': 0.949054, 'maly:xgb_enriched_d1_s100': 0.948993, 'yek:ps-s6-e11-realmlp-pytabk': 0.948838, 'maly:lgb_te_s20_cnt': 0.948769, 'maly:xgb_te_s20': 0.948734, 'maly:lgb_te_s100': 0.948728, 'alex:LightGBM_4_leaves_plus': 0.948682, 'mega:lgb_raw': 0.948573, 'alex:XGBoost_depth_2_bag3': 0.948547, 'maly:ebm_private_int5': 0.948483, 'alex:LightGBM_4_leaves_b': 0.948447, 'alex:XGBoost_depth_1_bag3': 0.948441, 'alex:LightGBM_tuned': 0.948394, 'alex:LightGBM_4_leaves': 0.948378, 'alex:XGBoost_depth_2': 0.948371, 'alex:XGBoost_depth_2_plus': 0.948358, 'alex:XGBoost_tuned': 0.948318, 'alex:XGBoost_depth_1': 0.948306, 'maly:ebm_raw': 0.948128, 'lazy:lgb_rawview': 0.948113, 'alex:RealMLP_digits': 0.947972, 'alex:XGBoost_depth_2_raw': 0.947671, 'alex:LightGBM_4_leaves_raw': 0.947634}, 'refit': {'mega:glm_boost': True, 'mega:glm_boost_xgb': True, 'mega:glm_boost_raw': True, 'nil:lgb_unified_96key_bag': True, 'lazy:xgb_d2_bag': True, 'nil:xgb_unified_96key_bag': True, 'lazy:xgb_l4_bag': True, 'lazy:lgb_s100_bag': True, 'nil:lgb_unified_s100_bag': True, 'mega:lgb_te20': True, 'lazy:lgb_l4_bag': True, 'alex:LightGBM_3_leaves,_Part_5_recipe,_5_encoder_seeds': False, 'mega:xgb_te20': True, 'alex:XGBoost_depth_2,_Part_3_recipe,_5_encoder_seeds,_full_refit': False, 'alex:LightGBM_3_leaves,_Part_5_recipe': False, 'maly:full_lgb_te20': True, 'mega:glm_gen': True, 'alex:XGBoost_depth_2_boosted_from_the_joint_GLM,_full_refit': False, 'maly:full_xgb_te20': True, 'alex:LightGBM_4_leaves,_Part_3_recipe': False, 'alex:XGBoost_depth_2,_Part_3_recipe': False, 'nil:lgb_unified_l7': True, 'maly:joint_token': False, 'alex:Joint_GLM,_full_refit': False, 'nil:glm_unified_96key': True, 'alex:LightGBM_4_leaves_digits_s100': False, 'alex:LightGBM_4_leaves_digits': False, 'alex:XGBoost_depth_2_digits': False, 'alex:LightGBM_4_leaves_digits,_no_pairs': False, 'alex:XGBoost_depth_2_digits,_no_pairs': False, 'alex:GLM_on_the_encodings': False, 'yek:ps-s6-e11-realmlp-pytorc': False, 'maly:lgb_enriched_s100': False, 'maly:xgb_enriched_d1_s100': False, 'yek:ps-s6-e11-realmlp-pytabk': False, 'maly:lgb_te_s20_cnt': False, 'maly:xgb_te_s20': False, 'maly:lgb_te_s100': False, 'alex:LightGBM_4_leaves_plus': False, 'mega:lgb_raw': True, 'alex:XGBoost_depth_2_bag3': False, 'maly:ebm_private_int5': False, 'alex:LightGBM_4_leaves_b': False, 'alex:XGBoost_depth_1_bag3': False, 'alex:LightGBM_tuned': False, 'alex:LightGBM_4_leaves': False, 'alex:XGBoost_depth_2': False, 'alex:XGBoost_depth_2_plus': False, 'alex:XGBoost_tuned': False, 'alex:XGBoost_depth_1': False, 'maly:ebm_raw': False, 'lazy:lgb_rawview': True, 'alex:RealMLP_digits': False, 'alex:XGBoost_depth_2_raw': False, 'alex:LightGBM_4_leaves_raw': False}}

import json as _json

P6_SOURCES = {   # member prefix -> the attached public input (kernel output or dataset) it is read from
    "lazy": "s6e11-digit-trie-te-joint-glm-nested-stack", "nil": "96-key-digit-trie-gpt-2-full-refit-hybrid",
    "mega": "s6e11-per-value-digit-te-shallow-tree-stack", "alex": "credit-card-fraud-lb-0-95",
    "yek": "ps-s6-e11-realmlp-py", "maly": "s6e11-pooled-stack-v9"}


def src_dir(prefix, sub=""):
    """Directory of an attached input, found by its slug (Kaggle mounts kernels and datasets under varying roots)."""
    slug = P6_SOURCES[prefix] + sub
    hits = sorted(d for d in glob.glob(os.path.join(INPUT, "**", slug + "*"), recursive=True) if os.path.isdir(d))
    if not hits:
        raise FileNotFoundError(f"attach the input '{slug}' (see the Part 6 setup)")
    return hits[0]


def load_member(name):
    """(oof, test) of one member. Test = the full-data refit when the source publishes one, else the fold average.
    Values go through float32 so every input is read at the same precision as in our submitted file."""
    p, b = name.split(":", 1) if ":" in name else ("ours", name)
    if p == "ours":
        o, t = np.load(find(f"{b}_oof.npy", ART)), np.load(find(f"{b}_full.npy", ART))
    elif p in ("lazy", "nil"):
        d = os.path.join(src_dir(p), "preds")
        o, t = np.load(os.path.join(d, f"{b}_oof.npy")), np.load(os.path.join(d, f"{b}_full.npy"))
    elif p == "mega":
        d = src_dir(p)
        o, t = np.load(os.path.join(d, f"oof_{b}.npy")), np.load(os.path.join(d, f"refit_{b}.npy"))
    elif p == "alex":
        d = src_dir(p)
        man = _json.load(open(os.path.join(d, "oof_manifest.json")))
        h = next(k for k, v in man.items() if v["name"].replace(" ", "_") == b)
        o, t = np.load(os.path.join(d, f"oof_{h}.npy")), np.load(os.path.join(d, f"test_{h}.npy"))
    elif p == "yek":
        d = src_dir(p, b[len("ps-s6-e11-realmlp-py"):])
        o = pd.read_csv(os.path.join(d, "oof_preds.csv")).iloc[:, -1].values
        t = pd.read_csv(os.path.join(d, "submission.csv")).iloc[:, -1].values
    else:  # maly: one column of the published 32-model bank
        d = src_dir(p)
        z = np.load(os.path.join(d, "pool_bank.npz"), allow_pickle=False)
        j = _json.load(open(os.path.join(d, "pool_manifest.json")))["names"].index("own_" + b)
        o, t = z["oof"][:, j], z["test"][:, j]
    o, t = np.asarray(o, np.float32).ravel().astype(float), np.asarray(t, np.float32).ravel().astype(float)
    assert len(o) == len(y) and len(t) == len(te) and np.isfinite(o).all() and np.isfinite(t).all(), name
    return o, t


def rlogit(r):
    r = np.clip(r, 1e-6, 1 - 1e-6)
    return np.log(r / (1 - r))


P6_C = 0.01      # penalty of the stacker (inverse strength)


def stacker():
    # a convex problem solved to convergence with Newton steps: the same solution on any machine and any
    # scikit-learn version (the default lbfgs tolerance stops early, and where it stops depends on the platform)
    return LogisticRegression(C=P6_C, tol=1e-10, max_iter=2000, solver="newton-cholesky")


P6_NAMES = BANK["ours"] + BANK["ext"]
P6_O, P6_T, rows, gone = [], [], [], []
for n in list(P6_NAMES):
    try:
        o, t = load_member(n)
    except (FileNotFoundError, KeyError, StopIteration, ValueError):      # a source dropped this member in a later version
        gone.append(n); P6_NAMES.remove(n); continue
    P6_O.append(rankdata(o) / len(o))
    P6_T.append(rankdata(t) / len(t))
    a = roc_auc_score(y, o)
    rows.append((n, a, BANK["auc"][n], a - BANK["auc"][n]))
P6_O, P6_T = np.column_stack(P6_O), np.column_stack(P6_T)
check = pd.DataFrame(rows, columns=["member", "OOF AUC", "expected", "difference"]).set_index("member")
changed = check[check["difference"].abs() > 2e-6]
print(f"{len(P6_NAMES)} members loaded, {len(gone)} no longer published {gone}; {len(changed)} differ from the AUCs we recorded"
      + ("" if changed.empty else " (a source published a new version):"))
if not changed.empty:
    display(changed)

In [ ]:
# nested CV: the stacker is fitted on four folds' out-of-fold rows and scored on the fifth
nested = np.zeros(len(y))
for a, b in FOLDS:
    nested[b] = rankdata(stacker().fit(rlogit(P6_O[a]), y[a]).decision_function(rlogit(P6_O[b]))) / len(b)
fold_auc = [roc_auc_score(y[b], nested[b]) for _, b in FOLDS]
print(f"stack of {P6_O.shape[1]}: nested CV AUC {roc_auc_score(y, nested):.6f} | folds " +
      " ".join(f"{v:.5f}" for v in fold_auc))
print(f"best single member: {max(BANK['auc'].values()):.6f} ({max(BANK['auc'], key=BANK['auc'].get)})")

# final stacker on every out-of-fold row, applied to the members' test predictions
ST = stacker().fit(rlogit(P6_O), y)
coef = pd.Series(ST.coef_[0], index=P6_NAMES)
display(coef.reindex(coef.abs().sort_values(ascending=False).index).head(12).round(3).to_frame("stack coefficient"))
P6_TEST = rankdata(ST.decision_function(rlogit(P6_T))) / len(te)
sub = pd.DataFrame({"id": te["id"], TARGET: P6_TEST})
sub.to_csv("submission_part6.csv", index=False)      # Part 7 writes submission.csv
print("submission_part6.csv written:", sub.shape)

## 6.5 What did not help in Part 6

| idea | result | note |
|---|---|---|
| 3-leaf trees on top of the GLM | 0.95132 vs 0.95137 (same seed) | megayak measured the same; 4 leaves stay |
| one-hot of every value and key in the starting GLM (Part 4 style, × 0.1) | 0.95119 | a stronger GLM, a weaker booster; zero gain in the bag |
| XGBoost depth 3 / learning rate 0.01 on top of the GLM | 0.95131 / 0.95132 | depth 2 at 0.02: 0.95133 |
| XGBoost booster bag next to the LightGBM bag | 0.951462 (5 seeds of both) vs 0.951476 (3 LightGBM) | the two boosters agree too much (ρ 0.998) |
| hill climbing over the pooled bank | picks the GLM boost alone; nested 0.951433 | only a stack uses the other members |
| keeping only members within 0.002 of the best | nested 0.951492 vs 0.951560 | the weak members carry the diversity |
| public score as the selector (0.9519-0.9521 public blends) | no CV | not used: see 6.1 |

**Planned after Part 6:** more variety for the stack, and the same predictions published for anyone who wants
to pool them.

---
# Part 7 — A language model reads the rows

*Public LB 0.95222 · nested CV 0.95206*

Part 6's stack was close to saturated. This part measures that, finds the one kind of model that still adds
something, and shows why the public board cannot confirm it.

In [ ]:
draw_part7(
    stats=[('Public LB', '0.95222', '+0.00007 over Part 6'), ('Nested CV of the stack', '0.95206', "+0.00042 over Part 6's members"), ('GPT-2 vs our best model', 'ρ 0.960', 'members this different score at most 0.9486'), ('Members', '65', '65 predictions, 8 authors, one split')],
    gains=[('LightGBM from the GLM, raw view', 0.01), ('CatBoost from the GLM', 0.04), ('EBM from the GLM', 0.05), ('XGBoost from the GLM, 5 seeds', 0.12), ('Residual MLPs, 2 bags of 5 seeds', 0.44), ('GPT-2 target-first classifier', 4.08)],
    points=[('lgb_diglead_l3_bag', 0.950627, 0.98775), ('glm_ohk2_bag', 0.950115, 0.98814), ('ebm_i25_bag4', 0.950262, 0.98486), ('lgb_dig_s100_bag5', 0.950488, 0.98491), ('lgb_rawdig_l4', 0.948375, 0.96298), ('lgb_dig_l7', 0.950155, 0.97949), ('mega:glm_boost', 0.951375, 0.99888), ('mega:glm_boost_xgb', 0.951323, 0.99878), ('mega:glm_boost_raw', 0.951003, 0.99681), ('nil:lgb_unified_96key_bag', 0.950757, 0.9875), ('lazy:xgb_d2_bag', 0.950756, 0.98878), ('nil:xgb_unified_96key_bag', 0.950742, 0.98873), ('lazy:xgb_l4_bag', 0.950724, 0.98761), ('lazy:lgb_s100_bag', 0.95072, 0.98542), ('nil:lgb_unified_s100_bag', 0.950697, 0.98553), ('mega:lgb_te20', 0.950695, 0.98591), ('lazy:lgb_l4_bag', 0.950686, 0.98682), ('alex:LightGBM_3_leaves,_Part_5_recipe,_5_encoder_seeds', 0.950656, 0.98808), ('mega:xgb_te20', 0.950643, 0.98664), ('alex:XGBoost_depth_2,_Part_3_recipe,_5_encoder_seeds,_full_refit', 0.950542, 0.98757), ('alex:LightGBM_3_leaves,_Part_5_recipe', 0.950516, 0.98655), ('maly:full_lgb_te20', 0.950513, 0.98586), ('mega:glm_gen', 0.950472, 0.99513), ('alex:XGBoost_depth_2_boosted_from_the_joint_GLM,_full_refit', 0.95042, 0.99007), ('maly:full_xgb_te20', 0.950404, 0.98638), ('alex:LightGBM_4_leaves,_Part_3_recipe', 0.950394, 0.9839), ('alex:XGBoost_depth_2,_Part_3_recipe', 0.950357, 0.98551), ('nil:lgb_unified_l7', 0.950251, 0.98032), ('maly:joint_token', 0.950145, 0.98932), ('alex:Joint_GLM,_full_refit', 0.950106, 0.98878), ('nil:glm_unified_96key', 0.950096, 0.98887), ('alex:LightGBM_4_leaves_digits_s100', 0.949511, 0.97937), ('alex:LightGBM_4_leaves_digits', 0.94951, 0.97917), ('alex:XGBoost_depth_2_digits', 0.94951, 0.98007), ('alex:LightGBM_4_leaves_digits,_no_pairs', 0.94936, 0.97833), ('alex:XGBoost_depth_2_digits,_no_pairs', 0.949336, 0.97868), ('alex:GLM_on_the_encodings', 0.949159, 0.98221), ('yek:ps-s6-e11-realmlp-pytorc', 0.949703, 0.97661), ('maly:lgb_enriched_s100', 0.949054, 0.97363), ('maly:xgb_enriched_d1_s100', 0.948993, 0.97716), ('yek:ps-s6-e11-realmlp-pytabk', 0.948838, 0.97174), ('maly:lgb_te_s20_cnt', 0.948769, 0.97274), ('maly:xgb_te_s20', 0.948734, 0.97403), ('maly:lgb_te_s100', 0.948728, 0.97266), ('alex:LightGBM_4_leaves_plus', 0.948682, 0.97186), ('mega:lgb_raw', 0.948573, 0.9644), ('alex:XGBoost_depth_2_bag3', 0.948547, 0.97209), ('maly:ebm_private_int5', 0.948483, 0.96485), ('alex:LightGBM_4_leaves_b', 0.948447, 0.97178), ('alex:XGBoost_depth_1_bag3', 0.948441, 0.97427), ('alex:LightGBM_tuned', 0.948394, 0.97452), ('alex:LightGBM_4_leaves', 0.948378, 0.97136), ('alex:XGBoost_depth_2', 0.948371, 0.97122), ('alex:XGBoost_depth_2_plus', 0.948358, 0.97189), ('alex:XGBoost_tuned', 0.948318, 0.9751), ('alex:XGBoost_depth_1', 0.948306, 0.97372), ('maly:ebm_raw', 0.948128, 0.96575), ('lazy:lgb_rawview', 0.948113, 0.96155), ('alex:RealMLP_digits', 0.947972, 0.94335), ('alex:XGBoost_depth_2_raw', 0.947671, 0.96036), ('alex:LightGBM_4_leaves_raw', 0.947634, 0.95904), ('gbn_bag5_w128', 0.95122, 0.99567), ('gbn_bag5_w512', 0.951223, 0.99618), ('gpt2', 0.950027, 0.95983)],
    folds0=[0.9506900806119718, 0.9521489163337, 0.9561004187946563, 0.9523521695206774, 0.9468679067313026],
    folds1=[0.9511082531665748, 0.9524550296653207, 0.9567464080947901, 0.9523412539911906, 0.9476321305113776],
    noise=[1.288, -5.834, 0.387, 2.677, 6.958, -1.179, 5.756, 0.562, 1.652, 4.88, 7.058, 9.948, 3.841, 2.47, 5.18, 4.832, 3.128, 4.397, 4.699, 6.782, -2.384, 2.741, 5.32, 2.98, 3.629, 3.637, 6.536, 6.31, -0.212, 5.522, 0.144, 3.007, 8.87, 5.675, 3.151, 2.16, 1.668, -3.277, 3.946, 3.24, 1.945, 12.098, 8.175, -3.938, 7.789, 4.055, 6.615, 9.984, 3.842, 0.654, 0.713, 6.906, 0.778, 1.75, -2.998, 5.89, 4.438, -2.14, 3.039, 0.107, 3.124, 4.096, 9.15, 8.225, -1.042, 1.339, 5.403, 4.487, 7.139, 10.913, 1.957, 0.708, -1.233, 3.428, 4.133, 4.587, 4.793, 4.113, 2.465, 2.787, 12.688, -4.242, 4.378, 4.125, 4.34, 4.525, -2.098, 0.178, 8.885, 3.172, -3.522, 8.349, 3.834, 8.203, 5.931, 7.08, 5.099, -1.876, 1.126, 8.734, 2.884, 4.174, 2.991, 5.091, 6.204, 0.062, 6.057, 2.186, 6.643, 4.878, 3.092, 5.609, 5.267, 10.838, 7.635, 9.817, -1.61, 12.47, 4.246, 3.926, 1.546, 7.137, 5.882, 5.558, 2.326, 3.433, 0.832, 8.71, 3.682, 3.482, 1.891, 2.433, 8.129, 0.933, 1.411, 1.5, 0.685, 4.612, 0.199, 8.537, 0.699, 13.964, 10.251, 8.914, -1.765, 10.052, 6.803, 11.522, 2.979, 5.213, 0.527, 5.495, 5.064, 7.412, -0.495, 2.892, 11.011, 4.568, -0.722, 5.211, 0.229, -2.555, 4.269, 5.779, 1.93, 1.179, 8.359, -3.271, 7.503, -4.614, 4.373, 5.007, 4.868, 4.82, 1.387, 3.588, 7.839, 3.897, 5.508, 4.749, 7.451, 2.475, 1.391, 3.333, 5.499, 5.969, 2.973, 2.842, 5.262, 5.634, 7.405, 7.908, 4.385, 6.195, 4.217, 6.804, 7.36, 6.596, 2.581, 2.202, 5.724, 0.199, -2.474, 5.622, 5.152, 1.938, 4.622, 5.934, 4.316, 7.701, 4.856, -0.219, 11.312, 6.572, 7.258, 0.615, 8.29, 12.563, 3.547, 6.485, -1.167, 2.728, -0.693, 3.185, 4.334, -1.351, 5.13, 7.811, 2.277, 3.796, 3.367, 0.593, -0.545, 7.302, 6.092, 6.381, 4.825, -2.257, 6.729, 3.153, 8.287, 7.003, 4.063, 2.919, 10.408, 3.645, 5.711, -3.806, -1.403, 1.401, 2.685, 2.003, 4.258, 4.856, 2.852, 6.231, 6.092, 6.382, 7.102, 4.553, -2.442, 2.89, -0.35, 0.926, 1.093, 4.963, 4.009, 5.004, -0.534, 0.871, 2.427, 6.142, 7.007, 4.455, 11.869, 3.934, 2.952, 3.996, 3.269, 3.32, 4.967, 2.0, -1.317, 3.285, -1.294, 7.01, 3.895, -0.247, 7.858, 2.959, 5.93, -1.587, 4.111, 0.998, 9.879, 6.916, 4.775, 8.551, 0.877, 7.176, 8.221, 2.891, -0.161, 4.513, 6.191, 6.647, 2.327, 2.031, 3.739, 8.772, 3.442, 7.892, 4.788, 4.201, 7.605, -2.906, 4.638, 5.843, 8.214, 2.175, 9.73, 6.751, 1.748, 4.584, 3.701, 10.288, 5.941, 2.904, 10.663, 1.112, -2.225, 3.739, -0.306, 3.515, 3.671, -0.859, 1.846, 2.05, 5.125, -0.576, 7.559, 7.691, 5.078, 5.219, 8.403, 4.235, 3.678, 1.787, 4.348, 2.017, -3.838, 2.393, 0.544, 5.163, -0.852, 3.182, 14.328, 4.86, 0.65, 0.931, 1.0, 2.554, 2.512, 13.459, 0.74, -0.71, -2.699, 2.637, 1.878, 6.423, 6.672, 2.034, 2.736, -1.565, 2.632, 5.854, 0.124, 1.933, 5.898, 3.179, 6.54, 9.004, 6.41, 2.336, -0.375, 4.685, 8.23, 6.899, 6.221, 8.352, 2.251, 10.636, 4.838, 12.016, 4.198, 10.078, 9.944, 2.405, 6.601, 6.32, 6.171, -5.873, 0.592, 6.047, 3.759, 5.606, 0.942, 7.847, 3.154, 0.434, -5.896, 8.854, 11.468, 4.572, 10.094, 3.052, 5.22, 8.488, 8.903, 5.887, 3.941, 1.489, 1.717, 4.584, 9.126, 4.443, 3.635, 4.341, 1.996, 4.266, 1.571, -0.07, 3.861, 5.499, -2.828, 0.696, 2.964, 0.901, 2.749, 7.311, 3.559, 4.898, 7.942, 2.894, 0.157, 4.649, 6.195, 0.862, 4.374, 6.702, 3.117, 7.041, 3.553, 9.252, 6.679, -2.0, 4.016, 10.009, 6.371, 0.991, -4.114, 4.098, 5.851, 3.488, 0.538, -1.786, 3.686, 6.462, 2.973, 2.148, 5.49, -6.724, 0.471, 0.352, 7.613, 8.48, 3.857, 7.586, 1.996, 2.635, 2.152, 4.069, 1.878, 4.059, 5.199, 3.699, 1.261, 6.599, 5.007, 11.225, 7.032, 1.539, 4.271, 8.907, -0.539, 4.708, 6.669, 3.015, 0.991, 2.872],
    observed=0.700,
    title="S6E11 Credit Card Fraud  ·  Part 7",
    subtitle="A language model reads the rows: GPT-2 and residual MLPs join the stack")
plt.show()

## 7.1 More of the same adds nothing

Every candidate below is added on its own to Part 6's 62 members; the score is the nested CV of the converged
stack (× 1e-4):

| candidate | own OOF AUC | gain in the stack |
|---|---|---|
| LightGBM from the GLM, raw view, 3 seeds | 0.95107 | +0.01 |
| CatBoost depth 3 from the GLM | 0.95118 | +0.04 |
| EBM with 10 interactions from the GLM, raw view | 0.95114 | +0.05 |
| XGBoost depth 2 from the GLM, 5 seeds | 0.95146 | +0.12 |

All of them start from the same GLM logit, so they mostly agree with the members already there. Bagging has
stopped helping as well: ten seeds of the GLM booster score 0.95150, five score 0.95151. `gboost.py` (6.2)
holds the code (`kind=cat`, `kind=ebm`).

## 7.2 Residual MLPs: the first gain

An MLP on the GPU takes the same inputs as the booster (raw columns, encodings, value counts, signed-log
transformed and standardized) and adds its output to the GLM's logit, so it also learns only the residual
(`gboost.py`, `kind=mlp`). It overfits within a few epochs at first (0.95080); a lower learning rate and more
dropout fix that:

| network | five seeds | bag of five |
|---|---|---|
| 512-512, dropout 0.4, lr 2e-4 | 0.95104-0.95119 | 0.95122 |
| 128-128, dropout 0.3, lr 3e-4 | 0.95102-0.95113 | 0.95122 |

Both bags together add **+0.44 × 1e-4** to the stack, on all five folds. That is small, but it is ten times
what the tree variants add, and it comes from a weaker model. Kept as two members they add more than as one
bag (+0.35). An MLP that does not start from the GLM scores only 0.9480 and adds nothing further. The bags are
loaded from our dataset: this notebook runs on CPU, and `gboost.py` trains them in two minutes per seed on a GPU.

## 7.3 GPT-2 writes the rows

The recipe is [Chris Deotte's](https://www.kaggle.com/competitions/playground-series-s6e10/discussion/754683),
from the S6E10 winning solution; [masayakawamata](https://www.kaggle.com/code/masayakawamata/s6e11-gpt-2-target-first-classifier-cv-0-9500)
applied it to this competition and published the predictions:

1. train a GPT-2 to write rows of the original dataset;
2. continue training on each fold's training rows, with **the label written first** and the feature order shuffled;
3. score each held-out row twice, with `is_fraud` True and False, and turn the two likelihoods into a prediction
   with Bayes' rule.

This is the generator's own story from Parts 3 and 6 run backwards: the label comes first and every field is
written given it. The model reaches 0.95003 from text alone, with no target encoding.

Such a jump deserves suspicion, so we checked it before trusting it. The split matches ours row by row, each
row was scored by a model that never trained on it, and the member's rank correlation with our GLM boost bag is
0.9598 on both the out-of-fold and the test rows, with matching score distributions.

What sets it apart is the combination. Eight other members are as far from our best model (rank correlation
below 0.97), but they score at most 0.9486: raw-view trees and RealMLPs. GPT-2 is that different at 0.9500. The
stack gives it the largest coefficient of all (+0.58).

The final stack is Part 6's 62 members, the GPT-2 member and the two MLP bags, written to `submission_part7.csv`:

In [ ]:
P6_SOURCES["masa"] = "s6e11-gpt-2-target-first-classifier-cv-0-9500"
d = src_dir("masa")
gpt_o = np.load(os.path.join(d, "oof.npy"), allow_pickle=False).astype(np.float32).astype(float)
gpt_t = np.load(os.path.join(d, "test.npy"), allow_pickle=False).astype(np.float32).astype(float)
print(f"GPT-2 target-first classifier: OOF AUC {roc_auc_score(y, gpt_o):.6f}, "
      f"rank correlation with the GLM boost bag {spearmanr(gpt_o, P6_O[:, 0 + P6_NAMES.index('gb_bag5')]).correlation:.4f}")

P7_NAMES = P6_NAMES + ["gpt2", "gbn_bag5_w512", "gbn_bag5_w128"]
P7_O = [P6_O, (rankdata(gpt_o) / len(y))[:, None]]
P7_T = [P6_T, (rankdata(gpt_t) / len(te))[:, None]]
for n in P7_NAMES[-2:]:      # already rank means of five seeds
    P7_O.append(np.load(find(f"{n}_oof.npy", ART))[:, None])
    P7_T.append(np.load(find(f"{n}_full.npy", ART))[:, None])
    print(f"{n}: OOF AUC {roc_auc_score(y, P7_O[-1][:, 0]):.6f}")
P7_O, P7_T = np.hstack(P7_O), np.hstack(P7_T)


def stacker7():
    # tol 1e-8: converged in about ten Newton steps (1e-10 only asks for more than double precision can give)
    return LogisticRegression(C=0.01, tol=1e-8, max_iter=200, solver="newton-cholesky")


nested62 = nested.copy()      # Part 6's nested predictions, kept for Part 8
nested = np.zeros(len(y))
for a, b in FOLDS:
    nested[b] = rankdata(stacker7().fit(rlogit(P7_O[a]), y[a]).decision_function(rlogit(P7_O[b]))) / len(b)
print(f"stack of {P7_O.shape[1]}: nested CV AUC {roc_auc_score(y, nested):.6f} | folds " +
      " ".join(f"{roc_auc_score(y[b], nested[b]):.5f}" for _, b in FOLDS))
ST7 = stacker7().fit(rlogit(P7_O), y)
coef = pd.Series(ST7.coef_[0], index=P7_NAMES)
display(coef.reindex(coef.abs().sort_values(ascending=False).index).head(12).round(3).to_frame("stack coefficient"))
P7_TEST = rankdata(ST7.decision_function(rlogit(P7_T))) / len(te)
sub = pd.DataFrame({"id": te["id"], TARGET: P7_TEST})
sub.to_csv("submission_part7.csv", index=False)      # Part 8 writes submission.csv
print("submission_part7.csv written:", sub.shape)
nested65 = nested.copy()

## 7.4 Why the public board cannot see it

The nested CV rose by 4.3 × 1e-4 and the public board by 0.7 × 1e-4. Before reading anything
into that, we measured how much a public score can say. We drew 60,000 random rows from the out-of-fold
predictions, 500 times, and compared the two stacks on each draw: the gain is
**+4.0 ± 3.5** (× 1e-4). 17% of the draws came out at or below what the public board showed.

Even folds of 140,000 rows disagree (+4.2, +3.1, +6.5, −0.1, +7.6). So the public result is about one standard
deviation low: neither a confirmation nor a warning. These draws are random and the public rows are one
fixed set, and Part 5 showed that set can behave unusually, so this is a guide and not a proof. We go with the
700,000 rows; the private board (80% of the test rows) will decide.

## 7.5 What did not help in Part 7

| idea | result | note |
|---|---|---|
| ten seeds of the GLM booster instead of five | 0.95150 vs 0.95151 | bagging is saturated |
| CatBoost, EBM and XGBoost boosters next to the MLPs and GPT-2 | no extra gain | they repeat the LightGBM booster |
| raw-view MLP on top of the two bags | +0.03 × 1e-4 | noise |
| MLP without the GLM start | 0.9480, +0.05 × 1e-4 | noise |

**Planned after Part 7:** more language-model members, and the two final submissions.

---
# Part 8 — Two stacks, one blend

*Public LB 0.95235 · nested CV 0.95199*

The board moved again after Part 7. We read what is up there, tested what could be tested with out-of-fold
predictions, and found the gain somewhere we had not looked: in our own two stacks.

In [ ]:
draw_part8(
    stats=[('Public LB', '0.95235', '+0.00013 over Part 7'), ('Nested CV', '0.95199', '−0.00007 from Part 7'), ("Weight of Part 6's stack", '50%', 'the stack without GPT-2'), ('Public files used', '0', 'two nested-CV stacks of ours')],
    curve=[(0.0, 0.951632, 0.95215), (0.25, 0.951845, None), (0.5, 0.951987, 0.95235), (0.75, 0.952057, 0.95232), (1.0, 0.952057, 0.95222)],
    pool=[('TabPFN-3.5, 2 members', -0.1), ('Sadam Torres raw trees, 3', 0.08), ('Sadam Torres digit GBDTs, 4', 0.1), ('masayakawamata singles, 2', 0.1), ("Will's trees, 7", 0.26), ('alex chilton weak members, 10', 0.49), ('all 28 (93 members)', 0.64)],
    settings=[('probit instead of logit', -0.78), ('C = 1', -0.4), ('C = 0.003', -0.39), ('C = 0.1', -0.11), ('0.75 stack + 0.25 hill climbing', -0.1), ('0.5 stack + 0.5 hill climbing', -0.51)],
    names=['Part 7', 'Part 6', 'Getting Started (Nov 4)', '0.95231 file', 'Part 8'],
    dist=[[0.0, 54.2, 18.6, 8.3, 13.3], [54.2, 0.0, 11.3, 21.4, 14.0], [18.6, 11.3, 0.0, 2.1, 1.7], [8.3, 21.4, 2.1, 0.0, 1.4], [13.3, 14.0, 1.7, 1.4, 0.0]],
    title="S6E11 Credit Card Fraud  ·  Part 8",
    subtitle="The best public score came from blending two of our own stacks, not from more members")
plt.show()

## 8.1 What is at the top of the public board now

The 0.9523-0.9524 notebooks mix public submission files, and those files contain one another:

* the 0.95231 file is 50% Part 7 of this notebook + 50% the Getting Started file of that day;
* Getting Started's current file is 30% Part 6 + 20% alex chilton's stack + 50% that 0.95231 notebook.

Their authors state this openly, and that the weights have no out-of-fold check. The table on the top figure
gives the rank distances. The useful thing we took from it: the 0.95231 file sits between Part 7 (with GPT-2)
and Part 6's family (without it). That is a blend we can build from our own stacks and score out of fold (8.3).

## 8.2 More members, other settings

Mohammed Majeed's [ladder](https://www.kaggle.com/code/mojjeed/s6e11-public-oof-stack-write-up-cv-0-95210) pooled
90 published members, TabPFN-3.5 among them. We added the ones our stack lacked to Part 7's 65 members
(nested CV, × 1e-4):

| added | gain |
|---|---|
| TabPFN-3.5, 2 members | -0.10 |
| Sadam Torres raw trees, 3 | +0.08 |
| Sadam Torres digit GBDTs, 4 | +0.10 |
| masayakawamata singles, 2 | +0.10 |
| Will's trees, 7 | +0.26 |
| alex chilton weak members, 10 | +0.49 |
| all 28 (93 members) | +0.64 |

TabPFN-3.5 is a new model family (OOF 0.9488), but on the raw columns it repeats what the raw-view trees say.
All 28 together reach nested CV 0.95212 and public 0.95218: below Part 7 on the public board, inside the noise of
7.4. So we did not keep them. Changing the stack itself loses CV every time:

| change | gain |
|---|---|
| probit instead of logit | -0.78 |
| C = 1 | -0.40 |
| C = 0.003 | -0.39 |
| C = 0.1 | -0.11 |
| 0.75 stack + 0.25 hill climbing | -0.10 |
| 0.5 stack + 0.5 hill climbing | -0.51 |

Probit links and hill-and-stack mixes appear in several high public notebooks. On these 65 members neither helps.

## 8.3 Blend the stack with GPT-2 and the stack without it

Part 6's stack and Part 7's stack differ mainly in the GPT-2 member. A rank mean of the two is a softer version of
that one decision, and both have nested out-of-fold predictions, so the blend can be scored honestly:

| weight of the Part 7 stack | nested CV | public LB |
|---|---|---|
| 0 | 0.951632 | 0.95215 |
| 0.25 | 0.951845 | not submitted |
| 0.5 | 0.951987 | 0.95235 |
| 0.75 | 0.952057 | 0.95232 |
| 1 | 0.952057 | 0.95222 |

The CV is flat from 0.75 to 1 and falls below that. The public board peaks at 0.5, where the CV has lost 0.00007.
`submission_part8.csv` is the **0.5** blend, the best public score here. Be clear about what that is: a weight chosen on
20% of the test rows, the habit 6.1 warns about, and differences inside the ±0.00035 of 7.4. It has only three
measured neighbours and a small CV cost, but the CV does not support it over 0.75. So `submission_mix75.csv`, the
blend that costs no CV (public 0.95232), is written too. Both are built below.

In [ ]:
curve = []
for w in (0.0, 0.25, 0.5, 0.75, 1.0):
    v = w * nested65 + (1 - w) * nested62
    curve.append((w, roc_auc_score(y, v)))
    print(f"weight of the Part 7 stack {w:.2f}: nested CV AUC {curve[-1][1]:.6f} | folds " +
          " ".join(f"{roc_auc_score(y[b], v[b]):.5f}" for _, b in FOLDS))


def stack_mix(w):
    return rankdata(w * rankdata(P7_TEST) + (1 - w) * rankdata(P6_TEST)) / len(te)


sub = pd.DataFrame({"id": te["id"], TARGET: stack_mix(0.5)})
sub.to_csv("submission_part8.csv", index=False)      # Part 9 writes submission.csv
pd.DataFrame({"id": te["id"], TARGET: stack_mix(0.75)}).to_csv("submission_mix75.csv", index=False)
print("submission_part8.csv (weight 0.5) and submission_mix75.csv written:", sub.shape)

## 8.4 Next: a second GPT-2

One language-model member was worth more than fifty tree members, and only one seed of it exists. The published recipe
is in the open, so we measured what another seed costs on our GPU (NVIDIA GB10): about 21 hours per fold to train
(real rows are longer than our first benchmark assumed) and 14 hours to score everything at 10 feature orders,
about 5 days in all. A pilot (new replica, fold 0) is running. If its fold 0 matches 0.9489 and lifts the stack on that fold, the other four folds follow.

**Planned after Part 8:** the second GPT-2 seed and the two final submissions.

---
# Part 9 — When a higher CV is not a higher score

*Public LB 0.95236 · nested CV 0.95199*

10-fold members, a digit CNN and more public members all raised the nested CV, and none of them raised the public
score. Soft labels raised neither. The one change that helped is the smallest, and it never touches the CV.

In [ ]:
draw_part9(
    stats=[('Public LB', '0.95236', '+0.00001 over Part 8'), ('Nested CV', '0.95199', 'unchanged: only the test side moved'), ('Refits averaged', '14 + 8 + 8', 'GLM boost, two residual MLPs'), ('Tried and rejected', '8 files', 'the CV rose on most, public on none')],
    cv_rows=[('10-fold members (stack)', 2.32), ('+ digit CNN + trees', 0.68), ('+ CNN + 28 members', 1.21), ('test-side bag', 0.0)],
    lb_rows=[('10-fold members (stack)', -0.7), ('+ digit CNN + trees', -1.1), ('+ CNN + 28 members', -1.5), ("0.7 + 0.3 Subh's file", -0.2), ('test-side bag', 0.1)],
    tenf_rows=[('GLM', 2.8), ('GLM boost, seed 0', 3.5), ('GLM boost, seed 1', 2.3), ('residual MLP bag, 512', 2.9), ('residual MLP bag, 128', 2.6)],
    pseudo_rows=[('weight 0.5, fold 0', -8.5), ('weight 0.5, fold 1', -6.1), ('weight 1, fold 0', -10.6), ('weight 1, fold 1', -7.4)],
    title="S6E11 Credit Card Fraud  ·  Part 9",
    subtitle="Three ways to raise the CV, and one small way to raise the public score")
plt.show()

## 9.1 Average more refits on the test side

The test prediction of a member is a refit on all rows, and we had more refits than the stack used: ten plus four
GLM boosters, and eight residual MLPs of each shape. The stack's weights come from the out-of-fold predictions, which do not change; only
three test columns are replaced by rank means of every refit. It lowers the variance of the test predictions and
cannot be measured out of fold, so the public board is the only check: 0.95236 against 0.95235. That is inside the noise, and
it is the only change here with no reason to be worse. It is written to `submission_part9.csv`:

In [ ]:
# the stacks of Part 8, fitted once more; only the test side of three members changes
P9_T = P7_T.copy()
for member, f in (("gb_bag5", "gb_testbag_full.npy"), ("gbn_bag5_w512", "gbn_w512_testbag_full.npy"), ("gbn_bag5_w128", "gbn_w128_testbag_full.npy")):
    P9_T[:, P7_NAMES.index(member)] = np.load(find(f, ART))
n62 = list(range(len(P6_NAMES)))
t65 = rankdata(stacker7().fit(rlogit(P7_O), y).decision_function(rlogit(P9_T))) / len(te)
t62 = rankdata(stacker7().fit(rlogit(P7_O[:, n62]), y).decision_function(rlogit(P9_T[:, n62]))) / len(te)
sub = pd.DataFrame({"id": te["id"], TARGET: rankdata(t65 + t62) / len(te)})
sub.to_csv("submission_part9.csv", index=False)      # Part 10 writes submission.csv
print("submission_part9.csv written:", sub.shape,
      "| rank correlation with the Part 8 file", round(spearmanr(sub[TARGET], stack_mix(0.5)).correlation, 6))

## 9.2 Ten folds lift the out-of-fold score only

Sadam Torres noted that 10-fold GLM boosters earn a large weight: their out-of-fold predictions come from models that
saw 90% of the rows, not 80%. It reproduces at once (`gboost.py`, `nfold=10`):

| model | 5 folds | 10 folds |
|---|---|---|
| GLM | 0.95047 | 0.95075 |
| GLM boost, seed 0 | 0.95137 | 0.95172 |
| residual MLP bag, 512 | 0.95122 | 0.95151 |
| residual MLP bag, 128 | 0.95122 | 0.95148 |

In the stack: nested CV 0.952253 on all five folds up. On the public board 0.95218 (Part 7: 0.95222), and 0.95228 in
the Part 8 recipe (0.95235). The reason is simple once seen: the test predictions were already refits on 100% of the
rows, so nothing about them improved. Only the out-of-fold score moved. Worse, a stack that mixes 10-fold members with
5-fold public members sees the former as stronger than they will be on the test rows. A gain that only exists on
the validation side is not a gain. (Part 11 corrects this conclusion: two public scores could not support it.)

## 9.3 Soft labels in the target encoding

The learning curve says more rows sharpen the encodings, and 300,000 test rows are sitting unused. So: predict their
fraud probability with a GLM, and add them to every count with that soft label (`gboost.py`, `pseudo=a`). The check is
clean, because inside each fold the validation rows count as unlabeled, exactly like the test rows.

| fold | Part 6 booster | weight 0.5 | weight 1 |
|---|---|---|---|
| 0 | 0.95039 | 0.94954 | 0.94933 |
| 1 | 0.95156 | 0.95095 | 0.95082 |

These numbers already exclude each row's own soft label from its encoding; the first attempt, without that, was as
bad. Stopped after two folds. Soft labels carry no new label information, and they may let training labels leak into the encodings through
the predictions, so the trees trust the encodings too much. We did not verify that explanation.

## 9.4 A digit CNN, more members, public files

[queyrusi's digit CNN](https://www.kaggle.com/code/queyrusi/fraud-a-shared-digit-cnn) reads the digit strings with shared
filters: OOF 0.94974, rank correlation 0.973 with our GLM boost and 0.966 with GPT-2. A genuinely different model, and
unlike 9.2 it changes the test side. Each row below uses the Part 8 recipe.

| change | nested CV | public LB |
|---|---|---|
| none (Part 8) | 0.951987 | 0.95235 |
| + digit CNN + its trees | 0.952055 | 0.95224 |
| + the same with the test bag | 0.952055 | 0.95225 |
| + those and 28 public members | 0.952108 | 0.95220 |

Then the method the top of the board uses, for the first time in this series, with no CV behind it: rank means with
Subh's public file, which sits 34e-4 away from ours, and masayakawamata's stack. 0.7 + 0.3 scored 0.95233, 0.5 + 0.5
scored 0.95227, and 0.6 + 0.3 + 0.1 scored 0.95230. None is kept.

The public board has rewarded fewer changes all week. With a noise of ±0.00035 we cannot tell whether that is real.
What we can say: more members and a higher nested CV are no longer moving this public score.

**Planned after Part 9:** the second GPT-2 seed and two final submissions, one on the public board and one on the nested CV.

---
# Part 10 — One public file

*Public LB 0.95239 · no nested CV for this file*

Part 8 said this series does not blend public submission files, and Part 9 tried it and failed with Subh's file. This
part does it, says so, and keeps the honest files next to it.

[Malyshev Danil's version 13](https://www.kaggle.com/code/malyshevdanil/predicting-credit-card-fraud-getting-started) is a rank
mean of our Part 8 and [amanatar's Sovereign Fusion](https://www.kaggle.com/code/amanatar/s6e11-sovereign-fusion-solution), scoring
0.95238. The Sovereign file is itself a 50/50 mix of a 48-member stack and a blend of public models; it sits 8.9e-4 from Part 9,
close enough to agree and far enough to differ. Swapping Part 8 for Part 9 gives 0.95239. amanatar republished that file after
this part, so the cell below no longer reproduces the scored file.

The gain over Part 9 is 0.00003 and the public noise is ±0.00035. It may not survive the private board. We
publish it because the public score is what this file is for; `submission_part7.csv` and `submission_part9.csv` remain
the versions we can defend with out-of-fold numbers.

In [ ]:
draw_part10(
    stats=[('Public LB', '0.95239', '+0.00003 over Part 9'), ('Nested CV', 'none', 'half of the file has no out-of-fold side'), ('Public file used', '1', "amanatar's Sovereign Fusion, 50%"), ('Distance to it', '8.9', '× 1e-4, rank correlation 0.99911')],
    ladder=[('Part 6', 0.95215), ('Part 7', 0.95222), ('Part 8', 0.95235), ('Part 9', 0.95236), ('Part 10', 0.95239)],
    names=['Part 7', 'Part 9', 'Sovereign', 'Part 10'],
    dist=[[0.0, 13.2, 7.3, 8.0], [13.2, 0.0, 8.9, 2.2], [7.3, 8.9, 0.0, 2.2], [8.0, 2.2, 2.2, 0.0]],
    title="S6E11 Credit Card Fraud  ·  Part 10",
    subtitle="One rank mean with one public file: the first score in this series without a CV behind it")
plt.show()

In [ ]:
P6_SOURCES["sov"] = "s6e11-sovereign-fusion-solution"
fusion = pd.read_csv(os.path.join(src_dir("sov"), "submission.csv")).set_index("id").loc[te["id"]].iloc[:, 0].values
print(f"Part 9 vs the public file: 1 - rank correlation {(1 - spearmanr(sub[TARGET], fusion).correlation) * 1e4:.1f} x 1e-4")
sub = pd.DataFrame({"id": te["id"], TARGET: rankdata(rankdata(sub[TARGET]) + rankdata(fusion)) / len(te)})
sub.to_csv("submission_part10.csv", index=False)      # Part 11 writes submission.csv
print("submission_part10.csv written:", sub.shape)

**Planned after Part 10:** the second GPT-2 seed.

---
# Part 11 — Every published member, on one split

*Public LB 0.95285 · nested CV 0.95245*

## 11.1 The pool was not saturated

Part 8 said more members no longer help. That was true of the 28 we added, not of the idea. Sameer Shaikh pooled 178, later 205
members and reached nested CV 0.9524-0.9525. We rebuilt that pool from the same 12 sources plus every member of this series:
public OOF banks from Sadam Torres (with newer GLM boosts and TabPFN), alex chilton, Malyshev Danil, lazyyeou, megayak, Will,
queyrusi, Vladimir Demidov, Nilanjan_Saha123, masayakawamata (GPT-2 and a stack) and Sameer Shaikh's own banks, including 10-fold
GLM boosts. 236 members load; those within 0.008 of the best, without near-copies, leave 200. The same converged logistic stack as
Part 7 (`C = 0.01`, Newton steps) gives nested CV 0.952504 and public 0.95267.

The stack ran offline: 200 members is a long input list and any source can republish. Its nested and test predictions are in our
dataset (`wide_stack_nested.npy`, `wide_stack_test.npy`) with the member list. The loader follows Sameer Shaikh's notebook.

## 11.2 Blend it with Part 9

The wide stack is 95.0e-4 away from Part 9, an unusually large distance between strong files.

| weight of the wide stack | nested CV | public LB |
|---|---|---|
| 1 | 0.952504 | 0.95267 |
| 0.8 | 0.952480 | 0.95282 |
| **0.7** | **0.952452** | **0.95285** |
| 0.5 | 0.952368 | 0.95283 |

Be clear about what this is: the CV falls slightly as Part 9 comes in, while the public score rises. We chose 0.7 on the public board,
as in Part 8, and that is inside the noise of ±0.00035. It costs 0.00005 of nested CV against the wide stack alone, and
`wide_stack_test.npy` is there for anyone who prefers it. Part 9's test file is frozen in our dataset too, since its sources have
been republished; the cells above now print slightly different numbers than the ones scored.

In [ ]:
wide_test = np.load(find("wide_stack_test.npy", ART))
wide_nested = np.load(find("wide_stack_nested.npy", ART))
part9_test = np.load(find("part9_test.npy", ART))      # the scored Part 9 file, frozen: its sources have moved on since
part9_nested = 0.5 * nested65 + 0.5 * nested62
print(f"wide stack: nested CV {roc_auc_score(y, wide_nested):.6f} | {len(pd.read_csv(find('wide_stack_members.csv', ART)))} members")
for w in (1.0, 0.8, 0.7, 0.5):
    v = w * wide_nested + (1 - w) * part9_nested
    print(f"{w} wide + {1 - w:.1f} Part 9: nested CV {roc_auc_score(y, v):.6f} | folds " +
          " ".join(f"{roc_auc_score(y[b], v[b]):.5f}" for _, b in FOLDS))
sub = pd.DataFrame({"id": te["id"], TARGET: rankdata(0.7 * rankdata(wide_test) / len(te) + 0.3 * rankdata(part9_test) / len(te)) / len(te)})
sub.to_csv("submission.csv", index=False)
print("submission.csv written:", sub.shape)

## 11.3 A correction

Part 9 said 10-fold members only lift the out-of-fold score, from two submissions that came in 0.00004 and 0.00007 below their
references. That difference is a tenth of the public noise of 7.4, and we applied that same warning to everything except this. The wide
stack contains 10-fold GLM boosts, its CV rose and the public score rose with it. The reasoning about the test side still holds: a 10-fold
OOF is stronger than a refit-based test prediction lets on. But "illusion" was not something two public scores could show.

---
## Next (written with Part 11)

A second GPT-2 seed was training at this point; Part 12 adds a different language model first.

---
# Part 12 — A second language model in the wide stack

*Public LB 0.95298 · nested CV 0.95252*

## 12.1 Where the next public scores came from

After Part 11, Areeb Ali reached 0.95291 by mixing Part 11's frozen files with megayak's Shallow-Tree Stack submission at 20%. That file is
far from ours for one reason: megayak trained their own target-first GPT-2 and published its out-of-fold predictions
([S6E11 target-first LM predictions](https://www.kaggle.com/datasets/megayak/s6e11-target-first-lm), CC0). It writes each column as one learned
token, scores 0.95043 out of fold on its own, and its rank correlation with masayakawamata's GPT-2 is only 0.980.

A submission file can only be mixed in by looking at the public board. Out-of-fold predictions on the same five folds can go into the stack
and be measured.

## 12.2 One member more

The same loader, gate and converged logistic stack as Part 11, with that one member added (201 members):

| | nested CV | public LB |
|---|---|---|
| Part 11 wide stack | 0.952504 | 0.95267 |
| wide stack + megayak's LM | **0.952565** | |
| Part 11: 0.7 wide + 0.3 Part 9 | 0.952461 | 0.95285 |
| **Part 12: 0.7 (wide + LM) + 0.3 Part 9** | **0.952522** | **0.95298** |

Four of five folds rise, in the stack alone and in the blend. The blend rows are recomputed by the cell below (the Part 9 side from the live
inputs, so Part 11's row reads 0.952461 here against the 0.952452 printed when it was published). The weights are Part 11's, not tuned again, and the public score was taken
once. No public submission file is used.

In [ ]:
draw_part11(
    stats=[('Public LB', '0.95298', '+0.00013 over Part 11'), ('Nested CV', '0.95252', '+0.00006 over Part 11'), ('Members', '201', "one new: megayak's LM"), ('Public files used', '0', 'all out-of-fold stacks')],
    ladder=[('Part 6', 0.95215), ('Part 8', 0.95235), ('Part 10', 0.95239), ('wide stack', 0.95267), ('Part 11', 0.95285), ('Part 12', 0.95298)],
    nested=[('Part 6', 0.951632), ('Part 7', 0.952057), ('Part 11 wide', 0.952504), ('Part 12', 0.952522)],
    sources=[('goodpjw2008', 68), ('alexchilton', 36), ('malyshevdanil', 23), ('sameershaikh27', 21), ('lazyyeou', 17), ('sadamtorres', 13), ('megayak', 8), ('wguesdon', 7), ('masayakawamata', 3), ('queyrusi', 2), ('nilanjansaha123', 2), ('yekenot', 1)],
    title="S6E11 Credit Card Fraud  ·  Part 12",
    subtitle="One more language model in the wide stack: a second view of the rows as text",
    src_title="201 members, 12 sources")
plt.show()

In [ ]:
lm_oof = np.load(find("lm_oof.npy")).astype(float)          # megayak's target-first language model (CC0 dataset)
print(f"megayak's language model alone: OOF AUC {roc_auc_score(y, lm_oof):.6f}")
wlm_test = np.load(find("wide_lm_stack_test.npy", ART))
wlm_nested = np.load(find("wide_lm_stack_nested.npy", ART))
print(f"wide stack + language model: nested CV {roc_auc_score(y, wlm_nested):.6f} | "
      f"{len(pd.read_csv(find('wide_lm_stack_members.csv', ART)))} members (Part 11: {roc_auc_score(y, wide_nested):.6f})")
for tag, n in (("Part 11 wide", wide_nested), ("wide + LM", wlm_nested)):
    v = 0.7 * rankdata(n) / len(n) + 0.3 * rankdata(part9_nested) / len(n)
    print(f"0.7 {tag:13s} + 0.3 Part 9: nested CV {roc_auc_score(y, v):.6f} | folds " +
          " ".join(f"{roc_auc_score(y[b], v[b]):.5f}" for _, b in FOLDS))
sub = pd.DataFrame({"id": te["id"], TARGET: rankdata(0.7 * rankdata(wlm_test) / len(te) + 0.3 * rankdata(part9_test) / len(te)) / len(te)})
sub.to_csv("submission.csv", index=False)
print("submission.csv written:", sub.shape)

---
## Next

Our own second GPT-2 seed is still training (about halfway). Language models are the one direction that keeps adding to the stack,
so it will join as its own member when its five folds are done.

Every member of this series is in [S6E11 OOF Bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank).

This notebook will keep growing. If it was useful, an upvote helps the series continue; and please upvote
the notebooks credited above, megayak's language model in particular, whose published predictions made this part possible.